# Ekonometrik / Ekonomik Modelleme Notebook'u

**Konfigürasyon odaklı, yeniden kullanılabilir** bir ekonometrik modelleme
çatısı. Amaç: kodun büyük kısmına dokunmadan, yalnızca en üstteki `CONFIG`
bloğunu değiştirerek farklı veri setleri ve değişkenler üzerinde çok sayıda
ekonometrik modeli kurmak, test etmek, karşılaştırmak, en iyi modeli seçmek,
forecast üretmek ve tüm çıktıları düzenli bir Excel dosyasına yazmak.

**Desteklenen model aileleri:** OLS, ADL, Distributed Lag, ARDL, ECM,
VAR, VECM, SARIMAX, Ridge, Lasso, ElasticNet.

**Çalışma mantığı**
1. Kullanıcı yalnızca `CONFIG` bloğunu düzenler.
2. Ana kod hücreleri sabit kalır.
3. Her model aynı hedef değişken ve aynı forecast senaryosu üzerinden
   karşılaştırılır.
4. Hata veren model tüm süreci durdurmaz; hata `10_ERRORS_WARNINGS`
   sayfasına yazılır.
5. Sonuçlar tek bir Excel dosyasına aktarılır ve her test için kısa,
   net **Türkçe yorum** üretilir.

> Not: Notebook, `input_file` bulunamazsa otomatik olarak sentetik bir örnek
> veri seti üretir (bkz. `CONFIG["generate_sample_if_missing"]`). Böylece
> "tak-çalıştır" mantığında baştan sona sorunsuz çalışır.

## 1. Kurulum ve importlar

Gerekli paketler import edilir. Opsiyonel paketler (`arch`, `pmdarima`,
`linearmodels`) yoksa notebook çökmez; ilgili özellikler devre dışı kalır ve
kullanıcıya açık bir uyarı verilir.

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import coint, adfuller

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

import matplotlib
try:
    get_ipython()  # noqa: F821  (notebook ortamı -> inline backend kalsın)
except NameError:
    matplotlib.use("Agg")  # script/headless çalıştırmada güvenli
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

# --- Opsiyonel paketleri sessizce dene ---
OPTIONAL = {}
for _pkg in ["arch", "pmdarima", "linearmodels"]:
    try:
        __import__(_pkg)
        OPTIONAL[_pkg] = True
    except Exception:
        OPTIONAL[_pkg] = False

# --- Çekirdek zaman serisi bileşenleri (statsmodels) ---
HAVE = {}
try:
    from statsmodels.tsa.ardl import ARDL, ardl_select_order
    HAVE["ardl"] = True
except Exception:
    HAVE["ardl"] = False
try:
    from statsmodels.tsa.api import VAR
    from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen, select_coint_rank
    HAVE["var"] = True
except Exception:
    HAVE["var"] = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAVE["sarimax"] = True
except Exception:
    HAVE["sarimax"] = False
try:
    from statsmodels.regression.rolling import RollingOLS
    HAVE["rolling"] = True
except Exception:
    HAVE["rolling"] = False

# Başlangıç uyarıları (Excel'de 10_ERRORS_WARNINGS sayfasına da düşer)
STARTUP_NOTES = []
for _p, _ok in OPTIONAL.items():
    if not _ok:
        STARTUP_NOTES.append(
            f"Opsiyonel paket '{_p}' bulunamadi; ilgili gelismis ozellikler devre disi "
            f"(notebook calismaya devam eder). Kurmak icin: pip install {_p}"
        )
for _k in ["ardl", "var", "sarimax"]:
    if not HAVE.get(_k):
        STARTUP_NOTES.append(f"statsmodels bileseni '{_k}' yuklenemedi; bu model ailesi atlanacak.")

print("Importlar tamam.")
print("Opsiyonel paketler:", OPTIONAL)
for _n in STARTUP_NOTES:
    print("UYARI:", _n)

## 2. CONFIG bloğu

**Notebook'un tek merkezi ayar noktası.** Kullanıcı normalde yalnızca burayı
değiştirir. Aşağıdaki örnek `CONFIG`, notebook'un ürettiği sentetik örnek veri
(`y`, `usdtry`, `policy_rate`) ile birebir çalışacak şekilde doldurulmuştur.
Kendi verinizi kullanmak için `input_file`, `date_col`, `target_col`, değişken
isimleri, transformlar, model spesifikasyonları ve forecast senaryolarını
güncelleyin. Değişken isimleri **hard-code edilmemiştir**; tamamı buradan yönetilir.

In [ ]:
CONFIG = {
    # --- Girdi / veri ---
    "input_file": "input.xlsx",
    "sheet_name": "Sheet1",
    "date_col": "date",
    "target_col": "y",

    "frequency": "M",          # D, W, M, Q, A
    "date_format": None,       # örn "%Y-%m-%d"; None ise otomatik

    "sample_start": None,      # tahmin örnekleminin başı (None = otomatik)
    "sample_end": None,        # tahmin örnekleminin sonu (None = otomatik)

    "forecast_start": None,    # None ise verinin bittiği yerin bir sonrası
    "forecast_end": None,      # None ise forecast_start + forecast_horizon

    "forecast_horizon": 12,    # forecast_end verilmezse kaç dönem ileri

    # --- Eksik veri politikası ---
    "missing_method": "interpolate",   # interpolate, ffill, bfill, dropna, none

    # --- Transformasyonlar ---
    "target_transform": "level",       # level, log, diff, logdiff, pct_change
    "exog_transforms": {
        "usdtry": "logdiff",
        "policy_rate": "level",
    },

    # --- Çalıştırılacak modeller ---
    "models_to_run": [
        "OLS",
        "ADL",
        "DISTRIBUTED_LAG",
        "ARDL",
        "ECM",
        "VAR",
        "VECM",
        "SARIMAX",
        "RIDGE",
        "LASSO",
        "ELASTICNET",
    ],

    # --- Model spesifikasyonları ---
    "model_specs": {
        "OLS": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [],
            "lags_x": {},              # {var: [laglar]}; boşsa çağdaş (lag 0)
            "add_constant": True,
        },
        "ADL": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
            "add_constant": True,
        },
        "DISTRIBUTED_LAG": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [],
            "lags_x": {"usdtry": [0, 1, 2], "policy_rate": [0, 1]},
            "add_constant": True,
        },
        "ARDL": {
            "exog": ["usdtry", "policy_rate"],
            "max_lag_y": 4,
            "max_lag_x": 4,
            "selection_criterion": "bic",   # aic/bic
        },
        "ECM": {
            "level_vars": ["usdtry", "policy_rate"],   # uzun dönem (seviye)
            "diff_vars": ["usdtry", "policy_rate"],    # kısa dönem (fark)
            "cointegration_method": "engle_granger",
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
        },
        "VAR": {
            "vars": ["y", "usdtry", "policy_rate"],
            "maxlags": 6,
            "ic": "bic",
            "granger": True,
            "irf": False,
        },
        "VECM": {
            "vars": ["y", "usdtry", "policy_rate"],
            "deterministic": "ci",
            "k_ar_diff": 1,
            "coint_rank": 1,
        },
        "SARIMAX": {
            "order": [1, 0, 1],
            "seasonal_order": [0, 0, 0, 0],
            "exog": ["policy_rate"],
        },
        "RIDGE": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
            "alpha_grid": [0.01, 0.1, 1, 10, 100],
        },
        "LASSO": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
            "alpha_grid": [0.001, 0.01, 0.1, 1, 10],
        },
        "ELASTICNET": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
            "alpha_grid": [0.001, 0.01, 0.1, 1, 10],
            "l1_ratio_grid": [0.2, 0.5, 0.8],
        },
    },

    # --- Forecast senaryoları ---
    # Tipler: constant, growth, path, linear, shock. Verilmeyen değişken için
    # son gözlem sabit tutulur (uyarı verilir).
    "forecast_scenarios": {
        "baseline": {
            "usdtry": {"type": "growth", "monthly_rate": 0.02},
            "policy_rate": {"type": "constant", "value": 45.0},
            # Politika faizi (TR) manuel patika. Tarihler ay-başına hizalıdır;
            # verilmeyen aralar lineer interpolate, uçlar ffill/bfill ile dolar.
            "policy_rate_tr": {
                "type": "path",
                "values": {
                    "2026-06-01": 36.3846,   # Haz.26
                    "2026-07-01": 36.3846,   # Tem.26
                    "2026-08-01": 36.3846,   # Agu.26
                    "2026-09-01": 35.1563,   # Eyl.26
                    "2026-10-01": 35.1563,   # Eki.26
                    "2026-11-01": 35.1563,   # Kas.26
                    "2026-12-01": 31.6364,   # Ara.26
                    "2027-01-01": 31.6364,   # Oca.27
                    "2027-02-01": 31.6364,   # Sub.27
                    "2027-03-01": 29.0,      # Mar.27
                    "2027-04-01": 29.0,      # Nis.27
                    "2027-05-01": 29.0,      # May.27
                },
            },
        },
    },
    "active_scenario": "baseline",

    # --- Değerlendirme ---
    "test_size": 12,               # son N gözlem test seti
    "selection_metric": "bic",     # aic, bic, rmse, mae, mape, adjusted_r2, clean_score

    # --- Beklenen katsayı işaretleri (transform edilmiş isimlerle) ---
    "expected_signs": {
        "dln_usdtry": "+",
        "policy_rate": "-",
    },

    # --- Robust standart hata ---
    "cov_type": "HAC",             # nonrobust, HC0, HC1, HC3, HAC
    "hac_lags": 4,

    # --- clean_score ceza puanları (değiştirilebilir) ---
    "clean_score": {
        "start": 100,
        "bg": 20,          # Breusch-Godfrey p<0.05
        "bp": 15,          # Breusch-Pagan p<0.05
        "white": 15,       # White p<0.05
        "reset": 20,       # RESET p<0.05
        "vif": 15,         # max VIF > 10
        "insignificant": 10,   # katsayıların büyük kısmı anlamsız
        "sign": 10,        # işaret beklentisi karşılanmıyor
        "forecast": 10,    # forecast performansı kötü
        "vif_threshold": 10.0,
        "insignificant_share": 0.5,   # anlamsız pay bu eşiği geçerse ceza
        "mape_bad": 20.0,             # test MAPE bu değerin üzerindeyse forecast cezası
    },

    # --- Kayan pencere (rolling) OLS analizi (opsiyonel katman) ---
    # Zamanla değişen katsayıları çıkarır; ana model seçim/forecast akışını
    # etkilemez. enabled=False iken hiç çalışmaz.
    "rolling": {
        "enabled": True,
        "model": "OLS",        # hangi regresyon spec'i kullanılsın (OLS/ADL/DISTRIBUTED_LAG)
        "window": 12,          # kayan pencere genişliği (gözlem)
        "min_nobs": 12,        # pencere için gereken min gözlem (genelde = window)
        "step": 1,             # çıktı kaç dönemde bir raporlansın
        "mode": "coefficients",  # şu an "coefficients" destekleniyor
    },

    # --- İLERİ ANALİZ: Heteroskedastisite teşhis + giderme katmanı (opsiyonel) ---
    # Mevcut ana akışı (model seçimi/forecast/Excel) ETKİLEMEZ; ayrı bir Excel
    # dosyasına yazar. enabled=False iken hiç çalışmaz. Amaç: heteroskedastisite
    # tespit edilince yalnızca HC3 uygulamak yerine kaynağı teşhis edip WLS/FGLS/
    # dönüşüm/yapısal kırılma/ARCH-GARCH alternatiflerini karşılaştırmalı sınamak.
    "advanced": {
        "enabled": True,
        "base_model": "OLS",              # bağımsız değişken + lag kaynağı spec'i (OLS/ADL/DISTRIBUTED_LAG)
        "independent_variables": [],      # boşsa base_model spec'inin exog'u kullanılır (orijinal/base isimler)
        "robust_covariance": "HC3",       # nonrobust/HC0/HC1/HC3/HAC
        "significance_level": 0.05,
        "max_lag_y": 6,
        "max_lag_x": 6,
        "lag_selection_criterion": "BIC", # AIC/BIC (ARDL varyantı)
        "test_structural_breaks": True,
        "manual_break_dates": [],         # ör. ["2018-08-01"]
        "test_wls": True,
        "test_fgls": True,
        "test_arch_garch": True,          # ARCH-LM anlamlıysa artık-tabanlı GARCH
        "goldfeld_quandt": True,
        "make_log_variants": True,        # ARDL_LogY / LogLog / Difference varyantları
        "bias_correction": True,          # log->seviye forecast bias düzeltmesi (exp(f+0.5 sigma^2))
        "fgls_max_iter": 5,
        "output_excel_path": "econometric_diagnosis_output.xlsx",
        "make_plots": True,
        "plot_dir": "plots_advanced",
    },

    # --- ARDL HETEROSKEDASTİSİTE DENETİMİ (gerçek statsmodels ARDL üzerinden) ---
    # Ana akışı etkilemez. Gerçek ARDL nesnesi kurar; ardl_order/ar_lags/dl_lags,
    # HC3-vs-klasik, tam tasarım matrisiyle BP (klasik+Koenker), çoklu-lag ARCH-LM,
    # güvenli White, yapısal kırılma ve alternatif spesifikasyonları denetler.
    "ardl_audit": {
        "enabled": True,
        "dependent": None,        # None -> target_col
        "independent": None,      # None -> advanced.independent_variables ya da base_model exog
        "max_p": 8,               # ardl_select_order için maksimum y gecikmesi
        "max_q": 4,               # maksimum x gecikmesi
        "ic": "bic",              # aic/bic/hqic
        "trend": "c",             # n/c/ct
        "causal": False,          # ARDL causal (cari dönem x dahil mi)
        "alpha": 0.05,
        "arch_lags": [1, 3, 6, 12],
        "manual_break_dates": [],
        "output_excel_path": "ardl_hetero_audit.xlsx",
        "make_plots": True,
        "plot_dir": "plots_ardl_audit",
    },

    # --- Grafik / çıktı ---
    "make_plots": True,
    "plot_dir": "plots",
    "output_excel": "model_outputs.xlsx",

    # --- Kolaylık ---
    "generate_sample_if_missing": True,
    "random_seed": 42,
}

## 3. Veri okuma ve hazırlama

- Excel okunur, tarih kolonu `datetime`'a çevrilir ve index yapılır.
- Frekansa göre sıralanır; eksik tarihler ve eksik değerler raporlanır.
- Eksik değerler `CONFIG["missing_method"]` ile (interpolate / ffill / bfill /
  dropna / none) işlenir.
- Transformasyonlar uygulanır: `level`, `log`, `diff`, `logdiff`, `pct_change`.
- Transform edilmiş değişken isimleri sistematik üretilir: `ln_x`, `d_x`,
  `dln_x`, `pct_x`.
- Log/logdiff yapılacak değişkende sıfır/negatif değer varsa **hata değil,
  uyarı** verilir ve o transform `level` olarak alınır.

In [ ]:
# Frekans kodu eşlemesi (pandas 3.x uyumlu). Ay/çeyrek/yıl için hem dönem-sonu
# (ME/QE/YE) hem dönem-başı (MS/QS/YS) kodları desteklenir.
FREQ_MAP = {"D": "D", "W": "W",
            "M": "ME", "ME": "ME", "MS": "MS",
            "Q": "QE", "QE": "QE", "QS": "QS",
            "A": "YE", "Y": "YE", "YE": "YE", "YS": "YS", "AS": "YS"}


def resolve_frequency(index, freq_code):
    """Veri index'ine göre uygun pandas frekansını (dönem-başı/sonu dahil) belirler.

    Bu, ay-başı (2014-01-01) veriyi ay-sonu (ME) takvimine sabitleyip tüm değerleri
    NaN'a düşüren anchor uyuşmazlığını önler. Önce frekans veriden çıkarılmaya
    çalışılır; olmazsa kısayol kodu (M/Q/A) verinin ilk gününe göre başa/sona
    hizalanır.
    """
    # 1) Frekansı doğrudan veriden çıkarmayı dene (en güvenilir yol)
    try:
        inferred = pd.infer_freq(index)
    except Exception:
        inferred = None
    if inferred:
        return inferred
    # 2) Kısayol kodunu anchor-farkındalı çöz
    code = str(freq_code).upper()
    try:
        first_day = int(index.min().day)
        first_month = int(index.min().month)
    except Exception:
        first_day, first_month = 1, 1
    if code in ("M", "ME", "MS"):
        return "MS" if first_day == 1 else "ME"
    if code in ("Q", "QE", "QS"):
        return "QS" if first_day == 1 else "QE"
    if code in ("A", "Y", "YE", "YS", "AS"):
        return "YS" if (first_day == 1 and first_month == 1) else "YE"
    return FREQ_MAP.get(code, freq_code)

# Transform ön ekleri
_TR_PREFIX = {"level": "", "log": "ln_", "diff": "d_", "logdiff": "dln_", "pct_change": "pct_"}


def transformed_name(col, transform):
    """Bir değişken ve transform için sistematik kolon adı üretir."""
    if transform == "level":
        return col
    return _TR_PREFIX.get(transform, transform + "_") + col


def apply_transform(series, transform):
    """Bir seriye transform uygular. Log/logdiff'te pozitiflik gerekir."""
    s = pd.Series(series).astype(float)
    if transform == "level":
        return s
    if transform == "log":
        return np.log(s)
    if transform == "diff":
        return s.diff()
    if transform == "logdiff":
        return np.log(s).diff()
    if transform == "pct_change":
        return s.pct_change()
    raise ValueError(f"Bilinmeyen transform: {transform}")


def invert_transform_step(prev_level, modeled_value, transform):
    """Bir dönemlik modellenmiş değeri (transform edilmiş) seviyeye çevirir."""
    if transform == "level":
        return modeled_value
    if transform == "log":
        return np.exp(modeled_value)
    if transform == "diff":
        return prev_level + modeled_value
    if transform == "logdiff":
        return prev_level * np.exp(modeled_value)
    if transform == "pct_change":
        return prev_level * (1.0 + modeled_value)
    raise ValueError(f"Bilinmeyen transform: {transform}")


def generate_sample_data(config):
    """input_file yoksa kullanılan sentetik örnek veri (aylık, 120 gözlem)."""
    seed = config.get("random_seed", 42)
    rng = np.random.default_rng(seed)
    n = 120
    dates = pd.date_range("2016-01-31", periods=n, freq="ME")

    # Politika faizi: sınırlandırılmış rastgele yürüyüş
    pr = np.zeros(n); pr[0] = 8.0
    for t in range(1, n):
        pr[t] = np.clip(pr[t - 1] + rng.normal(0, 1.2), 5, 50)

    # Kur: pozitif, birikimli büyüme
    growth = rng.normal(0.015, 0.02, n)
    usd = 3.0 * np.cumprod(1 + growth)

    dln_usd = np.concatenate([[0.0], np.diff(np.log(usd))])
    # Hedef (ör. bir endeks): AR(1) + kur şoku (+) + faiz (-)
    y = np.zeros(n); y[0] = 100.0
    for t in range(1, n):
        y[t] = (17.0 + 0.85 * y[t - 1] + 40.0 * dln_usd[t]
                - 0.15 * pr[t] + rng.normal(0, 1.0))

    return pd.DataFrame({
        config["date_col"]: dates,
        config["target_col"]: y,
        "usdtry": usd,
        "policy_rate": pr,
    })


def load_data(config):
    """Excel/veri okur; yoksa (ve izin varsa) sentetik örnek veri üretir."""
    path = Path(config["input_file"])
    if path.exists():
        raw = pd.read_excel(path, sheet_name=config.get("sheet_name", 0))
    else:
        if config.get("generate_sample_if_missing", True):
            print(f"'{path}' bulunamadi -> sentetik ornek veri uretiliyor.")
            raw = generate_sample_data(config)
        else:
            raise FileNotFoundError(
                f"Girdi dosyasi bulunamadi: {path}. CONFIG['input_file'] kontrol edin "
                f"veya generate_sample_if_missing=True yapin."
            )
    return raw


def prepare_data(raw_df, config):
    """Ham veriyi temizler, transform eder ve modellemeye hazır hale getirir.

    Meta bilgileri (registry, hedef kolonlar, uyarılar) prepared_df.attrs['meta']
    içinde saklanır; böylece istenen fonksiyon imzaları korunur.
    """
    warns = []
    df = raw_df.copy()
    dcol = config["date_col"]
    if dcol not in df.columns:
        raise KeyError(f"Tarih kolonu '{dcol}' veride bulunamadi. Mevcut kolonlar: {list(df.columns)}")

    df[dcol] = pd.to_datetime(df[dcol], format=config.get("date_format"), errors="coerce")
    if df[dcol].isna().any():
        warns.append("Bazi tarihler ayristirilamadi (NaT); ilgili satirlar dusuruldu.")
        df = df.dropna(subset=[dcol])
    if df[dcol].duplicated().any():
        warns.append("Tarih kolonunda tekrar eden gozlemler var; ilk kayitlar tutuldu.")
        df = df.drop_duplicates(subset=[dcol], keep="first")

    df = df.sort_values(dcol).set_index(dcol)
    # Frekansı veriye göre anchor-farkındalı çöz (ay-başı/ay-sonu vb.)
    freq = resolve_frequency(df.index, config["frequency"])

    # Tam takvim ile eksik tarih tespiti (+ güvenlik: takvim veriyle örtüşmezse
    # yeniden örneklemeden orijinal index korunur ki veri NaN'a düşmesin)
    try:
        full_idx = pd.date_range(df.index.min(), df.index.max(), freq=freq)
        overlap = df.index.intersection(full_idx)
        if len(df.index) > 0 and len(overlap) < 0.5 * len(df.index):
            missing_dates = pd.DatetimeIndex([])
            warns.append(
                f"Frekans takvimi ('{freq}') veriyle yeterince ortusmedi "
                f"(ortusen: {len(overlap)}/{len(df.index)}); yeniden orneklemeden "
                f"orijinal tarih index'i korundu."
            )
        else:
            missing_dates = full_idx.difference(df.index)
            if len(missing_dates) > 0:
                warns.append(f"{len(missing_dates)} eksik tarih tespit edildi ve takvime eklendi.")
            df = df.reindex(full_idx)
            df.index.name = dcol
    except Exception as e:
        missing_dates = pd.DatetimeIndex([])
        warns.append(f"Frekans yeniden orneklemesi yapilamadi ({e}); veri oldugu gibi kullanildi.")

    # Eksik değer raporu
    missing_report = df.isna().sum()
    total_missing = int(missing_report.sum())
    if total_missing > 0:
        warns.append(f"Toplam {total_missing} eksik deger var; '{config.get('missing_method')}' ile islendi.")

    method = config.get("missing_method", "interpolate")
    if method == "interpolate":
        df = df.interpolate(method="time").ffill().bfill()
    elif method == "ffill":
        df = df.ffill()
    elif method == "bfill":
        df = df.bfill()
    elif method == "dropna":
        df = df.dropna()
    # "none" -> dokunma

    # --- Transformlar ---
    registry = {}
    base_to_modeled = {}

    def add_transform(base, tr):
        if base not in df.columns:
            warns.append(f"Degisken '{base}' veride yok; transform atlandi.")
            return None
        if tr in ("log", "logdiff") and (df[base].dropna() <= 0).any():
            warns.append(f"'{base}' sifir/negatif deger iceriyor; '{tr}' yerine 'level' kullanildi.")
            registry[base] = {"base": base, "transform": "level"}
            base_to_modeled[base] = base
            return base
        try:
            name = transformed_name(base, tr)
            df[name] = apply_transform(df[base], tr)
        except Exception as e:
            warns.append(f"'{base}' icin '{tr}' transformu basarisiz ({e}); level kullanildi.")
            registry[base] = {"base": base, "transform": "level"}
            base_to_modeled[base] = base
            return base
        registry[name] = {"base": base, "transform": tr}
        base_to_modeled[base] = name
        return name

    target_base = config["target_col"]
    target_tr = config.get("target_transform", "level")
    target_modeled = add_transform(target_base, target_tr)
    for base, tr in config.get("exog_transforms", {}).items():
        add_transform(base, tr)

    # Örneklem kırpma
    ss, se = config.get("sample_start"), config.get("sample_end")
    if ss is not None:
        df = df[df.index >= pd.to_datetime(ss)]
    if se is not None:
        df = df[df.index <= pd.to_datetime(se)]

    df.attrs["meta"] = {
        "registry": registry,
        "base_to_modeled": base_to_modeled,
        "target_base": target_base,
        "target_modeled": target_modeled,
        "target_transform": registry.get(target_modeled, {}).get("transform", target_tr),
        "freq": freq,
        "warnings": warns,
        "missing_report": missing_report,
        "missing_dates": missing_dates,
    }
    return df

## 4. Yardımcı fonksiyonlar

Lag üretimi, regresyon matrisi kurulumu, güvenli model fit'i, metrik ve
**Türkçe yorum** üreten fonksiyonlar. Bu hücre modellerin ortak altyapısıdır.

In [ ]:
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, linear_reset

# --- Metrikler (NaN ve sıfır-gerçek-değere karşı güvenli) ---
def _rmse(actual, fitted):
    a, f = np.asarray(actual, float), np.asarray(fitted, float)
    m = ~(np.isnan(a) | np.isnan(f))
    if m.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - f[m]) ** 2)))


def _mae(actual, fitted):
    a, f = np.asarray(actual, float), np.asarray(fitted, float)
    m = ~(np.isnan(a) | np.isnan(f))
    if m.sum() == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - f[m])))


def _mape(actual, fitted):
    """MAPE; gerçek değeri sıfır olan gözlemler atlanır (hata vermez)."""
    a, f = np.asarray(actual, float), np.asarray(fitted, float)
    m = ~(np.isnan(a) | np.isnan(f)) & (a != 0)
    if m.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((a[m] - f[m]) / a[m])) * 100.0)


def make_lags(df, col, lags):
    """Verilen değişken için lag kolonları üretir: L1_col, L2_col, ..."""
    out = pd.DataFrame(index=df.index)
    for L in lags:
        out[f"L{L}_{col}"] = df[col].shift(L)
    return out


def build_regression_matrix(prepared_df, spec, config):
    """Model spesifikasyonuna göre y, X ve 'terim' meta bilgisini üretir.

    Dönen `terms` listesi forecast aşamasında X satırını yeniden kurmak için
    kullanılır. Her terim: {name, tcol (transform edilmiş kolon), base (orijinal
    değişken), lag, is_y}.
    """
    meta = prepared_df.attrs["meta"]
    b2m = meta["base_to_modeled"]
    target_modeled = meta["target_modeled"]

    terms = []
    # Hedef değişken lagleri
    for L in spec.get("lags_y", []):
        terms.append({"name": f"L{L}_{target_modeled}", "tcol": target_modeled,
                      "base": meta["target_base"], "lag": int(L), "is_y": True})
    # Bağımsız değişkenler (ve lagleri)
    for base in spec.get("exog", []):
        tcol = b2m.get(base, base)
        if tcol not in prepared_df.columns:
            raise KeyError(f"Bagimsiz degisken '{base}' (kolon '{tcol}') veride bulunamadi.")
        lags = spec.get("lags_x", {}).get(base, [0])
        for L in lags:
            name = tcol if L == 0 else f"L{L}_{tcol}"
            terms.append({"name": name, "tcol": tcol, "base": base, "lag": int(L), "is_y": False})

    # X matrisini kur
    X = pd.DataFrame(index=prepared_df.index)
    for t in terms:
        X[t["name"]] = prepared_df[t["tcol"]].shift(t["lag"])
    y = prepared_df[target_modeled].copy()

    add_constant = bool(spec.get("add_constant", True))
    return {"y": y, "X": X, "terms": terms, "add_constant": add_constant,
            "target_modeled": target_modeled}


def align_xy(y, X):
    """y ve X'i ortak (NaN'sız) gözlemlere hizalar."""
    data = pd.concat([y.rename("__y__"), X], axis=1).dropna()
    return data["__y__"], data.drop(columns="__y__")


# --- p-value yorumu ---
def interpret_pvalue(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "hesaplanamadi"
    if p < 0.01:
        return "guclu anlamli"
    if p < 0.05:
        return "istatistiksel olarak anlamli"
    if p < 0.10:
        return "zayif anlamli"
    return "anlamli degil"


# --- Beklenen işaret kontrolü ---
def _strip_lag_prefix(name):
    import re
    return re.sub(r"^L\d+_", "", str(name))


def expected_sign_for(var_name, expected_signs):
    base = _strip_lag_prefix(var_name)
    if base in expected_signs:
        return expected_signs[base]
    if var_name in expected_signs:
        return expected_signs[var_name]
    return None


def check_expected_signs(coef_df, expected_signs):
    """Katsayı işaretlerinin beklentiyle uyumunu kontrol eder."""
    total, ok = 0, 0
    signs_ok = []
    for _, row in coef_df.iterrows():
        var = row["variable"]
        exp = expected_sign_for(var, expected_signs)
        if exp is None or var == "const" or pd.isna(row.get("coef")):
            signs_ok.append(None)
            continue
        total += 1
        good = (row["coef"] > 0 and exp == "+") or (row["coef"] < 0 and exp == "-")
        signs_ok.append(bool(good))
        ok += int(good)
    if total == 0:
        return None, np.nan, signs_ok
    return (ok == total), ok / total, signs_ok


def significant_coef_ratio(coef_df, alpha=0.05):
    sub = coef_df[coef_df["variable"] != "const"]
    p = pd.to_numeric(sub["p_value"], errors="coerce").dropna()
    if len(p) == 0:
        return np.nan
    return float((p < alpha).mean())

### 4.1 Tanı testleri ve VIF

Otokorelasyon (Breusch-Godfrey, Ljung-Box, Durbin-Watson), değişen varyans
(Breusch-Pagan, White), normallik (Jarque-Bera), spesifikasyon (Ramsey RESET)
ve çoklu doğrusal bağlantı (VIF). Küçük örneklemde bazı testler çalışmayabilir;
bu durumda `NaN` döner ama süreç durmaz.

In [ ]:
def full_diagnostics(res, config):
    """statsmodels OLS results nesnesinden tam tanı testi seti (p-value'lar)."""
    d = {k: np.nan for k in ["bg_pvalue", "ljungbox_pvalue", "bp_pvalue",
                             "white_pvalue", "jb_pvalue", "reset_pvalue", "durbin_watson"]}
    resid = np.asarray(res.resid)
    n = int(res.nobs)
    hac = int(config.get("hac_lags", 4)) or 4
    try:
        bg = acorr_breusch_godfrey(res, nlags=min(hac, max(1, n // 4)))
        d["bg_pvalue"] = float(bg[1])
    except Exception:
        pass
    try:
        lags = min(10, max(1, n // 5))
        lb = acorr_ljungbox(resid, lags=[lags], return_df=True)
        d["ljungbox_pvalue"] = float(lb["lb_pvalue"].iloc[-1])
    except Exception:
        pass
    try:
        bp = het_breuschpagan(resid, res.model.exog)
        d["bp_pvalue"] = float(bp[1])
    except Exception:
        pass
    try:
        wh = het_white(resid, res.model.exog)
        d["white_pvalue"] = float(wh[1])
    except Exception:
        pass
    try:
        jb = jarque_bera(resid)
        d["jb_pvalue"] = float(jb[1])
    except Exception:
        pass
    try:
        d["durbin_watson"] = float(durbin_watson(resid))
    except Exception:
        pass
    try:
        rs = linear_reset(res, power=2, use_f=True)
        d["reset_pvalue"] = float(rs.pvalue)
    except Exception:
        pass
    return d


def residual_only_diagnostics(resid):
    """Sadece residual gerektiren testler (TS modelleri: SARIMAX/VAR/VECM)."""
    d = {k: np.nan for k in ["bg_pvalue", "ljungbox_pvalue", "bp_pvalue",
                             "white_pvalue", "jb_pvalue", "reset_pvalue", "durbin_watson"]}
    resid = np.asarray(pd.Series(resid).dropna())
    n = len(resid)
    if n < 5:
        return d
    try:
        lags = min(10, max(1, n // 5))
        lb = acorr_ljungbox(resid, lags=[lags], return_df=True)
        d["ljungbox_pvalue"] = float(lb["lb_pvalue"].iloc[-1])
    except Exception:
        pass
    try:
        jb = jarque_bera(resid)
        d["jb_pvalue"] = float(jb[1])
    except Exception:
        pass
    try:
        d["durbin_watson"] = float(durbin_watson(resid))
    except Exception:
        pass
    return d


def compute_vif(X):
    """Her değişken için VIF tablosu üretir (const hariç)."""
    if X is None or X.shape[1] == 0:
        return pd.DataFrame(columns=["variable", "VIF"])
    Xc = sm.add_constant(X, has_constant="add")
    rows = []
    cols = list(Xc.columns)
    for i, col in enumerate(cols):
        if col == "const":
            continue
        try:
            v = float(variance_inflation_factor(Xc.values, i))
        except Exception:
            v = np.nan
        rows.append({"variable": col, "VIF": v})
    return pd.DataFrame(rows)


def interpret_diagnostics(diag):
    """Tanı testlerini kısa Türkçe cümlelere çevirir."""
    c = {}
    bg = diag.get("bg_pvalue")
    c["bg"] = ("otokorelasyon bulgusu yok" if (bg is not None and not np.isnan(bg) and bg >= 0.05)
               else ("otokorelasyon riski var" if (bg is not None and not np.isnan(bg)) else "BG hesaplanamadi"))
    lb = diag.get("ljungbox_pvalue")
    c["ljungbox"] = ("Ljung-Box: artiklarda otokorelasyon yok" if (lb is not None and not np.isnan(lb) and lb >= 0.05)
                     else ("Ljung-Box: artiklarda otokorelasyon var" if (lb is not None and not np.isnan(lb)) else "Ljung-Box hesaplanamadi"))
    bp = diag.get("bp_pvalue")
    c["bp"] = ("degisen varyans bulgusu yok" if (bp is not None and not np.isnan(bp) and bp >= 0.05)
               else ("degisen varyans riski var" if (bp is not None and not np.isnan(bp)) else "BP hesaplanamadi"))
    wh = diag.get("white_pvalue")
    c["white"] = ("White: sabit varyans reddedilemiyor" if (wh is not None and not np.isnan(wh) and wh >= 0.05)
                  else ("White: degisen varyans riski var" if (wh is not None and not np.isnan(wh)) else "White hesaplanamadi"))
    jb = diag.get("jb_pvalue")
    c["jb"] = ("residual normal dagiliyor" if (jb is not None and not np.isnan(jb) and jb >= 0.05)
               else ("residual normal dagilmiyor" if (jb is not None and not np.isnan(jb)) else "JB hesaplanamadi"))
    rs = diag.get("reset_pvalue")
    c["reset"] = ("model spesifikasyonu uygun gorunuyor" if (rs is not None and not np.isnan(rs) and rs >= 0.05)
                  else ("model spesifikasyonu sorunlu olabilir" if (rs is not None and not np.isnan(rs)) else "RESET hesaplanamadi"))
    dw = diag.get("durbin_watson")
    if dw is not None and not np.isnan(dw):
        if dw < 1.5:
            c["dw"] = f"Durbin-Watson={dw:.2f}: pozitif otokorelasyon isareti"
        elif dw > 2.5:
            c["dw"] = f"Durbin-Watson={dw:.2f}: negatif otokorelasyon isareti"
        else:
            c["dw"] = f"Durbin-Watson={dw:.2f}: otokorelasyon isareti zayif"
    else:
        c["dw"] = "Durbin-Watson hesaplanamadi"
    return c


def interpret_vif(max_vif, threshold=10.0):
    if max_vif is None or np.isnan(max_vif):
        return "VIF hesaplanamadi"
    if max_vif > threshold:
        return f"coklu dogrusal baglanti riski yuksek (max VIF={max_vif:.1f})"
    return f"coklu dogrusal baglanti dusuk (max VIF={max_vif:.1f})"


def calculate_clean_score(diag, config, expected_ok, insignificant_share, mape_test, max_vif):
    """Modelin genel sağlığını 0-100 arası tek skora indirger."""
    cs = config["clean_score"]
    score = float(cs["start"])
    reasons = []

    def bad(key):
        v = diag.get(key)
        return (v is not None) and (not np.isnan(v)) and (v < 0.05)

    if bad("bg_pvalue"):
        score -= cs["bg"]; reasons.append("otokorelasyon (BG)")
    if bad("bp_pvalue"):
        score -= cs["bp"]; reasons.append("degisen varyans (BP)")
    if bad("white_pvalue"):
        score -= cs["white"]; reasons.append("degisen varyans (White)")
    if bad("reset_pvalue"):
        score -= cs["reset"]; reasons.append("spesifikasyon (RESET)")
    if max_vif is not None and not np.isnan(max_vif) and max_vif > cs["vif_threshold"]:
        score -= cs["vif"]; reasons.append("coklu dogrusal baglanti (VIF)")
    if insignificant_share is not None and not np.isnan(insignificant_share) and insignificant_share > cs["insignificant_share"]:
        score -= cs["insignificant"]; reasons.append("katsayilarin cogu anlamsiz")
    if expected_ok is False:
        score -= cs["sign"]; reasons.append("isaret beklentisi karsilanmiyor")
    if mape_test is not None and not np.isnan(mape_test) and mape_test > cs["mape_bad"]:
        score -= cs["forecast"]; reasons.append("forecast performansi zayif")

    return max(0.0, score), reasons


def make_failed_result(model_name, model_type, error):
    """Standart 'failed' sonuç sözlüğü."""
    return {
        "model_name": model_name, "model_type": model_type, "status": "failed",
        "fit_object": None, "coefficients": pd.DataFrame(),
        "diagnostics": {}, "vif": pd.DataFrame(), "metrics": {},
        "forecast": pd.DataFrame(), "comments": {}, "error": str(error), "_meta": {},
    }


def safe_fit_model(fit_fn, model_name, model_type):
    """Model fit'ini güvenli çalıştırır; hata olursa 'failed' sonuç döner."""
    try:
        return fit_fn()
    except Exception as e:
        import traceback
        return make_failed_result(model_name, model_type,
                                  f"{type(e).__name__}: {e} | {traceback.format_exc().splitlines()[-1]}")

## 5. Model aileleri

Her model **standart bir sonuç sözlüğü** döndürür:
`{model_name, model_type, status, fit_object, coefficients, diagnostics, vif,
metrics, forecast, comments, error, _meta}`. `_meta`, forecast aşamasının
ihtiyaç duyduğu tahmin fonksiyonu ve terim yapısını taşır.

Tüm regresyon tabanlı modeller (OLS/ADL/DL/ARDL/Ridge/Lasso/ElasticNet) ortak
bir **recursive (özyinelemeli) forecast** altyapısını paylaşır; bu altyapı hem
test seti backtest'i hem de gerçek forecast için kullanılır. ECM, SARIMAX, VAR
ve VECM kendi tahmin mantıklarını kullanır.

In [ ]:
# ---------- Ortak: seviye geri-dönüşümü ve recursive forecaster ----------
def invert_transform_array(prev_level, modeled, transform):
    prev_level = np.asarray(prev_level, float)
    modeled = np.asarray(modeled, float)
    if transform == "level":
        return modeled
    if transform == "log":
        return np.exp(modeled)
    if transform == "diff":
        return prev_level + modeled
    if transform == "logdiff":
        return prev_level * np.exp(modeled)
    if transform == "pct_change":
        return prev_level * (1.0 + modeled)
    return modeled


def make_linreg_predictor(params, add_constant):
    def predict(fvals):
        yhat = params.get("const", 0.0) if add_constant else 0.0
        for name, val in fvals.items():
            yhat += params.get(name, 0.0) * val
        return float(yhat)
    return predict


def make_sklearn_predictor(pipe, feature_order):
    def predict(fvals):
        x = np.array([[fvals[c] for c in feature_order]], float)
        return float(pipe.predict(x)[0])
    return predict


def recursive_linreg_forecast(meta_lr, exog_series, mod_init, lvl_init, n_hist):
    """Lagli regresyon modelleri için özyinelemeli forecast (seviye + modellenmiş)."""
    terms = meta_lr["terms"]
    tr = meta_lr["target_transform"]
    predict = meta_lr["predict_scalar"]
    N = len(mod_init)
    mod = np.array(mod_init, float)
    lvl = np.array(lvl_init, float)
    for i in range(n_hist, N):
        fvals, ok = {}, True
        for t in terms:
            src = mod if t["is_y"] else exog_series.get(t["tcol"])
            if src is None:
                ok = False; break
            j = i - t["lag"]
            if j < 0 or j >= N or np.isnan(src[j]):
                ok = False; break
            fvals[t["name"]] = src[j]
        if not ok:
            mod[i] = np.nan; lvl[i] = np.nan; continue
        yhat = predict(fvals)
        mod[i] = yhat
        prev = lvl[i - 1] if i - 1 >= 0 else np.nan
        lvl[i] = float(invert_transform_step(prev, yhat, tr))
    return mod, lvl


def _cov_kwargs(config):
    ct = config.get("cov_type", "nonrobust")
    if ct in (None, "nonrobust"):
        return {}
    if ct == "HAC":
        return {"cov_type": "HAC", "cov_kwds": {"maxlags": int(config.get("hac_lags", 4))}}
    return {"cov_type": ct}


def coef_table_from_sm(res, expected_signs):
    """statsmodels sonuç nesnesinden standart katsayı tablosu."""
    params = res.params
    try:
        ci = res.conf_int()
        ci.columns = [0, 1]
    except Exception:
        ci = None
    rows = []
    for name in params.index:
        try:
            p = float(res.pvalues[name])
        except Exception:
            p = np.nan
        try:
            se = float(res.bse[name])
        except Exception:
            se = np.nan
        try:
            tv = float(res.tvalues[name])
        except Exception:
            tv = np.nan
        lo = float(ci.loc[name, 0]) if ci is not None else np.nan
        hi = float(ci.loc[name, 1]) if ci is not None else np.nan
        rows.append({"variable": str(name), "coef": float(params[name]), "std_error": se,
                     "t_stat": tv, "p_value": p, "conf_low": lo, "conf_high": hi,
                     "significance_comment": interpret_pvalue(p)})
    df = pd.DataFrame(rows)
    _, _, signs_ok = check_expected_signs(df, expected_signs)
    df["expected_sign"] = [expected_sign_for(v, expected_signs) for v in df["variable"]]
    df["sign_ok"] = signs_ok
    return df


def _level_train_metrics(fitted_mod, y_index, prepared_df, tr, target_base):
    prev = prepared_df[target_base].shift(1).reindex(y_index).values
    actual = prepared_df[target_base].reindex(y_index).values
    lvlfit = invert_transform_array(prev, np.asarray(fitted_mod, float), tr)
    return _rmse(actual, lvlfit), _mae(actual, lvlfit), _mape(actual, lvlfit)


def _gaussian_ic(rss, n, k):
    """statsmodels OLS ile birebir aynı Gauss olabilirlik AIC/BIC (aynı ölçek).

    Böylece sklearn/VAR/VECM gibi native IC'si farklı ölçekte olan modeller,
    OLS/SARIMAX ile karşılaştırılabilir hale gelir.
    """
    if rss is None or n is None or rss <= 0 or n <= 0:
        return np.nan, np.nan
    base = n * np.log(rss / n) + n * (1.0 + np.log(2.0 * np.pi))
    return base + 2 * k, base + k * np.log(n)


def _metrics_dict(r2, adj, aic, bic, llf, nobs, tr3, te3, max_vif, sig_ratio, exp_ok, clean):
    return {
        "r2": r2, "adj_r2": adj, "aic": aic, "bic": bic, "loglik": llf, "nobs": nobs,
        "rmse_train": tr3[0], "mae_train": tr3[1], "mape_train": tr3[2],
        "rmse_test": te3[0], "mae_test": te3[1], "mape_test": te3[2],
        "max_vif": max_vif, "significant_coef_ratio": sig_ratio,
        "expected_signs_ok": exp_ok, "clean_score": clean,
    }


def build_comments(diag, vif, config, model_type, extra=None):
    dc = interpret_diagnostics(diag)
    max_vif = float(vif["VIF"].max()) if len(vif) > 0 else np.nan
    dc["vif"] = interpret_vif(max_vif, config["clean_score"]["vif_threshold"])
    if extra:
        dc.update(extra)
    return dc


def _make_result(model_name, model_type, res_obj, coef_df, diag, vif, metrics, comments, meta_obj):
    return {"model_name": model_name, "model_type": model_type, "status": "success",
            "fit_object": res_obj, "coefficients": coef_df, "diagnostics": diag, "vif": vif,
            "metrics": metrics, "forecast": pd.DataFrame(), "comments": comments,
            "error": None, "_meta": meta_obj}

### 5.1–5.4 OLS / ADL / Distributed Lag / (ARDL çekirdeği)

Bu modeller tek bir ortak `_fit_regression` fonksiyonuyla kurulur. Fark
yalnızca `CONFIG` içindeki lag yapısıdır. `cov_type` ile robust (HC0/HC1/HC3/
HAC) standart hatalar desteklenir.

In [ ]:
def _pipeline(mode, alpha, l1):
    if "ridge" in mode:
        est = Ridge(alpha=alpha)
    elif "lasso" in mode:
        est = Lasso(alpha=alpha, max_iter=100000)
    else:
        est = ElasticNet(alpha=alpha, l1_ratio=(l1 if l1 else 0.5), max_iter=100000)
    return Pipeline([("scaler", StandardScaler()), ("model", est)])


def _cv_fit_sklearn(mode, X, y, config, spec):
    alphas = spec.get("alpha_grid", [0.01, 0.1, 1, 10])
    l1s = spec.get("l1_ratio_grid", [0.5]) if mode.endswith("en") else [None]
    n = len(y)
    test_size = config.get("test_size", 12) or 12
    nsplit = int(min(5, max(2, n // max(1, test_size))))
    tscv = TimeSeriesSplit(n_splits=nsplit)
    best = None
    for a in alphas:
        for l1 in l1s:
            errs = []
            for tr_i, te_i in tscv.split(X):
                pipe = _pipeline(mode, a, l1)
                pipe.fit(X[tr_i], y[tr_i])
                errs.append(_rmse(y[te_i], pipe.predict(X[te_i])))
            m = np.nanmean(errs)
            if best is None or m < best[0]:
                best = (m, a, l1)
    _, a, l1 = best
    pipe = _pipeline(mode, a, l1)
    pipe.fit(X, y)
    pipe._best_alpha, pipe._best_l1 = a, l1
    return pipe


def coef_table_sklearn(pipe, feat_cols, add_c, expected_signs):
    scaler = pipe.named_steps["scaler"]
    est = pipe.named_steps["model"]
    scale = np.where(scaler.scale_ == 0, 1.0, scaler.scale_)
    coef_scaled = np.ravel(est.coef_)
    coef_orig = coef_scaled / scale
    rows = []
    if add_c:
        const = float(est.intercept_) - float(np.sum(coef_scaled * scaler.mean_ / scale))
        rows.append({"variable": "const", "coef": const, "std_error": np.nan, "t_stat": np.nan,
                     "p_value": np.nan, "conf_low": np.nan, "conf_high": np.nan,
                     "significance_comment": "sabit terim"})
    for c, co in zip(feat_cols, coef_orig):
        comment = "sifira dusuruldu" if abs(co) < 1e-10 else "duzenlilestirilmis katsayi (p-value yok)"
        rows.append({"variable": c, "coef": float(co), "std_error": np.nan, "t_stat": np.nan,
                     "p_value": np.nan, "conf_low": np.nan, "conf_high": np.nan,
                     "significance_comment": comment})
    df = pd.DataFrame(rows)
    _, _, signs_ok = check_expected_signs(df, expected_signs)
    df["expected_sign"] = [expected_sign_for(v, expected_signs) for v in df["variable"]]
    df["sign_ok"] = signs_ok
    return df


def _backtest_regression(prepared_df, config, spec, meta, terms, tr, fit_predictor):
    test_size = config.get("test_size", 0) or 0
    n = len(prepared_df)
    if test_size <= 0 or n <= test_size + 5:
        return np.nan, np.nan, np.nan
    try:
        train_df = prepared_df.iloc[:-test_size].copy()
        train_df.attrs["meta"] = meta
        pred_train, _ = fit_predictor(train_df)
        n_hist = len(train_df)
        tb, tm = meta["target_base"], meta["target_modeled"]
        mod_init = prepared_df[tm].values.astype(float).copy(); mod_init[n_hist:] = np.nan
        lvl_init = prepared_df[tb].values.astype(float).copy(); lvl_init[n_hist:] = np.nan
        exog_series = {t["tcol"]: prepared_df[t["tcol"]].values.astype(float)
                       for t in terms if not t["is_y"]}
        meta_bt = {"terms": terms, "target_transform": tr, "predict_scalar": pred_train}
        _, lvl = recursive_linreg_forecast(meta_bt, exog_series, mod_init, lvl_init, n_hist)
        actual = prepared_df[tb].values[n_hist:]
        return _rmse(actual, lvl[n_hist:]), _mae(actual, lvl[n_hist:]), _mape(actual, lvl[n_hist:])
    except Exception:
        return np.nan, np.nan, np.nan


def _fit_regression(prepared_df, config, spec, model_type, mode):
    """OLS-ailesi (mode='ols') ve sklearn-ailesi (mode='sklearn:*') ortak fit'i."""
    meta = prepared_df.attrs["meta"]
    tr, tb = meta["target_transform"], meta["target_base"]
    exp_signs = config.get("expected_signs", {})

    bm = build_regression_matrix(prepared_df, spec, config)
    y_full, X_full, terms, add_c = bm["y"], bm["X"], bm["terms"], bm["add_constant"]
    y, X = align_xy(y_full, X_full)
    if X.shape[1] == 0 and not add_c:
        raise ValueError("Model icin regresor bulunamadi (exog/lags bos).")
    feat_cols = list(X.columns)

    def fit_predictor(df):
        bmi = build_regression_matrix(df, spec, config)
        yy, XX = align_xy(bmi["y"], bmi["X"])
        if feat_cols:
            XX = XX.reindex(columns=feat_cols)
        if mode == "ols":
            Xc = sm.add_constant(XX, has_constant="add") if add_c else XX
            r = sm.OLS(yy, Xc).fit(**_cov_kwargs(config))
            return make_linreg_predictor(r.params.to_dict(), add_c), r
        else:
            pipe = _cv_fit_sklearn(mode, XX.values, yy.values, config, spec)
            return make_sklearn_predictor(pipe, feat_cols), pipe

    predict_full, fit_full = fit_predictor(prepared_df)

    if mode == "ols":
        res = fit_full
        coef_df = coef_table_from_sm(res, exp_signs)
        diag = full_diagnostics(res, config)
        r2, adj = float(res.rsquared), float(res.rsquared_adj)
        aic, bic, llf, nobs = float(res.aic), float(res.bic), float(res.llf), int(res.nobs)
        fitted_mod = res.fittedvalues
        fit_obj = res
    else:
        pipe = fit_full
        Xc = sm.add_constant(X, has_constant="add")
        aux = sm.OLS(y, Xc).fit()
        diag = full_diagnostics(aux, config)
        coef_df = coef_table_sklearn(pipe, feat_cols, add_c, exp_signs)
        fitted_mod = pd.Series(pipe.predict(X.values), index=X.index)
        resid = y.values - fitted_mod.values
        rss = float(np.sum(resid ** 2))
        sst = float(np.sum((y.values - y.values.mean()) ** 2))
        r2 = 1 - rss / sst if sst > 0 else np.nan
        nobs = len(y)
        k = int(np.sum(np.abs(coef_df.loc[coef_df["variable"] != "const", "coef"]) > 1e-10)) + (1 if add_c else 0)
        aic, bic = _gaussian_ic(rss, nobs, k)
        adj = 1 - (1 - r2) * (nobs - 1) / (nobs - k - 1) if (nobs - k - 1) > 0 else np.nan
        llf = np.nan
        fit_obj = pipe

    vif = compute_vif(X)
    max_vif = float(vif["VIF"].max()) if len(vif) > 0 else np.nan
    tr3 = _level_train_metrics(fitted_mod, X.index, prepared_df, tr, tb)
    te3 = _backtest_regression(prepared_df, config, spec, meta, terms, tr, fit_predictor)

    exp_ok, _, _ = check_expected_signs(coef_df, exp_signs)
    sig_ratio = significant_coef_ratio(coef_df)
    insig = (1 - sig_ratio) if not np.isnan(sig_ratio) else np.nan
    clean, reasons = calculate_clean_score(diag, config, exp_ok, insig, te3[2], max_vif)

    metrics = _metrics_dict(r2, adj, aic, bic, llf, nobs, tr3, te3, max_vif, sig_ratio, exp_ok, clean)
    comments = build_comments(diag, vif, config, model_type,
                              extra={"clean_reasons": "; ".join(reasons) if reasons else "temiz"})
    meta_lr = {"kind": "linreg", "predict_scalar": predict_full, "terms": terms,
               "target_transform": tr, "add_constant": add_c, "feat_cols": feat_cols}
    return _make_result(model_type, model_type, fit_obj, coef_df, diag, vif, metrics, comments, meta_lr)


def fit_linreg_family(model_name, prepared_df, config, spec):
    return _fit_regression(prepared_df, config, spec, model_name, "ols")

### 5.5 ARDL (otomatik lag seçimi)

`max_lag_y` / `max_lag_x` sınırları içinde AIC/BIC ile en iyi lag kombinasyonu
aranır; seçilen yapı ADL biçiminde kurulur (böylece tanı testleri ve forecast
altyapısı aynen kullanılır). statsmodels `ARDL` mevcutsa çekirdek doğrulama
için kullanılabilir.

In [ ]:
def fit_ardl(prepared_df, config, spec):
    exog = spec.get("exog", [])
    maxp, maxq = int(spec.get("max_lag_y", 4)), int(spec.get("max_lag_x", 4))
    crit = spec.get("selection_criterion", "bic")
    best = None
    for p in range(0, maxp + 1):
        for q in range(0, maxq + 1):
            trial = {"exog": exog, "lags_y": list(range(1, p + 1)),
                     "lags_x": {b: list(range(0, q + 1)) for b in exog}, "add_constant": True}
            try:
                bm = build_regression_matrix(prepared_df, trial, config)
                yy, XX = align_xy(bm["y"], bm["X"])
                if len(yy) < XX.shape[1] + 3:
                    continue
                r = sm.OLS(yy, sm.add_constant(XX, has_constant="add")).fit()
                ic = float(r.bic) if crit == "bic" else float(r.aic)
                if best is None or ic < best[0]:
                    best = (ic, p, q, trial)
            except Exception:
                continue
    if best is None:
        raise ValueError("ARDL icin uygun lag kombinasyonu bulunamadi (ornek cok kucuk olabilir).")
    _, p, q, trial = best
    res = _fit_regression(prepared_df, config, trial, "ARDL", "ols")
    res["comments"]["ardl"] = f"ARDL lag secimi ({crit.upper()}): p(y)={p}, q(x)={q}."
    res["metrics"]["ardl_p"], res["metrics"]["ardl_q"] = p, q
    return res

### 5.6 ECM (Engle-Granger, iki aşamalı)

1) Uzun dönem: `y = c + b·x + u` (seviyelerde).
2) Kısa dönem: `Δy = c + λ·u_{t-1} + Δy lagleri + Δx lagleri + e`.
`λ` (hata düzeltme terimi) negatif ve anlamlı ise mekanizma çalışıyor demektir.

In [ ]:
def _ecm_estimate(df, spec, config):
    meta = df.attrs["meta"]
    tb = meta["target_base"]
    level_vars = spec.get("level_vars", [])
    diff_vars = spec.get("diff_vars", [])

    lr_X = pd.DataFrame({v: df[v] for v in level_vars}, index=df.index)
    yy, XX = align_xy(df[tb], lr_X)
    lr_res = sm.OLS(yy, sm.add_constant(XX, has_constant="add")).fit()
    u = df[tb] - lr_res.predict(sm.add_constant(lr_X, has_constant="add"))

    dy = df[tb].diff()
    sr = pd.DataFrame(index=df.index)
    sr["ECT_L1"] = u.shift(1)
    for L in spec.get("lags_y", [1]):
        sr[f"L{L}_d_{tb}"] = dy.shift(L)
    dx_terms = []
    for base in diff_vars:
        dxs = df[base].diff()
        for L in spec.get("lags_x", {}).get(base, [0]):
            nm = f"d_{base}" if L == 0 else f"L{L}_d_{base}"
            sr[nm] = dxs.shift(L)
            dx_terms.append({"base": base, "lag": L, "name": nm})
    dyy, srX = align_xy(dy, sr)
    sr_res = sm.OLS(dyy, sm.add_constant(srX, has_constant="add")).fit(**_cov_kwargs(config))

    meta_ecm = {"kind": "ecm", "lr_params": lr_res.params.to_dict(), "level_vars": level_vars,
                "sr_params": sr_res.params.to_dict(), "lags_y": spec.get("lags_y", [1]),
                "dx_terms": dx_terms, "target_base": tb, "add_constant": True}
    return lr_res, sr_res, srX, dyy, u, meta_ecm


def ecm_forecast(meta, x_level, y_level_init, n_hist):
    lrp, sp = meta["lr_params"], meta["sr_params"]
    lv, lags_y, dxt, tb = meta["level_vars"], meta["lags_y"], meta["dx_terms"], meta["target_base"]
    y = np.array(y_level_init, float)
    N = len(y)

    def u_at(i):
        val = lrp.get("const", 0.0)
        for v in lv:
            val += lrp.get(v, 0.0) * x_level[v][i]
        return y[i] - val

    for i in range(n_hist, N):
        if np.isnan(y[i - 1]):
            y[i] = np.nan; continue
        pred = sp.get("const", 0.0) + sp.get("ECT_L1", 0.0) * u_at(i - 1)
        for L in lags_y:
            j = i - L
            dyv = 0.0 if j - 1 < 0 else (y[j] - y[j - 1])
            pred += sp.get(f"L{L}_d_{tb}", 0.0) * dyv
        for t in dxt:
            j = i - t["lag"]
            dxv = 0.0 if j - 1 < 0 else (x_level[t["base"]][j] - x_level[t["base"]][j - 1])
            pred += sp.get(t["name"], 0.0) * dxv
        y[i] = y[i - 1] + pred
    return y


def fit_ecm(prepared_df, config, spec):
    meta = prepared_df.attrs["meta"]
    tb = meta["target_base"]
    exp_signs = config.get("expected_signs", {})
    lr_res, sr_res, srX, dyy, u, meta_ecm = _ecm_estimate(prepared_df, spec, config)

    # Engle-Granger: uzun dönem artığına ADF
    try:
        adf = adfuller(u.dropna(), autolag="AIC")
        eg_p = float(adf[1])
    except Exception:
        eg_p = np.nan

    coef_sr = coef_table_from_sm(sr_res, exp_signs)
    coef_lr = coef_table_from_sm(lr_res, exp_signs)
    coef_lr["variable"] = ["LR_" + v for v in coef_lr["variable"]]
    coef_all = pd.concat([coef_sr, coef_lr], ignore_index=True)

    ect = sr_res.params.get("ECT_L1", np.nan)
    ect_p = sr_res.pvalues.get("ECT_L1", np.nan)
    if not np.isnan(ect) and ect < 0 and not np.isnan(ect_p) and ect_p < 0.05:
        ect_comment = f"Hata duzeltme terimi lambda={ect:.3f} negatif ve anlamli (p={ect_p:.3f}): ECM mekanizmasi calisiyor."
    elif not np.isnan(ect) and ect < 0:
        ect_comment = f"Hata duzeltme terimi lambda={ect:.3f} negatif fakat zayif/anlamsiz (p={ect_p:.3f})."
    else:
        ect_comment = "Hata duzeltme terimi negatif degil: ECM mekanizmasi teorik olarak zayif veya hatali olabilir."
    eg_comment = ("Engle-Granger: uzun donem iliskisi esbutunlesik (durgun artik)."
                  if (not np.isnan(eg_p) and eg_p < 0.05)
                  else "Engle-Granger: esbutunlesme kaniti zayif (artik durgun olmayabilir).")

    diag = full_diagnostics(sr_res, config)
    vif = compute_vif(srX)
    max_vif = float(vif["VIF"].max()) if len(vif) > 0 else np.nan

    # Metrikler (Δy -> seviye)
    fitted_dy = sr_res.fittedvalues
    idx = fitted_dy.index
    prev = prepared_df[tb].shift(1).reindex(idx).values
    actual_lvl = prepared_df[tb].reindex(idx).values
    fit_lvl = prev + fitted_dy.values
    tr3 = (_rmse(actual_lvl, fit_lvl), _mae(actual_lvl, fit_lvl), _mape(actual_lvl, fit_lvl))

    # Backtest
    te3 = (np.nan, np.nan, np.nan)
    test_size = config.get("test_size", 0) or 0
    if test_size > 0 and len(prepared_df) > test_size + 5:
        try:
            train_df = prepared_df.iloc[:-test_size].copy(); train_df.attrs["meta"] = meta
            _, _, _, _, _, meta_bt = _ecm_estimate(train_df, spec, config)
            n_hist = len(train_df)
            x_level = {v: prepared_df[v].values.astype(float) for v in meta_ecm["level_vars"]}
            y_init = prepared_df[tb].values.astype(float).copy(); y_init[n_hist:] = np.nan
            y_init[n_hist - 1] = prepared_df[tb].values[n_hist - 1]
            yhat = ecm_forecast(meta_bt, x_level, y_init, n_hist)
            actual = prepared_df[tb].values[n_hist:]
            te3 = (_rmse(actual, yhat[n_hist:]), _mae(actual, yhat[n_hist:]), _mape(actual, yhat[n_hist:]))
        except Exception:
            pass

    r2, adj = float(sr_res.rsquared), float(sr_res.rsquared_adj)
    aic, bic, llf, nobs = float(sr_res.aic), float(sr_res.bic), float(sr_res.llf), int(sr_res.nobs)
    exp_ok, _, _ = check_expected_signs(coef_all, exp_signs)
    sig_ratio = significant_coef_ratio(coef_sr)
    insig = (1 - sig_ratio) if not np.isnan(sig_ratio) else np.nan
    clean, reasons = calculate_clean_score(diag, config, exp_ok, insig, te3[2], max_vif)

    metrics = _metrics_dict(r2, adj, aic, bic, llf, nobs, tr3, te3, max_vif, sig_ratio, exp_ok, clean)
    metrics["eg_pvalue"] = eg_p
    metrics["ect_coef"] = float(ect) if not np.isnan(ect) else np.nan
    metrics["ect_pvalue"] = float(ect_p) if not np.isnan(ect_p) else np.nan
    comments = build_comments(diag, vif, config, "ECM",
                              extra={"ect": ect_comment, "cointegration": eg_comment,
                                     "clean_reasons": "; ".join(reasons) if reasons else "temiz"})
    return _make_result("ECM", "ECM", sr_res, coef_all, diag, vif, metrics, comments, meta_ecm)

### 5.7 VAR / 5.8 VECM / 5.9 SARIMAX / Ridge-Lasso-ElasticNet

- **VAR**: lag seçim tablosu, stabilite kontrolü, opsiyonel Granger nedensellik.
- **VECM**: Johansen eşbütünleşme testi, alpha (uyarlama) ve beta (uzun dönem)
  katsayıları.
- **SARIMAX**: `order`, `seasonal_order`, `exog` CONFIG'ten.
- **Ridge/Lasso/ElasticNet**: StandardScaler + TimeSeriesSplit CV ile alpha
  seçimi; Lasso'da sıfıra düşen katsayılar ayrıca raporlanır.

Not: VAR/VECM forecast'ları içsel (senaryodan bağımsız) üretilir; diğer
değişkenler sistem tarafından tahmin edilir.

In [ ]:
def reconstruct_levels(modeled, last_level, tr):
    out = np.empty(len(modeled), float)
    prev = last_level
    for k, m in enumerate(modeled):
        out[k] = float(invert_transform_step(prev, m, tr))
        prev = out[k]
    return out


# ---------------- SARIMAX ----------------
def fit_sarimax(prepared_df, config, spec):
    if not HAVE.get("sarimax"):
        raise RuntimeError("statsmodels SARIMAX bileseni yok.")
    meta = prepared_df.attrs["meta"]
    tr, tb, tm = meta["target_transform"], meta["target_base"], meta["target_modeled"]
    exog_bases = spec.get("exog", [])
    exog_cols = [meta["base_to_modeled"].get(b, b) for b in exog_bases]
    endog = prepared_df[tm]
    if exog_cols:
        data = pd.concat([endog.rename("__y__"), prepared_df[exog_cols]], axis=1).dropna()
        endog_a, exog_a = data["__y__"], data.drop(columns="__y__")
    else:
        endog_a, exog_a = endog.dropna(), None
    order = tuple(spec.get("order", [1, 0, 1]))
    sorder = tuple(spec.get("seasonal_order", [0, 0, 0, 0]))
    res = SARIMAX(endog_a, exog=exog_a, order=order, seasonal_order=sorder,
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

    coef_df = coef_table_from_sm(res, config.get("expected_signs", {}))
    diag = residual_only_diagnostics(res.resid)

    idx = endog_a.index
    prev = prepared_df[tb].shift(1).reindex(idx).values
    fit_lvl = invert_transform_array(prev, res.fittedvalues.reindex(idx).values, tr)
    actual_lvl = prepared_df[tb].reindex(idx).values
    tr3 = (_rmse(actual_lvl, fit_lvl), _mae(actual_lvl, fit_lvl), _mape(actual_lvl, fit_lvl))

    # R2 seviye uzayında, durum-uzayı ısınma (burn-in) gözlemleri atlanarak
    burn = max(3, int(order[0]) + (int(sorder[0]) * int(sorder[3]) if len(sorder) > 3 else 0))
    a_r, f_r = actual_lvl[burn:], fit_lvl[burn:]
    sst = float(np.nansum((a_r - np.nanmean(a_r)) ** 2))
    r2 = 1 - float(np.nansum((a_r - f_r) ** 2)) / sst if sst > 0 else np.nan
    aic, bic, llf, nobs = float(res.aic), float(res.bic), float(res.llf), int(res.nobs)

    te3 = (np.nan, np.nan, np.nan)
    test_size = config.get("test_size", 0) or 0
    if test_size > 0 and len(endog_a) > test_size + 5:
        try:
            tr_endog = endog_a.iloc[:-test_size]
            tr_exog = exog_a.iloc[:-test_size] if exog_a is not None else None
            te_exog = exog_a.iloc[-test_size:] if exog_a is not None else None
            rt = SARIMAX(tr_endog, exog=tr_exog, order=order, seasonal_order=sorder,
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            fc_mod = np.asarray(rt.get_forecast(steps=test_size, exog=te_exog).predicted_mean)
            last_level = prepared_df[tb].reindex(endog_a.index).values[-test_size - 1]
            fc_lvl = reconstruct_levels(fc_mod, last_level, tr)
            actual = prepared_df[tb].reindex(endog_a.index).values[-test_size:]
            te3 = (_rmse(actual, fc_lvl), _mae(actual, fc_lvl), _mape(actual, fc_lvl))
        except Exception:
            pass

    exp_ok, _, _ = check_expected_signs(coef_df, config.get("expected_signs", {}))
    sig_ratio = significant_coef_ratio(coef_df)
    insig = (1 - sig_ratio) if not np.isnan(sig_ratio) else np.nan
    clean, reasons = calculate_clean_score(diag, config, exp_ok, insig, te3[2], np.nan)
    metrics = _metrics_dict(r2, np.nan, aic, bic, llf, nobs, tr3, te3, np.nan, sig_ratio, exp_ok, clean)
    comments = interpret_diagnostics(diag)
    comments["model"] = f"SARIMAX order={order}, seasonal={sorder}, exog={exog_bases}."
    comments["clean_reasons"] = "; ".join(reasons) if reasons else "temiz"
    meta_sx = {"kind": "sarimax", "res": res, "exog_cols": exog_cols, "target_transform": tr,
               "target_base": tb, "order": order, "seasonal_order": sorder}
    return _make_result("SARIMAX", "SARIMAX", res, coef_df, diag, pd.DataFrame(), metrics, comments, meta_sx)


# ---------------- VAR ----------------
def var_coef_table(res, target, config):
    exp = config.get("expected_signs", {})
    params, ses = res.params[target], res.stderr[target]
    tv, pv = res.tvalues[target], res.pvalues[target]
    rows = []
    for name in params.index:
        co, se, p = float(params[name]), float(ses[name]), float(pv[name])
        rows.append({"variable": str(name), "coef": co, "std_error": se, "t_stat": float(tv[name]),
                     "p_value": p, "conf_low": co - 1.96 * se, "conf_high": co + 1.96 * se,
                     "significance_comment": interpret_pvalue(p)})
    df = pd.DataFrame(rows)
    _, _, signs_ok = check_expected_signs(df, exp)
    df["expected_sign"] = [expected_sign_for(v, exp) for v in df["variable"]]
    df["sign_ok"] = signs_ok
    return df


def fit_var(prepared_df, config, spec):
    if not HAVE.get("var"):
        raise RuntimeError("statsmodels VAR bileseni yok.")
    meta = prepared_df.attrs["meta"]
    vars_ = spec.get("vars", [])
    if len(vars_) < 2:
        raise ValueError("VAR icin en az 2 degisken gerekir (CONFIG['model_specs']['VAR']['vars']).")
    endog = prepared_df[vars_].dropna()
    target = meta["target_base"] if meta["target_base"] in vars_ else vars_[0]
    tpos = vars_.index(target)
    model = VAR(endog)
    mm = max(1, min(int(spec.get("maxlags", 6)), len(endog) // 3))

    sel_rows = []
    for L in range(1, mm + 1):
        try:
            rL = model.fit(L)
            sel_rows.append({"lag": L, "AIC": float(rL.aic), "BIC": float(rL.bic),
                             "HQIC": float(rL.hqic), "FPE": float(rL.fpe)})
        except Exception:
            pass
    sel_df = pd.DataFrame(sel_rows)
    ic = spec.get("ic", "bic").upper()
    if len(sel_df) > 0 and ic in sel_df.columns:
        best_lag = int(sel_df.loc[sel_df[ic].idxmin(), "lag"])
    else:
        best_lag = 1
    res = model.fit(best_lag)
    k = res.k_ar

    coef_df = var_coef_table(res, target, config)
    diag = residual_only_diagnostics(np.asarray(res.resid[target]))

    fitted = res.fittedvalues[target]
    idx = fitted.index
    actual = endog[target].reindex(idx)
    tr3 = (_rmse(actual, fitted), _mae(actual, fitted), _mape(actual, fitted))
    sst = float(np.sum((actual - actual.mean()) ** 2))
    r2 = 1 - float(np.sum((actual - fitted) ** 2)) / sst if sst > 0 else np.nan
    # AIC/BIC: hedef denklem artiklarindan, regresyon modelleriyle ayni olcekte
    resid_t = (actual - fitted).dropna().values
    n_t, k_t = len(resid_t), int(res.params[target].shape[0])
    aic, bic = _gaussian_ic(float(np.sum(resid_t ** 2)), n_t, k_t)
    llf, nobs = float(res.llf), int(res.nobs)

    te3 = (np.nan, np.nan, np.nan)
    test_size = config.get("test_size", 0) or 0
    if test_size > 0 and len(endog) > test_size + k + 2:
        try:
            train = endog.iloc[:-test_size]
            rt = VAR(train).fit(k)
            fc = rt.forecast(train.values[-rt.k_ar:], steps=test_size)
            pred = fc[:, tpos]
            actual_te = endog[target].values[-test_size:]
            te3 = (_rmse(actual_te, pred), _mae(actual_te, pred), _mape(actual_te, pred))
        except Exception:
            pass

    stable = None
    try:
        stable = bool(res.is_stable())
    except Exception:
        pass
    granger = {}
    if spec.get("granger", False):
        for other in vars_:
            if other == target:
                continue
            try:
                gc = res.test_causality(target, [other], kind="f")
                granger[other] = float(gc.pvalue)
            except Exception:
                granger[other] = np.nan

    exp_ok, _, _ = check_expected_signs(coef_df, config.get("expected_signs", {}))
    sig_ratio = significant_coef_ratio(coef_df)
    insig = (1 - sig_ratio) if not np.isnan(sig_ratio) else np.nan
    clean, reasons = calculate_clean_score(diag, config, exp_ok, insig, te3[2], np.nan)
    metrics = _metrics_dict(r2, np.nan, aic, bic, llf, nobs, tr3, te3, np.nan, sig_ratio, exp_ok, clean)
    metrics["var_lag"] = k
    metrics["stable"] = stable

    comments = interpret_diagnostics(diag)
    comments["model"] = f"VAR({k}) degiskenler={vars_}; hedef={target}."
    comments["stability"] = ("Sistem stabil (birim kok yok)." if stable else
                             ("Sistem stabil degil (dikkat)." if stable is not None else "Stabilite hesaplanamadi."))
    if granger:
        gtxt = ", ".join([f"{o}: p={p:.3f}" for o, p in granger.items()])
        comments["granger"] = f"Granger nedensellik ({target} uzerine) -> {gtxt}"
    comments["forecast_note"] = "VAR forecast'lari icseldir; forecast senaryolari uygulanmaz."
    comments["clean_reasons"] = "; ".join(reasons) if reasons else "temiz"

    meta_var = {"kind": "var", "res": res, "vars": vars_, "target_pos": tpos,
                "k_ar": k, "target_base": target}
    result = _make_result("VAR", "VAR", res, coef_df, diag, pd.DataFrame(), metrics, comments, meta_var)
    result["_meta"]["select_order"] = sel_df
    return result


# ---------------- VECM ----------------
def vecm_coef_table(res, vars_, target, config):
    exp = config.get("expected_signs", {})
    tpos = vars_.index(target)
    rows = []
    alpha = np.atleast_2d(res.alpha)
    try:
        alpha_p = np.atleast_2d(res.pvalues_alpha)
    except Exception:
        alpha_p = None
    rank = alpha.shape[1]
    for r in range(rank):
        co = float(alpha[tpos, r])
        p = float(alpha_p[tpos, r]) if alpha_p is not None else np.nan
        rows.append({"variable": f"alpha_ec{r + 1}", "coef": co, "std_error": np.nan, "t_stat": np.nan,
                     "p_value": p, "conf_low": np.nan, "conf_high": np.nan,
                     "significance_comment": interpret_pvalue(p)})
    beta = np.atleast_2d(res.beta)
    for r in range(beta.shape[1]):
        for vi, v in enumerate(vars_):
            if vi >= beta.shape[0]:
                break
            rows.append({"variable": f"beta_ec{r + 1}_{v}", "coef": float(beta[vi, r]),
                         "std_error": np.nan, "t_stat": np.nan, "p_value": np.nan,
                         "conf_low": np.nan, "conf_high": np.nan,
                         "significance_comment": "esbutunlesme (uzun donem) katsayisi"})
    df = pd.DataFrame(rows)
    df["expected_sign"] = [expected_sign_for(v, exp) for v in df["variable"]]
    df["sign_ok"] = [None] * len(df)
    return df


def fit_vecm(prepared_df, config, spec):
    if not HAVE.get("var"):
        raise RuntimeError("statsmodels VECM bileseni yok.")
    meta = prepared_df.attrs["meta"]
    vars_ = spec.get("vars", [])
    if len(vars_) < 2:
        raise ValueError("VECM icin en az 2 degisken gerekir.")
    endog = prepared_df[vars_].dropna()
    target = meta["target_base"] if meta["target_base"] in vars_ else vars_[0]
    tpos = vars_.index(target)
    k_ar_diff = int(spec.get("k_ar_diff", 1))
    rank = int(spec.get("coint_rank", 1))
    det = spec.get("deterministic", "ci")

    johansen_txt = "Johansen testi hesaplanamadi."
    try:
        joh = coint_johansen(endog.values, 0, k_ar_diff)
        n_ce = int(np.sum(joh.lr1 > joh.cvt[:, 1]))  # %5 kritik değere göre
        johansen_txt = (f"Johansen (trace, %5): tahmini esbutunlesme sayisi = {n_ce}. "
                        f"CONFIG coint_rank={rank}.")
    except Exception:
        pass

    res = VECM(endog, k_ar_diff=k_ar_diff, coint_rank=rank, deterministic=det).fit()
    coef_df = vecm_coef_table(res, vars_, target, config)

    resid = np.asarray(res.resid)
    diag = residual_only_diagnostics(resid[:, tpos])
    n_res = resid.shape[0]
    actual_tail = endog[target].values[-n_res:]
    fitted_tail = actual_tail - resid[:, tpos]
    tr3 = (_rmse(actual_tail, fitted_tail), _mae(actual_tail, fitted_tail), _mape(actual_tail, fitted_tail))
    sst = float(np.sum((actual_tail - actual_tail.mean()) ** 2))
    r2 = 1 - float(np.sum(resid[:, tpos] ** 2)) / sst if sst > 0 else np.nan
    llf = float(getattr(res, "llf", np.nan))
    # AIC/BIC: hedef denklem artiklarindan, diger modellerle ayni olcekte
    k_t = len(vars_) * k_ar_diff + rank + 1
    aic_v, bic_v = _gaussian_ic(float(np.sum(resid[:, tpos] ** 2)), n_res, k_t)

    te3 = (np.nan, np.nan, np.nan)
    test_size = config.get("test_size", 0) or 0
    if test_size > 0 and len(endog) > test_size + k_ar_diff + 3:
        try:
            train = endog.iloc[:-test_size]
            rt = VECM(train, k_ar_diff=k_ar_diff, coint_rank=rank, deterministic=det).fit()
            fc = rt.predict(steps=test_size)
            pred = np.asarray(fc)[:, tpos]
            actual_te = endog[target].values[-test_size:]
            te3 = (_rmse(actual_te, pred), _mae(actual_te, pred), _mape(actual_te, pred))
        except Exception:
            pass

    alpha_t = float(np.atleast_2d(res.alpha)[tpos, 0])
    if alpha_t < 0:
        alpha_comment = f"Hedef denklemin alpha (uyarlama) katsayisi {alpha_t:.3f} < 0: uzun donem dengeye geri donus var."
    else:
        alpha_comment = f"Hedef denklemin alpha katsayisi {alpha_t:.3f} >= 0: uyarlama mekanizmasi zayif/ters olabilir."

    exp_ok, _, _ = check_expected_signs(coef_df, config.get("expected_signs", {}))
    clean, reasons = calculate_clean_score(diag, config, exp_ok, np.nan, te3[2], np.nan)
    metrics = _metrics_dict(r2, np.nan, aic_v, bic_v, llf, n_res, tr3, te3, np.nan, np.nan, exp_ok, clean)
    metrics["coint_rank"] = rank
    metrics["alpha_target"] = alpha_t

    comments = interpret_diagnostics(diag)
    comments["model"] = f"VECM(k_ar_diff={k_ar_diff}, rank={rank}, det='{det}') degiskenler={vars_}."
    comments["johansen"] = johansen_txt
    comments["alpha"] = alpha_comment
    comments["forecast_note"] = "VECM forecast'lari icseldir; forecast senaryolari uygulanmaz."
    comments["clean_reasons"] = "; ".join(reasons) if reasons else "temiz"

    meta_vecm = {"kind": "vecm", "res": res, "vars": vars_, "target_pos": tpos, "target_base": target}
    return _make_result("VECM", "VECM", res, coef_df, diag, pd.DataFrame(), metrics, comments, meta_vecm)


# ---------------- Ridge / Lasso / ElasticNet ----------------
def fit_sklearn_family(model_name, prepared_df, config, spec):
    mode = {"RIDGE": "sklearn:ridge", "LASSO": "sklearn:lasso", "ELASTICNET": "sklearn:en"}[model_name]
    res = _fit_regression(prepared_df, config, spec, model_name, mode)
    pipe = res["fit_object"]
    a = getattr(pipe, "_best_alpha", None)
    l1 = getattr(pipe, "_best_l1", None)
    res["comments"]["cv"] = f"CV ile secilen alpha={a}" + (f", l1_ratio={l1}" if l1 else "")
    res["metrics"]["best_alpha"] = a
    if l1:
        res["metrics"]["best_l1"] = l1
    if model_name == "LASSO":
        cf = res["coefficients"]
        zeros = cf[(cf["variable"] != "const") & (cf["coef"].abs() < 1e-10)]["variable"].tolist()
        res["comments"]["lasso_zero"] = ("Sifira dusen katsayilar: " + ", ".join(zeros)) if zeros else "Sifira dusen katsayi yok."
    return res


# ---------------- Dispatcher ----------------
def run_model(model_name, prepared_df, config):
    spec = config["model_specs"].get(model_name, {})
    mt = model_name
    if model_name in ("OLS", "ADL", "DISTRIBUTED_LAG"):
        return safe_fit_model(lambda: fit_linreg_family(model_name, prepared_df, config, spec), model_name, mt)
    if model_name == "ARDL":
        return safe_fit_model(lambda: fit_ardl(prepared_df, config, spec), model_name, mt)
    if model_name == "ECM":
        return safe_fit_model(lambda: fit_ecm(prepared_df, config, spec), model_name, mt)
    if model_name == "VAR":
        return safe_fit_model(lambda: fit_var(prepared_df, config, spec), model_name, mt)
    if model_name == "VECM":
        return safe_fit_model(lambda: fit_vecm(prepared_df, config, spec), model_name, mt)
    if model_name == "SARIMAX":
        return safe_fit_model(lambda: fit_sarimax(prepared_df, config, spec), model_name, mt)
    if model_name in ("RIDGE", "LASSO", "ELASTICNET"):
        return safe_fit_model(lambda: fit_sklearn_family(model_name, prepared_df, config, spec), model_name, mt)
    return make_failed_result(model_name, mt, "Bilinmeyen model turu.")

## 6. Forecast modülü

Forecast, `CONFIG["forecast_scenarios"]` içindeki aktif senaryoya göre çalışır.
Desteklenen senaryo tipleri: `constant`, `growth`, `path`, `linear`, `shock`.
Bağımsız değişken için senaryo tanımlanmamışsa son gözlem sabit tutulur (uyarı
verilir). Lagli modeller özyinelemeli (recursive) tahmin eder; log/diff/logdiff
modellerinde sonuç otomatik olarak seviyeye geri çevrilir.

In [ ]:
def get_active_scenario(config):
    return config["forecast_scenarios"].get(config.get("active_scenario", "baseline"), {})


def build_future_index(prepared_df, config):
    freq = prepared_df.attrs["meta"]["freq"]
    last = prepared_df.index[-1]
    fs, fe = config.get("forecast_start"), config.get("forecast_end")
    start = pd.date_range(last, periods=2, freq=freq)[1] if fs is None else pd.Timestamp(fs)
    if fe is None:
        h = int(config.get("forecast_horizon", 12))
        return pd.date_range(start, periods=h, freq=freq)
    return pd.date_range(start, pd.Timestamp(fe), freq=freq)


def scenario_level_series(base, scen, future_index, last_level, warns):
    spec = scen.get(base)
    n = len(future_index)
    if spec is None:
        warns.append(f"'{base}' icin senaryo tanimlanmadi; son gozlem ({last_level:.4g}) sabit tutuldu.")
        return pd.Series([last_level] * n, index=future_index)
    typ = spec.get("type")
    if typ == "constant":
        return pd.Series([spec["value"]] * n, index=future_index)
    if typ == "growth":
        r = spec.get("monthly_rate", spec.get("rate", 0.0))
        return pd.Series([last_level * ((1 + r) ** (k + 1)) for k in range(n)], index=future_index)
    if typ == "linear":
        return pd.Series(np.linspace(spec["start"], spec["end"], n), index=future_index)
    if typ == "path":
        s = pd.Series({pd.Timestamp(k): float(v) for k, v in spec.get("values", {}).items()})
        s = s.reindex(future_index.union(s.index)).interpolate().reindex(future_index).ffill().bfill()
        if s.isna().all():
            s = pd.Series([last_level] * n, index=future_index)
        return s
    if typ == "shock":
        sd = pd.Timestamp(spec.get("shock_date"))
        g, sh = spec.get("after_shock_growth", 0.0), spec.get("shock_size", 0.0)
        vals, lvl, done = [], last_level, False
        for d in future_index:
            lvl = lvl * (1 + g)
            if (not done) and d >= sd:
                lvl = lvl * (1 + sh); done = True
            vals.append(lvl)
        return pd.Series(vals, index=future_index)
    warns.append(f"'{base}' icin bilinmeyen senaryo tipi '{typ}'; son gozlem sabit tutuldu.")
    return pd.Series([last_level] * n, index=future_index)


def build_forecast_inputs(prepared_df, config, future_index):
    """Gelecek dönem için seviye ve transform edilmiş bağımsız değişken serileri."""
    meta = prepared_df.attrs["meta"]
    scen = get_active_scenario(config)
    warns = []
    combined = prepared_df.index.append(future_index)

    bases = set(config.get("exog_transforms", {}).keys()) | set(scen.keys())
    for _n, sp in config["model_specs"].items():
        bases |= set(sp.get("exog", []))
        bases |= set(sp.get("level_vars", []))
        bases |= set(sp.get("diff_vars", []))

    level_full, trans_full = {}, {}
    for b in bases:
        if b not in prepared_df.columns:
            continue
        last_level = float(prepared_df[b].dropna().iloc[-1])
        fut = scenario_level_series(b, scen, future_index, last_level, warns)
        full_level = pd.concat([prepared_df[b], fut])
        full_level = full_level[~full_level.index.duplicated(keep="last")].reindex(combined)
        level_full[b] = full_level.values.astype(float)
        tcol = meta["base_to_modeled"].get(b, b)
        reg_tr = meta["registry"].get(tcol, {}).get("transform", "level")
        try:
            trans_full[tcol] = apply_transform(full_level, reg_tr).values.astype(float)
        except Exception:
            trans_full[tcol] = full_level.values.astype(float)
    return combined, level_full, trans_full, warns


def run_forecasts(results, prepared_df, config):
    meta = prepared_df.attrs["meta"]
    tr, tb, tm = meta["target_transform"], meta["target_base"], meta["target_modeled"]
    future_index = build_future_index(prepared_df, config)
    combined, level_full, trans_full, warns = build_forecast_inputs(prepared_df, config, future_index)
    n_hist, N = len(prepared_df), len(combined)
    out = {"future_index": future_index, "scenario_warnings": warns, "forecasts": {}, "errors": {}}

    for res in results:
        if res["status"] != "success":
            continue
        m = res.get("_meta", {})
        kind = m.get("kind")
        name = res["model_name"]
        try:
            if kind == "linreg":
                mod_init = np.full(N, np.nan); lvl_init = np.full(N, np.nan)
                mod_init[:n_hist] = prepared_df[tm].values
                lvl_init[:n_hist] = prepared_df[tb].values
                exog_series = {t["tcol"]: trans_full.get(t["tcol"]) for t in m["terms"] if not t["is_y"]}
                meta_lr = {"terms": m["terms"], "target_transform": tr, "predict_scalar": m["predict_scalar"]}
                _, lvl = recursive_linreg_forecast(meta_lr, exog_series, mod_init, lvl_init, n_hist)
                series = pd.Series(lvl[n_hist:], index=future_index)
            elif kind == "ecm":
                x_level = {v: level_full.get(v) for v in m["level_vars"]}
                y_init = np.full(N, np.nan); y_init[:n_hist] = prepared_df[tb].values
                yhat = ecm_forecast(m, x_level, y_init, n_hist)
                series = pd.Series(yhat[n_hist:], index=future_index)
            elif kind == "sarimax":
                res_sx = m["res"]; h = len(future_index)
                if m["exog_cols"]:
                    fx = np.column_stack([trans_full.get(c)[n_hist:] for c in m["exog_cols"]])
                    fc = res_sx.get_forecast(steps=h, exog=fx).predicted_mean
                else:
                    fc = res_sx.get_forecast(steps=h).predicted_mean
                last_level = float(prepared_df[tb].dropna().iloc[-1])
                series = pd.Series(reconstruct_levels(np.asarray(fc), last_level, tr), index=future_index)
            elif kind == "var":
                res_v, k, vars_, tpos = m["res"], m["k_ar"], m["vars"], m["target_pos"]
                endog = prepared_df[vars_].dropna()
                fc = res_v.forecast(endog.values[-k:], steps=len(future_index))
                series = pd.Series(fc[:, tpos], index=future_index)
            elif kind == "vecm":
                res_v, tpos = m["res"], m["target_pos"]
                fc = np.asarray(res_v.predict(steps=len(future_index)))
                series = pd.Series(fc[:, tpos], index=future_index)
            else:
                continue
            res["forecast"] = series.to_frame("forecast")
            out["forecasts"][name] = series
        except Exception as e:
            out["errors"][name] = f"{type(e).__name__}: {e}"
    return out


def attach_selected_forecast(forecast_results, best_model):
    """En iyi model için 'selected_model_forecast' serisini ekler."""
    if best_model and best_model in forecast_results["forecasts"]:
        forecast_results["selected_model"] = best_model
        forecast_results["selected_model_forecast"] = forecast_results["forecasts"][best_model]
    else:
        forecast_results["selected_model"] = best_model
        forecast_results["selected_model_forecast"] = None
    return forecast_results

## 7. Model karşılaştırma ve en iyi model seçimi

Tüm modeller tek bir karşılaştırma tablosunda toplanır. Seçim kriteri
`CONFIG["selection_metric"]` ile belirlenir (bic/aic → en düşük; rmse/mae/mape
→ en düşük test hatası; adjusted_r2/clean_score → en yüksek). Seçilen model tanı
testlerinde sorunluysa rapor bunu açıkça belirtir.

In [ ]:
_SELECTION_MAP = {"bic": "bic", "aic": "aic", "rmse": "rmse_test", "mae": "mae_test",
                  "mape": "mape_test", "adjusted_r2": "adj_r2", "clean_score": "clean_score"}
_HIGHER_BETTER = {"adjusted_r2", "clean_score", "r2"}


def _selection_value(metrics, metric):
    return metrics.get(_SELECTION_MAP.get(metric, "bic"), np.nan)


def compare_models(results, config):
    metric = config.get("selection_metric", "bic")
    dep = transformed_name(config["target_col"], config.get("target_transform", "level"))
    rows = []
    for res in results:
        mt = res.get("metrics", {})
        diag = res.get("diagnostics", {})
        spec = config["model_specs"].get(res["model_name"], {})
        exogv = spec.get("exog") or spec.get("vars") or spec.get("level_vars") or []
        rows.append({
            "model_name": res["model_name"], "model_type": res["model_type"], "status": res["status"],
            "nobs": mt.get("nobs", np.nan), "dependent_variable": dep,
            "exog_variables": ", ".join(exogv) if exogv else "",
            "r2": mt.get("r2", np.nan), "adj_r2": mt.get("adj_r2", np.nan),
            "aic": mt.get("aic", np.nan), "bic": mt.get("bic", np.nan),
            "rmse_train": mt.get("rmse_train", np.nan), "mae_train": mt.get("mae_train", np.nan),
            "mape_train": mt.get("mape_train", np.nan), "rmse_test": mt.get("rmse_test", np.nan),
            "mae_test": mt.get("mae_test", np.nan), "mape_test": mt.get("mape_test", np.nan),
            "bg_pvalue": diag.get("bg_pvalue", np.nan), "ljungbox_pvalue": diag.get("ljungbox_pvalue", np.nan),
            "bp_pvalue": diag.get("bp_pvalue", np.nan), "white_pvalue": diag.get("white_pvalue", np.nan),
            "jb_pvalue": diag.get("jb_pvalue", np.nan), "reset_pvalue": diag.get("reset_pvalue", np.nan),
            "durbin_watson": diag.get("durbin_watson", np.nan), "max_vif": mt.get("max_vif", np.nan),
            "significant_coef_ratio": mt.get("significant_coef_ratio", np.nan),
            "expected_signs_ok": mt.get("expected_signs_ok", None),
            "clean_score": mt.get("clean_score", np.nan),
            "selection_metric_value": _selection_value(mt, metric),
            "selected_best_model": False,
        })
    return pd.DataFrame(rows)


def select_best_model(comparison_df, results, config):
    metric = config.get("selection_metric", "bic")
    ok = comparison_df[comparison_df["status"] == "success"].copy()
    if len(ok) == 0:
        return None
    valcol = "selection_metric_value"
    sub = ok.dropna(subset=[valcol])
    if len(sub) == 0:  # metrik hesaplanamadıysa clean_score'a düş
        sub = ok.dropna(subset=["clean_score"])
        valcol, metric = "clean_score", "clean_score"
        if len(sub) == 0:
            best = ok.iloc[0]["model_name"]
            comparison_df["selected_best_model"] = comparison_df["model_name"] == best
            return best
    if metric in _HIGHER_BETTER:
        best = sub.loc[sub[valcol].idxmax(), "model_name"]
    else:
        best = sub.loc[sub[valcol].idxmin(), "model_name"]
    comparison_df["selected_best_model"] = comparison_df["model_name"] == best
    return best

## 8. Otomatik Türkçe yorum (özet rapor)

Kaç model çalıştı, en iyi model hangisi ve neden seçildi, testler temiz mi, en
büyük sorun ne, forecast ne söylüyor ve model kullanılabilir mi — hepsi kısa ve
net Türkçe cümlelerle özetlenir.

In [ ]:
def _fmt(v, nd=4):
    try:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "NA"
        return f"{v:.{nd}g}"
    except Exception:
        return str(v)


def generate_text_report(results, comparison_df, best_model, forecast_results, config):
    n_ok = sum(1 for r in results if r["status"] == "success")
    n_fail = sum(1 for r in results if r["status"] == "failed")
    metric = config.get("selection_metric", "bic")
    L = []
    L.append(f"Toplam {len(results)} model denendi: {n_ok} basarili, {n_fail} hatali calisti.")
    if n_fail:
        failed = [r["model_name"] for r in results if r["status"] == "failed"]
        L.append(f"Hata veren modeller: {', '.join(failed)} (ayrinti icin 10_ERRORS_WARNINGS).")

    if best_model is None:
        L.append("Hicbir model basariyla secilemedi; CONFIG ve veri kontrol edilmeli.")
        return "\n".join(L)

    best_res = next(r for r in results if r["model_name"] == best_model)
    brow = comparison_df[comparison_df["model_name"] == best_model].iloc[0]
    diag = best_res["diagnostics"]

    L.append(f"Secim kriteri '{metric}' oldugu icin en iyi model olarak "
             f"{best_model} secildi (kriter degeri={_fmt(brow['selection_metric_value'])}, "
             f"clean_score={_fmt(brow['clean_score'], 3)}/100).")

    def sent(key, ok_txt, bad_txt):
        v = diag.get(key)
        if v is None or np.isnan(v):
            return None
        return f"{ok_txt} (p={v:.3f})" if v >= 0.05 else f"{bad_txt} (p={v:.3f})"

    for key, ok_t, bad_t in [
        ("bg_pvalue", "Breusch-Godfrey: otokorelasyon bulgusu yok",
         "Breusch-Godfrey: otokorelasyon riski var"),
        ("bp_pvalue", "Breusch-Pagan: sabit varyans reddedilemiyor",
         "Breusch-Pagan: degisen varyans riski var, robust standart hatalar tercih edilmeli"),
        ("jb_pvalue", "Jarque-Bera: artiklar normal dagiliyor",
         "Jarque-Bera: artiklar normal dagilmiyor"),
        ("reset_pvalue", "RESET: spesifikasyon uygun",
         "RESET: model spesifikasyonu sorunlu olabilir"),
    ]:
        s = sent(key, ok_t, bad_t)
        if s:
            L.append(s)

    mv = brow.get("max_vif")
    if mv is not None and not (isinstance(mv, float) and np.isnan(mv)):
        L.append(interpret_vif(mv, config["clean_score"]["vif_threshold"]).capitalize() + ".")

    problems = []
    for key, lab in [("bg_pvalue", "otokorelasyon"), ("bp_pvalue", "degisen varyans (BP)"),
                     ("white_pvalue", "degisen varyans (White)"), ("reset_pvalue", "spesifikasyon")]:
        v = diag.get(key)
        if v is not None and not np.isnan(v) and v < 0.05:
            problems.append(lab)
    if mv is not None and not (isinstance(mv, float) and np.isnan(mv)) and mv > config["clean_score"]["vif_threshold"]:
        problems.append("coklu dogrusal baglanti")
    L.append("En dikkat cekici sorun(lar): " + (", ".join(problems) if problems else "belirgin sorun yok") + ".")

    fser = forecast_results.get("forecasts", {}).get(best_model)
    if fser is not None and len(fser) > 0:
        first, last = float(fser.iloc[0]), float(fser.iloc[-1])
        direction = "kademeli yukselis" if last > first * 1.001 else (
            "kademeli dusus" if last < first * 0.999 else "yatay seyir")
        L.append(f"Forecast tarafinda {best_model}, hedef degiskenin {fser.index[0].date()} - "
                 f"{fser.index[-1].date()} araliginda {first:.4g} -> {last:.4g} ({direction}) "
                 f"seyrettigini gosteriyor.")

    cs = brow.get("clean_score")
    if (cs is not None and not (isinstance(cs, float) and np.isnan(cs)) and cs >= 70) and not problems:
        L.append("Sonuc: Model forecast icin makul gorunuyor; yine de senaryo varsayimlari gozden gecirilmeli.")
    else:
        L.append("Sonuc: Model kullanilabilir ancak tani sorunlari nedeniyle forecast oncesi revizyon "
                 "(gecikme yapisi ve/veya robust standart hatalar) onerilir.")
    return "\n".join(L)

## 9. Excel çıktısı

Tüm sonuçlar tek bir Excel dosyasına (11 sayfa) yazılır. Sayfalar: `00_README`,
`01_RAW_DATA`, `02_TRANSFORMED_DATA`, `03_MODEL_COMPARISON`,
`04_BEST_MODEL_SUMMARY`, `05_COEFFICIENTS_ALL`, `06_DIAGNOSTICS_ALL`,
`07_VIF_ALL`, `08_FORECASTS`, `09_SCENARIOS`, `10_ERRORS_WARNINGS`.
Biçim: kalın başlıklar, freeze panes, autofilter, otomatik kolon genişliği,
p-value kolonları 4 ondalık, büyük sayılar binlik ayraçlı, tarih formatı düzgün.

In [ ]:
def _write_sheet(writer, df, sheet, pval_cols=None, date_cols=None, max_width=48):
    pval_cols = pval_cols or []
    date_cols = date_cols or []
    df = df.copy()
    df.to_excel(writer, sheet_name=sheet, index=False, startrow=0)
    wb, ws = writer.book, writer.sheets[sheet]
    header_fmt = wb.add_format({"bold": True, "bg_color": "#1F4E78", "font_color": "white",
                                "border": 1, "align": "center", "valign": "vcenter"})
    num_fmt = wb.add_format({"num_format": "#,##0.00"})
    int_fmt = wb.add_format({"num_format": "#,##0"})
    p_fmt = wb.add_format({"num_format": "0.0000"})
    nrows, ncols = df.shape
    for c, col in enumerate(df.columns):
        ws.write(0, c, str(col), header_fmt)
        series = df[col]
        try:
            maxlen = int(series.astype(str).map(len).max())
        except Exception:
            maxlen = 10
        width = min(max_width, max(len(str(col)), maxlen) + 2)
        if col in date_cols:
            ws.set_column(c, c, max(12, width))
        elif col in pval_cols:
            ws.set_column(c, c, width, p_fmt)
        elif pd.api.types.is_integer_dtype(series):
            ws.set_column(c, c, width, int_fmt)
        elif pd.api.types.is_float_dtype(series):
            ws.set_column(c, c, width, num_fmt)
        else:
            ws.set_column(c, c, width)
    if ncols > 0:
        ws.autofilter(0, 0, max(nrows, 1), ncols - 1)
    ws.freeze_panes(1, 0)


def write_readme(writer, prepared_df, config, best_model, report):
    wb = writer.book
    ws = wb.add_worksheet("00_README")
    writer.sheets["00_README"] = ws
    title_fmt = wb.add_format({"bold": True, "font_size": 16, "font_color": "#1F4E78"})
    key_fmt = wb.add_format({"bold": True, "bg_color": "#DDEBF7", "border": 1, "valign": "top"})
    val_fmt = wb.add_format({"border": 1, "valign": "top", "text_wrap": True})
    ws.set_column(0, 0, 26)
    ws.set_column(1, 1, 95)
    ws.write(0, 0, "Ekonometrik Modelleme Ciktilari", title_fmt)
    rows = [
        ("Notebook", "Konfigurasyon odakli, yeniden kullanilabilir ekonometrik modelleme"),
        ("Girdi dosyasi", str(config.get("input_file"))),
        ("Tarih araligi", f"{prepared_df.index.min().date()} - {prepared_df.index.max().date()}"),
        ("Frekans", str(config.get("frequency"))),
        ("Hedef degisken", str(config.get("target_col"))),
        ("Hedef transform", str(config.get("target_transform"))),
        ("Calistirilan modeller", ", ".join(config.get("models_to_run", []))),
        ("Secim kriteri", str(config.get("selection_metric"))),
        ("En iyi model", str(best_model)),
        ("Aktif senaryo", str(config.get("active_scenario"))),
        ("Test boyutu", str(config.get("test_size"))),
        ("Uretim zamani", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")),
    ]
    r = 2
    for k, v in rows:
        ws.write(r, 0, k, key_fmt)
        ws.write(r, 1, v, val_fmt)
        r += 1
    r += 1
    ws.write(r, 0, "Genel Yorum", key_fmt)
    nlines = max(6, report.count("\n") + 1)
    ws.merge_range(r, 1, r + nlines, 1, report, val_fmt)
    ws.freeze_panes(2, 0)


def write_best_summary(writer, results, comparison_df, best_model, config):
    sheet = "04_BEST_MODEL_SUMMARY"
    wb = writer.book
    if best_model is None:
        pd.DataFrame({"Bilgi": ["En iyi model secilemedi."]}).to_excel(writer, sheet_name=sheet, index=False)
        return
    best_res = next(r for r in results if r["model_name"] == best_model)
    brow = comparison_df[comparison_df["model_name"] == best_model].iloc[0]
    stats = brow.to_frame("deger")
    stats.index.name = "metrik"
    stats = stats.reset_index()
    coef = best_res["coefficients"].copy()
    diag = best_res["diagnostics"]
    diag_df = pd.DataFrame([{"test": k, "deger": diag.get(k)} for k in diag])
    comments = best_res["comments"]
    com_df = pd.DataFrame({"konu": list(comments.keys()),
                           "yorum": [str(v) for v in comments.values()]})
    title_fmt = wb.add_format({"bold": True, "font_size": 13, "font_color": "#1F4E78"})
    hdr_fmt = wb.add_format({"bold": True, "bg_color": "#1F4E78", "font_color": "white", "border": 1})

    def block(title, df, startrow):
        df.to_excel(writer, sheet_name=sheet, index=False, startrow=startrow + 1)
        ws = writer.sheets[sheet]
        ws.write(startrow, 0, title, title_fmt)
        for c, col in enumerate(df.columns):
            ws.write(startrow + 1, c, str(col), hdr_fmt)
        return startrow + 1 + len(df) + 3

    r = block(f"EN IYI MODEL: {best_model} - Ana Istatistikler", stats, 0)
    r = block("Katsayi Tablosu", coef, r)
    r = block("Tani Testleri", diag_df, r)
    r = block("Turkce Yorum", com_df, r)
    ws = writer.sheets[sheet]
    ws.set_column(0, 0, 30)
    ws.set_column(1, 3, 22)
    ws.set_column(4, 10, 16)
    ws.freeze_panes(1, 0)


def build_all_coefficients(results):
    cols = ["model_name", "variable", "coef", "std_error", "t_stat", "p_value",
            "conf_low", "conf_high", "significance_comment", "expected_sign", "sign_ok"]
    frames = []
    for res in results:
        c = res.get("coefficients")
        if res["status"] != "success" or c is None or len(c) == 0:
            continue
        c = c.copy()
        c.insert(0, "model_name", res["model_name"])
        for col in cols:
            if col not in c.columns:
                c[col] = np.nan
        frames.append(c[cols])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=cols)


def build_all_diagnostics(results):
    keys = ["bg_pvalue", "ljungbox_pvalue", "bp_pvalue", "white_pvalue",
            "jb_pvalue", "reset_pvalue", "durbin_watson"]
    rows = []
    for res in results:
        d = res.get("diagnostics", {}) or {}
        dc = interpret_diagnostics(d) if d else {}
        row = {"model_name": res["model_name"], "status": res["status"]}
        row.update({k: d.get(k, np.nan) for k in keys})
        row.update({"bg_yorum": dc.get("bg", ""), "bp_yorum": dc.get("bp", ""),
                    "jb_yorum": dc.get("jb", ""), "reset_yorum": dc.get("reset", ""),
                    "dw_yorum": dc.get("dw", "")})
        rows.append(row)
    return pd.DataFrame(rows)


def build_all_vif(results):
    frames = []
    for res in results:
        v = res.get("vif")
        if v is not None and len(v) > 0:
            v = v.copy()
            v.insert(0, "model_name", res["model_name"])
            frames.append(v)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["model_name", "variable", "VIF"])


def build_forecast_table(forecast_results, best_model):
    fi = forecast_results.get("future_index")
    if fi is None or len(fi) == 0:
        return pd.DataFrame(columns=["date"])
    df = pd.DataFrame({"date": pd.to_datetime(fi)})
    for name, ser in forecast_results.get("forecasts", {}).items():
        df[name] = pd.Series(ser).reindex(fi).values
    if best_model and best_model in forecast_results.get("forecasts", {}):
        df[f"SELECTED_{best_model}"] = pd.Series(forecast_results["forecasts"][best_model]).reindex(fi).values
    return df


def build_scenarios_table(config):
    rows = []
    for scen_name, vars_ in config.get("forecast_scenarios", {}).items():
        for var, sp in vars_.items():
            params = {k: v for k, v in sp.items() if k != "type"}
            rows.append({"scenario": scen_name, "active": scen_name == config.get("active_scenario"),
                         "variable": var, "type": sp.get("type"), "parameters": str(params)})
    if not rows:
        rows.append({"scenario": "-", "active": False, "variable": "-", "type": "-", "parameters": "-"})
    return pd.DataFrame(rows)


def build_errors_table(results, prepared_df, forecast_results):
    rows = []
    for n in STARTUP_NOTES:
        rows.append({"type": "startup", "source": "paketler", "message": n})
    meta = prepared_df.attrs.get("meta", {})
    for w in meta.get("warnings", []):
        rows.append({"type": "veri", "source": "prepare_data", "message": w})
    mr = meta.get("missing_report")
    if mr is not None:
        for col, cnt in mr.items():
            if cnt and cnt > 0:
                rows.append({"type": "eksik_deger", "source": str(col), "message": f"{int(cnt)} eksik deger dolduruldu/islenrdi"})
    for w in forecast_results.get("scenario_warnings", []):
        rows.append({"type": "forecast", "source": "senaryo", "message": w})
    for name, err in forecast_results.get("errors", {}).items():
        rows.append({"type": "forecast_hata", "source": name, "message": err})
    for res in results:
        if res["status"] == "failed":
            rows.append({"type": "model_hata", "source": res["model_name"], "message": res.get("error", "")})
    if not rows:
        rows.append({"type": "info", "source": "-", "message": "Uyari veya hata yok."})
    return pd.DataFrame(rows)


def export_to_excel(raw_df, prepared_df, results, comparison_df, best_model,
                    forecast_results, report, config, rolling_results=None):
    path = config.get("output_excel", "model_outputs.xlsx")
    dcol = config.get("date_col", "date")
    with pd.ExcelWriter(path, engine="xlsxwriter", datetime_format="yyyy-mm-dd",
                        date_format="yyyy-mm-dd") as writer:
        write_readme(writer, prepared_df, config, best_model, report)

        raw_out = raw_df.copy()
        _write_sheet(writer, raw_out, "01_RAW_DATA",
                     date_cols=[dcol] if dcol in raw_out.columns else [])

        tdf = prepared_df.reset_index()
        tdf = tdf.rename(columns={tdf.columns[0]: dcol})
        _write_sheet(writer, tdf, "02_TRANSFORMED_DATA", date_cols=[dcol])

        _write_sheet(writer, comparison_df, "03_MODEL_COMPARISON",
                     pval_cols=[c for c in comparison_df.columns if "pvalue" in c])

        write_best_summary(writer, results, comparison_df, best_model, config)

        _write_sheet(writer, build_all_coefficients(results), "05_COEFFICIENTS_ALL",
                     pval_cols=["p_value"])
        _write_sheet(writer, build_all_diagnostics(results), "06_DIAGNOSTICS_ALL",
                     pval_cols=["bg_pvalue", "ljungbox_pvalue", "bp_pvalue", "white_pvalue",
                                "jb_pvalue", "reset_pvalue"])
        _write_sheet(writer, build_all_vif(results), "07_VIF_ALL")
        _write_sheet(writer, build_forecast_table(forecast_results, best_model),
                     "08_FORECASTS", date_cols=["date"])
        _write_sheet(writer, build_scenarios_table(config), "09_SCENARIOS")
        _write_sheet(writer, build_errors_table(results, prepared_df, forecast_results),
                     "10_ERRORS_WARNINGS")

        if rolling_results is not None and len(rolling_results.get("tidy", [])) > 0:
            _write_sheet(writer, rolling_results["tidy"], "11_ROLLING_OLS",
                         pval_cols=["p_value"], date_cols=["date"])
    return path

## 9.1 Kayan pencere (rolling) OLS — zamanla değişen katsayılar

`CONFIG["rolling"]` ile yönetilen **opsiyonel** analiz. Seçilen regresyon
spesifikasyonu (varsayılan `OLS`) sabit genişlikte bir pencere ile kaydırılarak
her dönem yeniden tahmin edilir; katsayıların ve standart hataların **zaman
içindeki seyri** çıkarılır. Ana model seçim/forecast akışını **etkilemez**;
`enabled=False` iken hiç çalışmaz. Çıktı: `11_ROLLING_OLS` Excel sayfası ve
±2 standart hata bantlı katsayı grafiği.

In [ ]:
def run_rolling_ols(prepared_df, config):
    """CONFIG['rolling'] ayarına göre kayan pencere OLS katsayılarını üretir."""
    rc = config.get("rolling", {})
    if not rc.get("enabled", False):
        return None
    if not HAVE.get("rolling", False):
        return {"error": "statsmodels RollingOLS bulunamadi; kayan pencere atlandi.",
                "tidy": pd.DataFrame()}
    model_name = rc.get("model", "OLS")
    spec = config["model_specs"].get(model_name, {})
    window = int(rc.get("window", 36))
    min_nobs = int(rc.get("min_nobs", window))
    step = max(1, int(rc.get("step", 1)))
    try:
        bm = build_regression_matrix(prepared_df, spec, config)
        y, X = align_xy(bm["y"], bm["X"])
        if bm["add_constant"]:
            X = sm.add_constant(X, has_constant="add")
        if X.shape[1] == 0:
            return {"error": "Kayan pencere icin regresor bulunamadi.", "tidy": pd.DataFrame()}
        if len(y) < window:
            return {"error": f"Gozlem sayisi ({len(y)}) pencere boyundan ({window}) kucuk.",
                    "tidy": pd.DataFrame()}
        rres = RollingOLS(y, X, window=window, min_nobs=min_nobs, expanding=False).fit()
        # statsmodels sürümüne göre params ndarray dönebilir -> DataFrame'e sar
        cols, idx = list(X.columns), y.index
        params = pd.DataFrame(np.asarray(rres.params), index=idx, columns=cols)
        bse = pd.DataFrame(np.asarray(rres.bse), index=idx, columns=cols)
        tvals = pd.DataFrame(np.asarray(rres.tvalues), index=idx, columns=cols)
        pvals = pd.DataFrame(np.asarray(rres.pvalues), index=idx, columns=cols)
        recs = []
        for dt in params.index[::step]:
            for col in params.columns:
                c = params.loc[dt, col]
                if pd.isna(c):
                    continue
                p = float(pvals.loc[dt, col])
                recs.append({"date": dt, "variable": str(col), "coef": float(c),
                             "std_error": float(bse.loc[dt, col]),
                             "t_stat": float(tvals.loc[dt, col]), "p_value": p,
                             "significance_comment": interpret_pvalue(p)})
        return {"model": model_name, "window": window, "min_nobs": min_nobs, "step": step,
                "params": params, "bse": bse, "tidy": pd.DataFrame(recs), "error": None}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}", "tidy": pd.DataFrame()}


def plot_rolling_coefficients(rolling_results, config):
    """Kayan pencere katsayılarını ±2 standart hata bandıyla çizer/kaydeder."""
    if not rolling_results or rolling_results.get("params") is None:
        return None
    params, bse = rolling_results["params"], rolling_results["bse"]
    outdir = Path(config.get("plot_dir", "plots"))
    outdir.mkdir(parents=True, exist_ok=True)
    cols = [c for c in params.columns if c != "const"] or list(params.columns)
    n = len(cols)
    fig, axes = plt.subplots(n, 1, figsize=(10, 2.6 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        p = params[col].dropna()
        se = bse[col].reindex(p.index)
        ax.plot(p.index, p.values, color="#1F4E78")
        ax.fill_between(p.index, p.values - 2 * se.values, p.values + 2 * se.values,
                        color="#1F4E78", alpha=0.15)
        ax.axhline(0, color="red", lw=0.8, ls="--")
        ax.set_title(f"{col} - kayan pencere OLS (W={rolling_results['window']}) +/-2 s.h.")
        ax.grid(alpha=0.3)
    fig.tight_layout()
    path = outdir / "07_rolling_ols_coefficients.png"
    fig.savefig(path, dpi=110, bbox_inches="tight")
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)
    return str(path)

## 10. Grafikler

Hedef değişken zaman serisi, en iyi modelin gerçek vs fitted / residual /
residual histogram grafikleri, actual vs forecast ve model karşılaştırma bar
grafiği. Grafikler `CONFIG["plot_dir"]` altına PNG olarak da kaydedilir.

In [ ]:
def make_plots(results, prepared_df, comparison_df, best_model, forecast_results, config):
    outdir = Path(config.get("plot_dir", "plots"))
    outdir.mkdir(parents=True, exist_ok=True)
    meta = prepared_df.attrs["meta"]
    tb = meta["target_base"]
    paths = []

    def _save(fig, name):
        p = outdir / name
        fig.savefig(p, dpi=110, bbox_inches="tight")
        paths.append(str(p))
        try:
            plt.show()
        except Exception:
            pass
        plt.close(fig)

    # 1) Hedef değişken zaman serisi
    try:
        fig, ax = plt.subplots(figsize=(10, 4))
        prepared_df[tb].plot(ax=ax, color="#1F4E78")
        ax.set_title(f"Hedef degisken zaman serisi: {tb}")
        ax.grid(alpha=0.3)
        _save(fig, "01_target_series.png")
    except Exception as e:
        print("Grafik 1 atlandi:", e)

    best_res = next((r for r in results if r["model_name"] == best_model), None)

    # 2) Gerçek vs fitted + 3) residual + 4) histogram (en iyi model, modellenmiş uzay)
    fitted = None
    if best_res is not None:
        try:
            fitted = pd.Series(best_res["fit_object"].fittedvalues)
        except Exception:
            fitted = None
    if fitted is not None and len(fitted) > 0:
        try:
            tm = meta["target_modeled"]
            actual = prepared_df[tm].reindex(fitted.index)
            fig, ax = plt.subplots(figsize=(10, 4))
            actual.plot(ax=ax, label="gercek", color="#1F4E78")
            fitted.plot(ax=ax, label="fitted", color="#E1701A", alpha=0.8)
            ax.set_title(f"Gercek vs Fitted (modellenmis): {best_model}")
            ax.legend(); ax.grid(alpha=0.3)
            _save(fig, "02_actual_vs_fitted.png")

            resid = (actual - fitted).dropna()
            fig, ax = plt.subplots(figsize=(10, 3.2))
            ax.plot(resid.index, resid.values, color="#555")
            ax.axhline(0, color="red", lw=1)
            ax.set_title(f"Residual plot: {best_model}")
            ax.grid(alpha=0.3)
            _save(fig, "03_residuals.png")

            fig, ax = plt.subplots(figsize=(6, 4))
            ax.hist(resid.values, bins=20, color="#1F4E78", alpha=0.85)
            ax.set_title(f"Residual histogram: {best_model}")
            _save(fig, "04_residual_hist.png")
        except Exception as e:
            print("Grafik 2-4 atlandi:", e)

    # 5) Actual vs forecast (seviye)
    fser = forecast_results.get("forecasts", {}).get(best_model)
    if fser is not None and len(fser) > 0:
        try:
            hist_tail = prepared_df[tb].iloc[-min(36, len(prepared_df)):]
            fig, ax = plt.subplots(figsize=(10, 4))
            hist_tail.plot(ax=ax, label="gercek", color="#1F4E78")
            fser.plot(ax=ax, label=f"forecast ({best_model})", color="#C00000", marker="o", ms=3)
            ax.axvline(hist_tail.index[-1], color="gray", ls="--", alpha=0.6)
            ax.set_title("Gercek vs Forecast (secilen model)")
            ax.legend(); ax.grid(alpha=0.3)
            _save(fig, "05_actual_vs_forecast.png")
        except Exception as e:
            print("Grafik 5 atlandi:", e)

    # 6) Model karşılaştırma bar (clean_score)
    try:
        ok = comparison_df[comparison_df["status"] == "success"].dropna(subset=["clean_score"])
        if len(ok) > 0:
            fig, ax = plt.subplots(figsize=(10, 4))
            colors = ["#C00000" if m == best_model else "#1F4E78" for m in ok["model_name"]]
            ax.bar(ok["model_name"], ok["clean_score"], color=colors)
            ax.set_title("Model karsilastirma: clean_score (kirmizi = secilen)")
            ax.set_ylabel("clean_score")
            plt.xticks(rotation=45, ha="right")
            ax.grid(alpha=0.3, axis="y")
            _save(fig, "06_model_comparison.png")
    except Exception as e:
        print("Grafik 6 atlandi:", e)

    return paths

## 11. Çalıştırma akışı

Aşağıdaki hücre tüm süreci **spesifikasyondaki akışla birebir** çalıştırır:
veri okuma → hazırlık → modeller → karşılaştırma → en iyi model → forecast →
rapor → grafikler → Excel. Hata veren model tüm süreci durdurmaz; hatası
`10_ERRORS_WARNINGS` sayfasına yazılır. Notebook baştan sona tek seferde çalışır.

In [ ]:
print("=" * 70)
print("1) Veri okuma ve hazirlama")
raw_df = load_data(CONFIG)
prepared_df = prepare_data(raw_df, CONFIG)
for w in prepared_df.attrs["meta"].get("warnings", []):
    print("   UYARI:", w)

print("\n2) Modeller calistiriliyor")
results = []
for model_name in CONFIG["models_to_run"]:
    result = run_model(model_name, prepared_df, CONFIG)
    flag = "OK " if result["status"] == "success" else "HATA"
    extra = "" if result["status"] == "success" else f" | {result.get('error')}"
    print(f"   [{flag}] {model_name}{extra}")
    results.append(result)

print("\n3) Karsilastirma ve en iyi model secimi")
comparison_df = compare_models(results, CONFIG)
best_model = select_best_model(comparison_df, results, CONFIG)
print(f"   Secilen en iyi model: {best_model}")

print("\n4) Forecast uretimi")
forecast_results = run_forecasts(results, prepared_df, CONFIG)
forecast_results = attach_selected_forecast(forecast_results, best_model)
for w in forecast_results.get("scenario_warnings", []):
    print("   UYARI:", w)

print("\n5) Otomatik Turkce yorum")
report = generate_text_report(results, comparison_df, best_model, forecast_results, CONFIG)
print("-" * 70)
print(report)
print("-" * 70)

if CONFIG.get("make_plots", True):
    print("\n6) Grafikler")
    try:
        make_plots(results, prepared_df, comparison_df, best_model, forecast_results, CONFIG)
    except Exception as e:
        print("   Grafik uretiminde hata:", e)

print("\n6.1) Kayan pencere (rolling) OLS")
rolling_results = run_rolling_ols(prepared_df, CONFIG)
if rolling_results is None:
    print("   Kapali (CONFIG['rolling']['enabled']=False).")
elif rolling_results.get("error"):
    print("   Not:", rolling_results["error"])
else:
    print(f"   {rolling_results['model']} icin W={rolling_results['window']} kayan pencere "
          f"katsayilari hesaplandi ({rolling_results['tidy']['date'].nunique()} pencere).")
    if CONFIG.get("make_plots", True):
        try:
            plot_rolling_coefficients(rolling_results, CONFIG)
        except Exception as e:
            print("   Rolling grafik hatasi:", e)

print("\n7) Excel ciktisi")
output_path = export_to_excel(raw_df=raw_df, prepared_df=prepared_df, results=results,
                              comparison_df=comparison_df, best_model=best_model,
                              forecast_results=forecast_results, report=report, config=CONFIG,
                              rolling_results=rolling_results)
print(f"   Excel yazildi: {output_path}")
print("=" * 70)

## 12. Sonuç tablosunu incele

Karşılaştırma tablosunu ve seçilen modelin forecast'ini aşağıda görebilirsiniz.

In [ ]:
def _show(df):
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string())


print("Model karsilastirma tablosu:")
_show(comparison_df[["model_name", "status", "r2", "adj_r2", "aic", "bic",
                     "rmse_test", "mae_test", "mape_test", "clean_score",
                     "selection_metric_value", "selected_best_model"]])

print(f"\nSecilen model: {best_model}")
if forecast_results.get("selected_model_forecast") is not None:
    print("Secilen modelin forecast'i:")
    _show(forecast_results["selected_model_forecast"].to_frame("forecast"))

# BÖLÜM 2 — Heteroskedastisite Teşhis ve İleri Ekonometrik Analiz

Bu bölüm mevcut ana akışı (veri yükleme, model karşılaştırma, forecast, Excel)
**değiştirmez**; `CONFIG["advanced"]["enabled"]` ile açılan **opsiyonel** bir
katmandır ve ayrı bir Excel dosyasına (`econometric_diagnosis_output.xlsx`) yazar.

**Temel amaç:** Heteroskedastisite tespit edildiğinde yalnızca HC3 uygulamak
yerine *sorunun kaynağını teşhis etmek* (otokorelasyon mu, ARCH etkisi mi,
spesifikasyon/kırılma mı), uygun alternatifleri (HAC, dönüşüm, yapısal kırılma
kuklaları, WLS, FGLS, ARCH/GARCH) sınamak ve karşılaştırmalı raporlamaktır.

> **Kritik nokta:** HC3 yalnızca standart hataları ve istatistiksel çıkarımı
> heteroskedastisiteye karşı dayanıklı yapar; **model artıklarının varyans
> yapısını değiştirmez**. Bu yüzden HC3 sonrası BP/White testinin hâlâ anlamlı
> çıkması "düzeltme başarısız" demek değildir.

## 2.1 İleri analiz importları ve yardımcılar

ADF/KPSS, ARCH-LM, Goldfeld–Quandt, CUSUM, etkili gözlem (OLSInfluence), ARDL
bounds testi (UECM) ve `arch` (GARCH) bileşenleri. Eksik olanlar sessizce
devre dışı kalır; süreç durmaz.

In [ ]:
from statsmodels.tsa.stattools import kpss as _kpss
from statsmodels.stats.diagnostic import (
    het_arch, het_goldfeldquandt, breaks_cusumolsresid, recursive_olsresiduals,
)
from statsmodels.stats.outliers_influence import OLSInfluence

HAVE_ADV = {}
try:
    from statsmodels.tsa.ardl import ARDL as _ARDL, UECM as _UECM
    HAVE_ADV["uecm"] = True
except Exception:
    HAVE_ADV["uecm"] = False
try:
    from arch import arch_model as _arch_model
    from arch.univariate import ARX
    HAVE_ADV["arch"] = True
except Exception:
    HAVE_ADV["arch"] = False

if not HAVE_ADV.get("arch"):
    STARTUP_NOTES.append("Opsiyonel paket 'arch' yok; ARCH/GARCH varyans modelleri atlanacak "
                         "(ana akis ve diger ileri analizler calisir).")


def _adv_alpha(config):
    return float(config.get("advanced", {}).get("significance_level", 0.05))


def _adv_independents(config, prepared_df):
    """İleri analizde kullanılacak bağımsız değişkenler (orijinal/base isimler)."""
    adv = config.get("advanced", {})
    inds = list(adv.get("independent_variables") or [])
    if not inds:
        base = adv.get("base_model", "OLS")
        inds = list(config.get("model_specs", {}).get(base, {}).get("exog", []))
    # yalnızca veride bulunanları tut
    return [v for v in inds if v in prepared_df.columns]


def hac_maxlags(n):
    """Newey-West için otomatik maksimum gecikme: floor(4*(n/100)^(2/9))."""
    return int(max(1, np.floor(4.0 * (max(n, 1) / 100.0) ** (2.0 / 9.0))))


def cov_fit_kwargs(cov_type, nobs, config):
    """statsmodels .fit() için kovaryans argümanları (HC3/HAC/nonrobust)."""
    if cov_type in (None, "nonrobust"):
        return {}
    if cov_type == "HAC":
        return {"cov_type": "HAC", "cov_kwds": {"maxlags": hac_maxlags(nobs)}}
    return {"cov_type": cov_type}

## 2.2 Durağanlık: ADF + KPSS ve I(0)/I(1)/I(2) yorumu

Her değişken için hem düzeyde hem birinci farkta ADF ve KPSS çalıştırılır; iki
test birlikte yorumlanarak bütünleşme derecesi sınıflanır. **ARDL'ye I(2)
değişken girmemelidir**; I(2) şüphesi varsa ileri ARDL adımı uyarı verir.

In [ ]:
def run_adf(series):
    """ADF testi (H0: birim kök var / durağan değil)."""
    s = pd.Series(series).dropna()
    try:
        r = adfuller(s, autolag="AIC")
        return {"stat": float(r[0]), "pvalue": float(r[1]), "lags": int(r[2]), "nobs": int(r[3])}
    except Exception as e:
        return {"stat": np.nan, "pvalue": np.nan, "lags": np.nan, "error": str(e)}


def run_kpss(series):
    """KPSS testi (H0: seri durağan)."""
    s = pd.Series(series).dropna()
    try:
        r = _kpss(s, regression="c", nlags="auto")
        return {"stat": float(r[0]), "pvalue": float(r[1]), "lags": int(r[2])}
    except Exception as e:
        return {"stat": np.nan, "pvalue": np.nan, "lags": np.nan, "error": str(e)}


def classify_integration(adf_l, kpss_l, adf_d, kpss_d, alpha):
    """ADF+KPSS sonuçlarından bütünleşme derecesini sınıflar."""
    def sd(adf_p, kpss_p):
        adf_stat = (adf_p is not None) and (not np.isnan(adf_p)) and (adf_p < alpha)   # durağan
        kpss_stat = (kpss_p is not None) and (not np.isnan(kpss_p)) and (kpss_p >= alpha)  # durağan
        return adf_stat, kpss_stat
    adf_s_l, kpss_s_l = sd(adf_l, kpss_l)
    adf_s_d, kpss_s_d = sd(adf_d, kpss_d)
    if adf_s_l and kpss_s_l:
        return "I(0)"
    if adf_s_d and kpss_s_d:
        return "muhtemelen I(1)"
    if (not adf_s_d) and (not kpss_s_d):
        return "I(2) olma ihtimali"
    if adf_s_l != kpss_s_l:
        return "testler celiskili (duzeyde)"
    return "testler celiskili"


def stationarity_table(prepared_df, config):
    """Bağımlı + bağımsız değişkenler için ADF/KPSS (düzey+fark) tablosu.

    Dönüş: (DataFrame, i2_variables listesi).
    """
    meta = prepared_df.attrs["meta"]
    alpha = _adv_alpha(config)
    vars_ = [meta["target_base"]] + _adv_independents(config, prepared_df)
    rows, i2_vars = [], []
    for v in dict.fromkeys(vars_):  # sırayı koru, tekrarı at
        s = prepared_df[v].astype(float)
        adf_l, kpss_l = run_adf(s), run_kpss(s)
        adf_d, kpss_d = run_adf(s.diff()), run_kpss(s.diff())
        order = classify_integration(adf_l["pvalue"], kpss_l["pvalue"],
                                     adf_d["pvalue"], kpss_d["pvalue"], alpha)
        if order == "I(2) olma ihtimali":
            i2_vars.append(v)
        rows.append({
            "degisken": v,
            "ADF_duzey_stat": adf_l["stat"], "ADF_duzey_p": adf_l["pvalue"],
            "KPSS_duzey_stat": kpss_l["stat"], "KPSS_duzey_p": kpss_l["pvalue"],
            "ADF_fark_stat": adf_d["stat"], "ADF_fark_p": adf_d["pvalue"],
            "KPSS_fark_stat": kpss_d["stat"], "KPSS_fark_p": kpss_d["pvalue"],
            "butunlesme": order,
        })
    return pd.DataFrame(rows), i2_vars

## 2.3 İleri heteroskedastisite / ARCH testleri ve karar mekanizması

BP, White, Goldfeld–Quandt ve ARCH-LM birlikte değerlendirilir. Karar
mekanizması (Durum 1–4) sorunun kaynağını ayırır ve uygun stratejiyi önerir.

In [ ]:
def arch_lm_test(resid, nlags=None):
    """ARCH-LM testi (H0: koşullu değişen varyans / ARCH etkisi yok)."""
    r = np.asarray(pd.Series(resid).dropna())
    n = len(r)
    if nlags is None:
        nlags = int(min(12, max(1, n // 5)))
    try:
        stat, p, f, fp = het_arch(r, nlags=nlags)
        return {"stat": float(stat), "pvalue": float(p), "lags": nlags,
                "fstat": float(f), "fpvalue": float(fp)}
    except Exception as e:
        return {"stat": np.nan, "pvalue": np.nan, "lags": nlags, "error": str(e)}


def goldfeld_quandt_test(y, X):
    """Goldfeld–Quandt testi (H0: sabit varyans)."""
    try:
        f, p, _order = het_goldfeldquandt(np.asarray(y, float), np.asarray(X, float))
        return {"stat": float(f), "pvalue": float(p)}
    except Exception as e:
        return {"stat": np.nan, "pvalue": np.nan, "error": str(e)}


def interpret_test_result(pvalue, alpha, reject_msg, fail_msg):
    """İstatistiksel olarak doğru dille test yorumu (kanıt yokluğu vurgusu)."""
    if pvalue is None or (isinstance(pvalue, float) and np.isnan(pvalue)):
        return "test hesaplanamadi"
    return reject_msg if pvalue < alpha else fail_msg


def heteroskedasticity_interpretation(plan, alpha):
    """HC3'ün ne yaptığını/yapmadığını doğru anlatan Türkçe yorum bloğu."""
    L = []
    if plan["arch"]:
        L.append("ARCH-LM anlamli: ortalama denklem artiklarinda kosullu (zamanla degisen) "
                 "varyans/ARCH etkisi bulunduguna dair kanit vardir. Bu durumda dogru yaklasim "
                 "ortalama denklemi koruyup artiklar uzerinde ARCH/GARCH varyans modeli kurmaktir.")
    if plan["autocorr"] and plan["hetero"]:
        L.append("Hem otokorelasyon hem heteroskedastisite birlikte gorulmektedir; standart hatalar "
                 "icin HAC (Newey-West) kovaryansi kullanilmalidir.")
    elif plan["hetero"]:
        L.append("Otokorelasyon yokken heteroskedastisite gorulmektedir; cikarim icin HC3 raporlanabilir, "
                 "ancak sorunun kaynagi da arastirilmalidir (donusum, yapisal kirilma, WLS/FGLS).")
    elif plan["autocorr"]:
        L.append("Heteroskedastisite gozlenmezken otokorelasyon vardir; HAC (Newey-West) standart "
                 "hatalari uygundur.")
    else:
        L.append("Otokorelasyon ve heteroskedastisiteye dair belirgin kanit yoktur.")
    L.append("HC3 artiklarin varyans yapisini DEGISTIRMEZ; yalnizca standart hatalari "
             "heteroskedastisiteye karsi dayanikli hale getirir. Bu nedenle HC3 uygulandiktan sonra "
             "BP/White testinin anlamli kalmasi duzeltmenin basarisiz oldugu anlamina gelmez.")
    L.append("Heteroskedastisitenin FIILEN giderilmesi icin model donusumu (log/fark), varyans "
             "modellemesi (WLS/FGLS/GARCH) veya yapisal kirilma duzeltmesi gerekir.")
    return " ".join(L)


def heteroskedasticity_decision(diag, arch_res, config):
    """Durum 1–4 karar mekanizması: sorunun kaynağını teşhis eder, strateji önerir.

    diag: full_diagnostics çıktısı (bg/bp/white p-değerleri).
    arch_res: arch_lm_test çıktısı.
    """
    alpha = _adv_alpha(config)

    def sig(v):
        return (v is not None) and (not np.isnan(v)) and (v < alpha)

    autocorr = sig(diag.get("bg_pvalue"))
    hetero = sig(diag.get("bp_pvalue")) or sig(diag.get("white_pvalue"))
    arch = sig(arch_res.get("pvalue"))

    plan = {"autocorr": autocorr, "hetero": hetero, "arch": arch}
    if arch:
        plan.update(case="Durum 3 - ARCH etkisi",
                    primary_cov="HC3", actions=["residual_garch", "HC3"])
    elif autocorr and hetero:
        plan.update(case="Durum 2 - Otokorelasyon + Heteroskedastisite",
                    primary_cov="HAC", actions=["HAC"])
    elif hetero:
        plan.update(case="Durum 1/4 - Heteroskedastisite (otokorelasyon yok)",
                    primary_cov="HC3",
                    actions=["HC3", "log_y", "log_log", "difference", "break_dummies", "wls", "fgls"])
    elif autocorr:
        plan.update(case="Otokorelasyon (heteroskedastisite yok)",
                    primary_cov="HAC", actions=["HAC"])
    else:
        plan.update(case="Belirgin sorun yok",
                    primary_cov=config["advanced"].get("robust_covariance", "nonrobust"), actions=[])
    plan["interpretation"] = heteroskedasticity_interpretation(plan, alpha)
    return plan

## 2.4 Model kurulumu, dönüşüm varyantları ve remediation motoru

İleri analiz seviye değişkenler üzerinde çalışır (heteroskedastisite giderme
dönüşümleri — log/fark — seviye ilişkisine göre tanımlıdır). ARDL lag yapısı
AIC/BIC ile seçilir; varyantlar (Level / LogY / LogLog / Difference /
BreakAdjusted) ayrı modeller olarak saklanır. WLS, FGLS ve artık-tabanlı
ARCH/GARCH yalnızca teşhis gerektirdiğinde çalışır.

In [ ]:
def build_variant_design(prepared_df, config, y_transform="level", x_log=False,
                         lags_y=0, lags_x=0, dummies=None):
    """Seviye değişkenlerden esnek ARDL/OLS tasarım matrisi kurar.

    y_transform: level/log/diff/logdiff. x_log: bağımsızlara log uygula.
    lags_y: bağımlı değişken gecikme sayısı. lags_x: her bağımsız için maksimum lag.
    dummies: eklenecek kukla DataFrame (opsiyonel). Dönüş: (y, X, y_etiket).
    """
    meta = prepared_df.attrs["meta"]
    tb = meta["target_base"]
    inds = _adv_independents(config, prepared_df)
    yb = prepared_df[tb].astype(float)

    if y_transform == "level":
        y, ytag = yb.copy(), "Level"
    elif y_transform == "log":
        if (yb <= 0).any():
            raise ValueError("log(y) icin bagimli degiskende sifir/negatif deger var.")
        y, ytag = np.log(yb), "LogY"
    elif y_transform == "diff":
        y, ytag = yb.diff(), "Difference"
    elif y_transform == "logdiff":
        if (yb <= 0).any():
            raise ValueError("logdiff(y) icin bagimli degiskende sifir/negatif deger var.")
        y, ytag = np.log(yb).diff(), "LogDiff"
    else:
        raise ValueError(f"Bilinmeyen y_transform: {y_transform}")

    parts = []
    for v in inds:
        s = prepared_df[v].astype(float)
        if x_log:
            if (s <= 0).any():
                raise ValueError(f"log({v}) icin sifir/negatif deger var.")
            s, nm = np.log(s), f"ln_{v}"
        else:
            nm = v
        parts.append(s.rename(nm))
        for L in range(1, int(lags_x) + 1):
            parts.append(s.shift(L).rename(f"L{L}_{nm}"))
    for L in range(1, int(lags_y) + 1):
        parts.append(y.shift(L).rename(f"L{L}_dep"))

    # Not: prepared_df kaynaklı Series'ler aynı .attrs['meta']'yı taşıdığından
    # pd.concat pandas 3.x'te attrs karşılaştırmasında hata verir; bu yüzden
    # tasarım matrisi doğrudan kolon atamasıyla (attrs taşımadan) kurulur.
    X = pd.DataFrame(index=prepared_df.index)
    for part in parts:
        X[part.name] = np.asarray(part, float)
    if dummies is not None and len(dummies.columns) > 0:
        dre = dummies.reindex(prepared_df.index)
        for c in dre.columns:
            X[c] = np.asarray(dre[c], float)
    X["__y__"] = np.asarray(y, float)
    data = X.dropna()
    return data["__y__"], data.drop(columns="__y__"), ytag


def _adv_metrics(res, y):
    """Modellenmiş uzayda R2/AIC/BIC/RMSE/MAE/MAPE."""
    fitted = res.fittedvalues
    rmse = _rmse(y.values, fitted.values)
    mae = _mae(y.values, fitted.values)
    mape = _mape(y.values, fitted.values)
    return {"r2": float(res.rsquared), "adj_r2": float(res.rsquared_adj),
            "aic": float(res.aic), "bic": float(res.bic), "nobs": int(res.nobs),
            "rmse": rmse, "mae": mae, "mape": mape}


def fit_ols_record(name, y, X, config, cov_type="nonrobust", y_transform="Level",
                   p_order=0, q_order=0, kind="OLS"):
    """Tek bir (W)OLS modelini standart 'ileri model kaydı' olarak fit eder."""
    exp_signs = config.get("expected_signs", {})
    Xc = sm.add_constant(X, has_constant="add")
    res = sm.OLS(y, Xc).fit(**cov_fit_kwargs(cov_type, len(y), config))
    diag = full_diagnostics(res, config)
    arch_res = arch_lm_test(res.resid)
    diag["arch_pvalue"] = arch_res.get("pvalue", np.nan)
    diag["arch_lags"] = arch_res.get("lags", np.nan)
    gq = goldfeld_quandt_test(y, Xc)
    diag["gq_pvalue"] = gq.get("pvalue", np.nan)
    vif = compute_vif(X)
    max_vif = float(vif["VIF"].max()) if len(vif) > 0 else np.nan
    coef_df = coef_table_from_sm(res, exp_signs)
    return {
        "name": name, "kind": kind, "y_transform": y_transform, "cov_type": cov_type,
        "p_order": p_order, "q_order": q_order, "fit": res, "diag": diag, "vif": vif,
        "max_vif": max_vif, "coef_df": coef_df, "metrics": _adv_metrics(res, y),
        "y": y, "X": X, "arch": arch_res,
        "used_hc3": cov_type in ("HC0", "HC1", "HC3"), "used_hac": cov_type == "HAC",
        "used_wls": False, "used_fgls": False, "weights": None, "note": "",
    }


def select_ardl_order_level(prepared_df, config):
    """Seviye değişkenler üzerinde AIC/BIC ile ARDL (p,q) gecikme seçimi."""
    adv = config["advanced"]
    max_p, max_q = int(adv.get("max_lag_y", 6)), int(adv.get("max_lag_x", 6))
    crit = str(adv.get("lag_selection_criterion", "BIC")).upper()
    best = None
    search = []
    for p in range(0, max_p + 1):
        for q in range(0, max_q + 1):
            try:
                y, X, _ = build_variant_design(prepared_df, config, "level", lags_y=p, lags_x=q)
                if len(y) < X.shape[1] + 3:
                    continue
                r = sm.OLS(y, sm.add_constant(X, has_constant="add")).fit()
                ic = float(r.bic) if crit == "BIC" else float(r.aic)
                search.append({"p_lag_y": p, "q_lag_x": q, "AIC": float(r.aic),
                               "BIC": float(r.bic), "nobs": int(r.nobs)})
                if best is None or ic < best[0]:
                    best = (ic, p, q)
            except Exception:
                continue
    if best is None:
        return 1, 1, pd.DataFrame(search)
    return best[1], best[2], pd.DataFrame(search)

## 2.5 WLS ve FGLS

WLS yalnızca ağırlık yapısı makul tahmin edilebiliyorsa; sıfıra bölünme ve
aşırı ağırlığa karşı floor + winsorization uygulanır. FGLS yardımcı varyans
regresyonuyla iteratif çalışır (yakınsama kontrollü).

In [ ]:
def _winsorize_weights(w):
    w = np.asarray(w, float)
    lo, hi = np.nanpercentile(w, [1, 99])
    return np.clip(w, max(lo, 1e-8), max(hi, 1e-8))


def _bp_after(res, Xc, w=None):
    """(Ağırlıklı) artıklar üzerinde BP p-değeri — heteroskedastisite kaldı mı?"""
    try:
        r = res.resid.values if w is None else (np.sqrt(w) * res.resid.values)
        return float(het_breuschpagan(r, np.asarray(Xc))[1])
    except Exception:
        return np.nan


def fit_wls_variants(y, X, config):
    """Birkaç ağırlık şemasıyla WLS dener, BP sonrasına göre en iyisini seçer."""
    Xc = sm.add_constant(X, has_constant="add")
    ols = sm.OLS(y, Xc).fit()
    yhat, resid = ols.fittedvalues.values, ols.resid.values
    cands = {"1/yhat^2": 1.0 / np.maximum(yhat ** 2, 1e-8),
             "1/|yhat|": 1.0 / np.maximum(np.abs(yhat), 1e-6)}
    try:
        aux = sm.OLS(np.log(np.maximum(resid ** 2, 1e-12)), Xc).fit()
        cands["aux_var^-1"] = 1.0 / np.maximum(np.exp(aux.fittedvalues.values), 1e-8)
    except Exception:
        pass
    best = None
    for wname, w in cands.items():
        w = _winsorize_weights(w)
        try:
            res = sm.WLS(y, Xc, weights=w).fit()
            bp = _bp_after(res, Xc, w)
            key = bp if not np.isnan(bp) else -1
            if best is None or key > best[0]:
                best = (key, wname, res, w)
        except Exception:
            continue
    if best is None:
        return None
    return {"bp_after": best[0], "weight_name": best[1], "res": best[2], "weights": best[3]}


def fit_fgls(y, X, config):
    """İteratif FGLS: yardımcı varyans regresyonu -> ağırlık -> tekrar tahmin."""
    Xc = sm.add_constant(X, has_constant="add")
    max_iter = int(config["advanced"].get("fgls_max_iter", 5))
    res = sm.OLS(y, Xc).fit()
    w, prev, iters = None, None, 0
    for iters in range(1, max_iter + 1):
        aux = sm.OLS(np.log(np.maximum(res.resid.values ** 2, 1e-12)), Xc).fit()
        w = _winsorize_weights(1.0 / np.maximum(np.exp(aux.fittedvalues.values), 1e-8))
        res = sm.WLS(y, Xc, weights=w).fit()
        cur = res.params.values
        if prev is not None and np.max(np.abs(cur - prev)) < 1e-6:
            break
        prev = cur
    return {"res": res, "weights": w, "iterations": iters, "bp_after": _bp_after(res, Xc, w)}

## 2.6 Artık-tabanlı ARCH/GARCH (yalnızca ARCH-LM anlamlıysa)

ARDL ortalama denklemi korunur; artıklarda koşullu varyans varsa `arch`
paketiyle ARCH(1)/GARCH(1,1) ve alternatifler AIC/BIC ile karşılaştırılır.

In [ ]:
def fit_residual_garch(resid, config):
    """Artıklar üzerinde ARCH/GARCH spesifikasyonlarını karşılaştırır."""
    if not HAVE_ADV.get("arch"):
        return {"available": False, "note": "arch paketi kurulu degil; GARCH atlandi."}
    s = pd.Series(resid).dropna()
    r = np.asarray(s, float)
    specs = [("ARCH(1)", 1, 0), ("GARCH(1,1)", 1, 1), ("GARCH(1,2)", 1, 2), ("GARCH(2,1)", 2, 1)]
    rows, best = [], None
    for nm, p, q in specs:
        try:
            am = _arch_model(r, mean="Zero", vol="GARCH", p=p, q=q, rescale=True)
            fr = am.fit(disp="off")
            rows.append({"model": nm, "AIC": float(fr.aic), "BIC": float(fr.bic),
                         "loglik": float(fr.loglikelihood)})
            if best is None or fr.bic < best[0]:
                best = (fr.bic, nm, fr)
        except Exception as e:
            rows.append({"model": nm, "AIC": np.nan, "BIC": np.nan, "error": str(e)[:120]})
    out = {"available": True, "table": pd.DataFrame(rows), "best_name": None, "fit": None, "cond_vol": None}
    if best is not None:
        out["best_name"] = best[1]
        out["fit"] = best[2]
        out["cond_vol"] = pd.Series(np.asarray(best[2].conditional_volatility), index=s.index)
    return out

## 2.7 Yapısal kırılma, aykırı/etkili gözlem ve ARDL bounds testi

In [ ]:
def make_break_dummies(index, break_dates, kinds=("level",)):
    """Verilen tarihler için level/pulse/trend kukla değişkenleri üretir."""
    D = pd.DataFrame(index=index)
    t = np.arange(len(index))
    for bd in break_dates:
        bd = pd.Timestamp(bd)
        tag = bd.date()
        if "level" in kinds:
            D[f"D_lvl_{tag}"] = (index >= bd).astype(float)
        if "pulse" in kinds:
            D[f"D_pulse_{tag}"] = (index == bd).astype(float)
        if "trend" in kinds:
            before = int((index < bd).sum())
            D[f"D_trend_{tag}"] = np.where(index >= bd, t - before, 0.0)
    return D


def chow_test(y, X, break_date):
    """Bilinen tarihte Chow yapısal kırılma testi."""
    from scipy.stats import f as _f
    Xc = sm.add_constant(X, has_constant="add")
    idx = y.index
    bd = pd.Timestamp(break_date)
    mask = idx < bd
    k = Xc.shape[1]
    if mask.sum() < k + 2 or (~mask).sum() < k + 2:
        return {"F": np.nan, "pvalue": np.nan, "note": "kirilma etrafinda yetersiz gozlem"}
    rss_p = sm.OLS(y, Xc).fit().ssr
    rss1 = sm.OLS(y[mask], Xc[mask]).fit().ssr
    rss2 = sm.OLS(y[~mask], Xc[~mask]).fit().ssr
    n = len(y)
    denom = (rss1 + rss2) / (n - 2 * k)
    if denom <= 0:
        return {"F": np.nan, "pvalue": np.nan, "note": "hesaplanamadi"}
    F = ((rss_p - (rss1 + rss2)) / k) / denom
    return {"F": float(F), "pvalue": float(_f.sf(F, k, n - 2 * k)), "break_date": str(bd.date())}


def cusum_stability(y, X):
    """OLS artıklarına dayalı CUSUM istikrar testi."""
    Xc = sm.add_constant(X, has_constant="add")
    try:
        res = sm.OLS(y, Xc).fit()
        stat, p, _ = breaks_cusumolsresid(res.resid, ddof=Xc.shape[1])
        return {"stat": float(stat), "pvalue": float(p), "stable": bool(p >= 0.05)}
    except Exception as e:
        return {"stat": np.nan, "pvalue": np.nan, "error": str(e)}


def detect_residual_jumps(y, X, z=3.0):
    """Artıklarda |std artık| > z olan olası kırılma/aykırı tarihleri döndürür."""
    Xc = sm.add_constant(X, has_constant="add")
    res = sm.OLS(y, Xc).fit()
    sr = res.resid / (res.resid.std() or 1.0)
    hits = sr[sr.abs() > z]
    return [pd.Timestamp(d) for d in hits.index]


def influence_analysis(y, X):
    """Aykırı/etkili gözlem analizi (yardımcı OLS + OLSInfluence)."""
    Xc = sm.add_constant(X, has_constant="add")
    res = sm.OLS(y, Xc).fit()
    infl = OLSInfluence(res)
    n, k = len(y), Xc.shape[1]
    # statsmodels sürümüne göre bunlar Series dönebilir -> numpy'a çevir
    std_r = np.asarray(infl.resid_studentized_internal)
    stud_r = np.asarray(infl.resid_studentized_external)
    lev = np.asarray(infl.hat_matrix_diag)
    cooks = np.asarray(infl.cooks_distance[0])
    dfb = np.asarray(infl.dfbetas)
    resid_v = np.asarray(res.resid)
    lev_thr, cook_thr = 2.0 * k / n, 4.0 / n
    rows = []
    for i, dt in enumerate(y.index):
        flag = (abs(std_r[i]) > 3) or (lev[i] > lev_thr) or (cooks[i] > cook_thr)
        rows.append({"tarih": dt, "artik": float(resid_v[i]),
                     "std_resid": float(std_r[i]), "student_resid": float(stud_r[i]),
                     "leverage": float(lev[i]), "cooks_d": float(cooks[i]),
                     "max_abs_dfbeta": float(np.max(np.abs(dfb[i]))), "aykiri_mi": bool(flag)})
    df = pd.DataFrame(rows)
    return df, df[df["aykiri_mi"]].copy()


def ardl_bounds_test(prepared_df, config, p_order, q_order):
    """UECM üzerinden ARDL bounds eşbütünleşme testi (case 3: kısıtsız sabit)."""
    if not HAVE_ADV.get("uecm"):
        return {"available": False, "note": "statsmodels UECM yok."}
    meta = prepared_df.attrs["meta"]
    tb = meta["target_base"]
    inds = _adv_independents(config, prepared_df)
    if not inds:
        return {"available": False, "note": "Bagimsiz degisken yok."}
    df = prepared_df[[tb] + inds].dropna()
    try:
        u = _UECM(df[tb], lags=max(1, int(p_order)), exog=df[inds],
                  order=max(1, int(q_order)), trend="c")
        r = u.fit()
        bt = r.bounds_test(case=3)
        cv = bt.crit_vals
        alpha = _adv_alpha(config)
        pct = 100 * (1 - alpha)
        j = int(np.argmin(np.abs(np.asarray(cv.index, dtype=float) - pct)))
        lower, upper = float(cv.iloc[j]["lower"]), float(cv.iloc[j]["upper"])
        stat = float(bt.stat)
        if stat > upper:
            concl = "kesin esbutunlesme var"
        elif stat < lower:
            concl = "kesin esbutunlesme yok"
        else:
            concl = "sonuc belirsiz"
        return {"available": True, "F": stat, "alpha": alpha, "percentile": float(cv.index[j]),
                "I0_lower": lower, "I1_upper": upper, "conclusion": concl,
                "case": "3 (kisitsiz sabit, trend yok)"}
    except Exception as e:
        return {"available": True, "error": str(e)}

## 2.8 Kısa ve uzun dönem etkiler (delta yöntemi)

ARDL uzun dönem çarpanı: `LR_x = (Σ β_j) / (1 − Σ φ_i)`. Standart hata, seçilen
(HC3/HAC) kovaryans matrisiyle delta yöntemiyle hesaplanır.

In [ ]:
def short_run_effects(record, config):
    """Kısa dönem katsayıları (bağımsız değişkenlerin çağdaş ve gecikmeli terimleri)."""
    coef = record["coef_df"]
    # bağımsızları terim isminden çıkar (const ve L*_dep hariç)
    rows = []
    for _, r in coef.iterrows():
        v = r["variable"]
        if v == "const" or v.endswith("_dep") or v.startswith("D_"):
            continue
        rows.append({"terim": v, "katsayi": r["coef"], "std_hata": r["std_error"],
                     "p_deger": r["p_value"], "yorum": r["significance_comment"]})
    return pd.DataFrame(rows)


def long_run_effects(record, config):
    """ARDL uzun dönem çarpanları + delta yöntemi ile SE/p/CI."""
    from scipy.stats import norm
    res = record["fit"]
    params = res.params
    cov = res.cov_params()
    p_order, q_order = record.get("p_order", 0), record.get("q_order", 0)
    phi_names = [f"L{L}_dep" for L in range(1, p_order + 1) if f"L{L}_dep" in params.index]
    phi_sum = float(sum(params[n] for n in phi_names))
    denom = 1.0 - phi_sum
    # bağımsız değişken kök isimleri (log varyantında ln_ önekli olabilir)
    base_vars = sorted({re.sub(r"^L\d+_", "", v) for v in params.index
                        if v not in ("const",) and not v.endswith("_dep") and not v.startswith("D_")})
    rows = []
    for v in base_vars:
        beta_names = [n for n in params.index if re.sub(r"^L\d+_", "", n) == v]
        if not beta_names or abs(denom) < 1e-9:
            continue
        num = float(sum(params[n] for n in beta_names))
        lr = num / denom
        grad = pd.Series(0.0, index=params.index)
        for n in beta_names:
            grad[n] = 1.0 / denom
        for n in phi_names:
            grad[n] = num / (denom ** 2)
        g = grad.values
        var = float(g @ cov.values @ g)
        se = np.sqrt(var) if var > 0 else np.nan
        if se and se > 0:
            z = lr / se
            pval = float(2 * (1 - norm.cdf(abs(z))))
            lo, hi = lr - 1.96 * se, lr + 1.96 * se
        else:
            z = pval = lo = hi = np.nan
        rows.append({"degisken": v, "uzun_donem_carpan": lr, "std_hata": se, "z": z,
                     "p_deger": pval, "ci_alt": lo, "ci_ust": hi})
    return pd.DataFrame(rows), phi_sum

## 2.9 Orkestrasyon, karşılaştırma, forecast, yorum ve Excel çıktısı

`run_advanced_analysis` tüm ileri analizi tek bir sözlükte toplar; her adım
try/except ile korunur (bir adım hata verse bile diğerleri çalışır ve hata
`Errors_Warnings` sayfasına yazılır). Sonuçlar ayrı bir Excel dosyasına
(20 sayfa) ve ayrı grafiklere yazılır.

In [ ]:
def _adv_data_quality(prepared_df, config):
    """Sayısal olmayan/sonsuz/eksik değer ve log uygunluk kontrolleri."""
    meta = prepared_df.attrs["meta"]
    rows = []
    cols = [meta["target_base"]] + _adv_independents(config, prepared_df)
    for c in dict.fromkeys(cols):
        s = prepared_df[c]
        n_inf = int(np.isinf(pd.to_numeric(s, errors="coerce")).sum())
        n_na = int(s.isna().sum())
        n_nonpos = int((pd.to_numeric(s, errors="coerce") <= 0).sum())
        rows.append({"degisken": c, "gozlem": int(s.notna().sum()), "eksik": n_na,
                     "sonsuz": n_inf, "sifir_veya_negatif": n_nonpos,
                     "log_uygun": "evet" if n_nonpos == 0 else "HAYIR (log/logdiff riskli)"})
    return pd.DataFrame(rows)


def _adv_descriptive(prepared_df, config):
    """Betimsel istatistikler (çarpıklık/basıklık dahil) + korelasyon."""
    cols = list(dict.fromkeys([prepared_df.attrs["meta"]["target_base"]] +
                              _adv_independents(config, prepared_df)))
    sub = prepared_df[cols].apply(pd.to_numeric, errors="coerce")
    desc = pd.DataFrame({
        "degisken": cols,
        "gozlem": [int(sub[c].notna().sum()) for c in cols],
        "ortalama": [float(sub[c].mean()) for c in cols],
        "std_sapma": [float(sub[c].std()) for c in cols],
        "min": [float(sub[c].min()) for c in cols],
        "max": [float(sub[c].max()) for c in cols],
        "carpiklik": [float(sub[c].skew()) for c in cols],
        "basiklik": [float(sub[c].kurtosis()) for c in cols],
    })
    corr = sub.corr().reset_index().rename(columns={"index": "degisken"})
    return desc, corr


def _hetero_flags(record, alpha):
    bp = record["diag"].get("bp_pvalue")
    wh = record["diag"].get("white_pvalue")
    arch = record["diag"].get("arch_pvalue")
    hetero = ((bp is not None and not np.isnan(bp) and bp < alpha) or
              (wh is not None and not np.isnan(wh) and wh < alpha))
    return hetero, bp, wh, arch


def advanced_forecast(prepared_df, primary, config):
    """Primary (Level) ARDL için senaryo bazlı recursive forecast + sabit-varyans aralığı."""
    try:
        meta = prepared_df.attrs["meta"]
        tb = meta["target_base"]
        res = primary["fit"]
        X = primary["X"]
        if any(str(c).startswith(("D_", "ln_")) for c in X.columns):
            return None  # yalnızca sade Level primary için forecast
        terms = []
        for col in X.columns:
            col = str(col)
            m = re.match(r"^L(\d+)_dep$", col)
            if m:
                terms.append({"name": col, "tcol": tb, "lag": int(m.group(1)), "is_y": True})
                continue
            m = re.match(r"^L(\d+)_(.+)$", col)
            if m:
                terms.append({"name": col, "tcol": m.group(2), "lag": int(m.group(1)), "is_y": False})
            else:
                terms.append({"name": col, "tcol": col, "lag": 0, "is_y": False})
        future_index = build_future_index(prepared_df, config)
        combined, level_full, _trans, warns = build_forecast_inputs(prepared_df, config, future_index)
        n_hist, N = len(prepared_df), len(combined)
        mod_init = np.full(N, np.nan); lvl_init = np.full(N, np.nan)
        mod_init[:n_hist] = prepared_df[tb].values
        lvl_init[:n_hist] = prepared_df[tb].values
        exog_series = {t["tcol"]: level_full.get(t["tcol"]) for t in terms if not t["is_y"]}
        if any(v is None for v in exog_series.values()):
            return None
        meta_lr = {"terms": terms, "target_transform": "level",
                   "predict_scalar": make_linreg_predictor(res.params.to_dict(), True)}
        _, lvl = recursive_linreg_forecast(meta_lr, exog_series, mod_init, lvl_init, n_hist)
        point = lvl[n_hist:]
        sigma = float(np.sqrt(res.mse_resid)) if hasattr(res, "mse_resid") else float(res.resid.std())
        df = pd.DataFrame({"date": pd.to_datetime(future_index), "tahmin": point,
                           "alt_ci": point - 1.96 * sigma, "ust_ci": point + 1.96 * sigma})
        return {"df": df, "sigma": sigma, "warns": warns}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}


def run_advanced_analysis(prepared_df, config, forecast_results=None):
    """Tüm ileri (heteroskedastisite teşhis + giderme) analizini çalıştırır."""
    adv_cfg = config.get("advanced", {})
    if not adv_cfg.get("enabled", False):
        return None
    alpha = _adv_alpha(config)
    A = {"errors": [], "warnings": [], "records": [], "config": adv_cfg}

    def guard(label, fn):
        try:
            return fn()
        except Exception as e:
            A["errors"].append({"adim": label, "hata": f"{type(e).__name__}: {e}"})
            return None

    A["data_quality"] = guard("data_quality", lambda: _adv_data_quality(prepared_df, config))
    dd = guard("descriptive", lambda: _adv_descriptive(prepared_df, config))
    A["descriptive"], A["correlation"] = (dd if dd else (pd.DataFrame(), pd.DataFrame()))
    st = guard("stationarity", lambda: stationarity_table(prepared_df, config))
    A["stationarity"], A["i2_vars"] = (st if st else (pd.DataFrame(), []))
    if A["i2_vars"]:
        A["warnings"].append(f"I(2) suphesi olan degisken(ler): {', '.join(A['i2_vars'])}. "
                             f"ARDL'ye I(2) degisken dahil edilmemeli; sonuclar dikkatle yorumlanmali.")

    p, q, lag_df = guard("lag_search", lambda: select_ardl_order_level(prepared_df, config)) or (1, 1, pd.DataFrame())
    A["lag_search"] = lag_df
    A["p_order"], A["q_order"] = p, q

    # --- Primary ARDL (Level) ---
    def _primary():
        y, X, _ = build_variant_design(prepared_df, config, "level", lags_y=p, lags_x=q)
        return fit_ols_record(f"ARDL_Level(p={p},q={q})", y, X, config,
                              cov_type="nonrobust", y_transform="Level",
                              p_order=p, q_order=q, kind="ARDL")
    primary = guard("primary_model", _primary)
    if primary is None:
        A["warnings"].append("Primary ARDL kurulamadi; ileri analiz sinirli calisti.")
        return A
    A["records"].append(primary)
    A["primary"] = primary

    # --- Heteroskedastisite karar mekanizması ---
    decision = guard("hetero_decision",
                     lambda: heteroskedasticity_decision(primary["diag"], primary["arch"], config))
    A["decision"] = decision or {}
    base_hetero, _, _, _ = _hetero_flags(primary, alpha)

    # --- Primary robust yeniden tahmin (karar mekanizmasına göre) ---
    prim_cov = (decision or {}).get("primary_cov", adv_cfg.get("robust_covariance", "HC3"))
    def _robust():
        return fit_ols_record(f"ARDL_Level_{prim_cov}", primary["y"], primary["X"], config,
                              cov_type=prim_cov, y_transform="Level", p_order=p, q_order=q, kind="ARDL")
    A["primary_robust"] = guard("primary_robust", _robust)
    if A["primary_robust"]:
        A["records"].append(A["primary_robust"])

    # --- Dönüşüm varyantları ---
    if adv_cfg.get("make_log_variants", True):
        for label, kw in [("ARDL_LogY", dict(y_transform="log")),
                          ("ARDL_LogLog", dict(y_transform="log", x_log=True)),
                          ("ARDL_Difference", dict(y_transform="diff"))]:
            def _mk(label=label, kw=kw):
                y, X, ytag = build_variant_design(prepared_df, config, lags_y=p, lags_x=q, **kw)
                return fit_ols_record(label, y, X, config, cov_type="nonrobust",
                                      y_transform=ytag, p_order=p, q_order=q, kind="ARDL")
            rec = guard(label, _mk)
            if rec:
                A["records"].append(rec)

    # --- Yapısal kırılma tespiti ve BreakAdjusted model ---
    breaks = list(adv_cfg.get("manual_break_dates", []))
    if adv_cfg.get("test_structural_breaks", True):
        jumps = guard("break_detect",
                      lambda: detect_residual_jumps(primary["y"], primary["X"])) or []
        breaks = list(dict.fromkeys([str(pd.Timestamp(b).date()) for b in breaks] +
                                    [str(b.date()) for b in jumps]))
    A["break_dates"] = breaks
    A["cusum"] = guard("cusum", lambda: cusum_stability(primary["y"], primary["X"])) or {}
    A["chow"] = []
    for b in breaks:
        ch = guard(f"chow_{b}", lambda b=b: chow_test(primary["y"], primary["X"], b))
        if ch:
            ch["break_date"] = b
            A["chow"].append(ch)
    if breaks:
        def _break_model():
            D = make_break_dummies(prepared_df.index, breaks, kinds=("level",))
            y, X, _ = build_variant_design(prepared_df, config, "level", lags_y=p, lags_x=q, dummies=D)
            return fit_ols_record("ARDL_BreakAdjusted", y, X, config, cov_type="nonrobust",
                                  y_transform="Level", p_order=p, q_order=q, kind="ARDL")
        rec = guard("break_model", _break_model)
        if rec:
            A["records"].append(rec)

    # --- WLS / FGLS (heteroskedastisite varsa) ---
    if adv_cfg.get("test_wls", True) and base_hetero:
        w = guard("wls", lambda: fit_wls_variants(primary["y"], primary["X"], config))
        if w:
            rec = fit_ols_record(f"WLS({w['weight_name']})", primary["y"], primary["X"], config,
                                 y_transform="Level", p_order=p, q_order=q, kind="WLS")
            # WLS'i gerçek WLS fit'iyle değiştir
            rec["fit"] = w["res"]; rec["used_wls"] = True; rec["weights"] = w["weights"]
            rec["coef_df"] = coef_table_from_sm(w["res"], config.get("expected_signs", {}))
            rec["metrics"] = _adv_metrics(w["res"], primary["y"])
            rec["diag"]["bp_pvalue_after"] = w["bp_after"]
            rec["note"] = f"Agirlik: {w['weight_name']}; floor+winsorization uygulandi."
            A["records"].append(rec)
    if adv_cfg.get("test_fgls", True) and base_hetero:
        f_ = guard("fgls", lambda: fit_fgls(primary["y"], primary["X"], config))
        if f_:
            rec = fit_ols_record("FGLS", primary["y"], primary["X"], config,
                                 y_transform="Level", p_order=p, q_order=q, kind="FGLS")
            rec["fit"] = f_["res"]; rec["used_fgls"] = True; rec["weights"] = f_["weights"]
            rec["coef_df"] = coef_table_from_sm(f_["res"], config.get("expected_signs", {}))
            rec["metrics"] = _adv_metrics(f_["res"], primary["y"])
            rec["diag"]["bp_pvalue_after"] = f_["bp_after"]
            rec["note"] = f"{f_['iterations']} iterasyon; yardimci varyans regresyonu."
            A["records"].append(rec)

    # --- ARCH/GARCH (yalnızca ARCH-LM anlamlıysa) ---
    if adv_cfg.get("test_arch_garch", True) and primary["arch"].get("pvalue", 1) is not None \
            and not np.isnan(primary["arch"].get("pvalue", np.nan)) \
            and primary["arch"]["pvalue"] < alpha:
        A["garch"] = guard("garch", lambda: fit_residual_garch(primary["fit"].resid, config)) or {}
    else:
        A["garch"] = {"available": HAVE_ADV.get("arch", False),
                      "note": "ARCH-LM anlamli degil; GARCH calistirilmadi." }

    # --- Bounds testi, kısa/uzun dönem, etkili gözlem ---
    A["bounds"] = guard("bounds", lambda: ardl_bounds_test(prepared_df, config, p, q)) or {}
    _sr_model = A["primary_robust"] if A.get("primary_robust") is not None else primary
    sr = guard("short_run", lambda: short_run_effects(_sr_model, config))
    A["short_run"] = sr if sr is not None else pd.DataFrame()
    lr = guard("long_run", lambda: long_run_effects(_sr_model, config))
    A["long_run"], A["phi_sum"] = (lr if lr is not None else (pd.DataFrame(), np.nan))
    infl = guard("influence", lambda: influence_analysis(primary["y"], primary["X"]))
    A["influence_all"], A["influence_flag"] = (infl if infl is not None else (pd.DataFrame(), pd.DataFrame()))

    # --- ECM hata düzeltme (Engle-Granger, seviye bağımsızlar) ---
    def _ecm():
        inds = _adv_independents(config, prepared_df)
        spec = {"level_vars": inds, "diff_vars": inds, "cointegration_method": "engle_granger",
                "lags_y": [1], "lags_x": {}}
        lr_res, sr_res, srX, dyy, u, meta_ecm = _ecm_estimate(prepared_df, spec, config)
        ect = float(sr_res.params.get("ECT_L1", np.nan))
        ect_p = float(sr_res.pvalues.get("ECT_L1", np.nan))
        return {"ect_coef": ect, "ect_pvalue": ect_p,
                "ect_ok": (ect < 0 and ect_p < alpha)}
    A["ecm"] = guard("ecm", _ecm) or {}

    # --- Forecast (primary level) ---
    A["forecast"] = guard("forecast", lambda: advanced_forecast(prepared_df, primary, config)) or {}
    A["main_forecast"] = forecast_results

    # --- Karşılaştırma + yorum ---
    A["comparison"] = build_advanced_comparison(A, config)
    A["interpretation"] = generate_advanced_interpretation(A, config)
    return A

## 2.10 Karşılaştırma skoru, otomatik yorum, grafikler ve Excel export

Model seçimi tek başına AIC/BIC'e değil; işaret doğruluğu, I(2) yokluğu,
otokorelasyon, stabilite, ECM uygunluğu, bounds testi, forecast performansı,
AIC/BIC, heteroskedastisite yönetimi ve VIF önceliklerine dayanır. **Temiz
model skoru** 0–100'dür ve her bileşen ayrı kolonda gösterilir (kara kutu değil).

In [ ]:
def _sig(v, alpha):
    return (v is not None) and (not (isinstance(v, float) and np.isnan(v))) and (v < alpha)


def _kv(d, kcol="alan", vcol="deger"):
    return pd.DataFrame([{kcol: k, vcol: (str(v) if not isinstance(v, (int, float)) else v)}
                         for k, v in (d or {}).items()])


def _record_hetero_reduced(record, base_hetero, alpha):
    if record.get("used_wls") or record.get("used_fgls"):
        after = record["diag"].get("bp_pvalue_after")
        now = _sig(after, alpha)
        return "evet" if (base_hetero and not now) else ("hayir" if now else "evet")
    if record.get("used_hc3") or record.get("used_hac"):
        return "hayir (yalnizca standart hata; varyans degismez)"
    bp = record["diag"].get("bp_pvalue")
    now = _sig(bp, alpha)
    if not base_hetero:
        return "baslangicta hetero yok"
    return "evet" if not now else "hayir"


def build_advanced_comparison(A, config):
    """Tüm ileri modeller için ayrıntılı karşılaştırma + şeffaf temiz-skor bileşenleri."""
    alpha = _adv_alpha(config)
    recs = A["records"]
    i2_ok = len(A.get("i2_vars", [])) == 0
    bounds = A.get("bounds", {})
    bounds_ok = bounds.get("conclusion") == "kesin esbutunlesme var"
    bounds_unc = bounds.get("conclusion") == "sonuc belirsiz"
    stable = A.get("cusum", {}).get("stable")
    ect_ok = A.get("ecm", {}).get("ect_ok")
    base_hetero, _, _, _ = _hetero_flags(A.get("primary", recs[0]), alpha) if recs else (False, 0, 0, 0)

    rmses = [r["metrics"].get("rmse", np.nan) for r in recs]
    bics = [r["metrics"].get("bic", np.nan) for r in recs]
    rmin, rmax = np.nanmin(rmses), np.nanmax(rmses)
    bmin, bmax = np.nanmin(bics), np.nanmax(bics)

    def _lin(x, lo, hi, pts):  # düşük iyi -> yüksek puan
        if np.isnan(x) or hi == lo:
            return pts * 0.5
        return pts * (1 - (x - lo) / (hi - lo))

    rows = []
    for r in recs:
        d, m = r["diag"], r["metrics"]
        exp_ok = None
        if "expected_signs_ok" in r.get("metrics", {}):
            exp_ok = r["metrics"]["expected_signs_ok"]
        # işaret bilgisi coef_df'ten
        signs = [s for s in r["coef_df"].get("sign_ok", []) if s is not None] if len(r["coef_df"]) else []
        exp_ok = (all(signs) if signs else None)
        hetero_now = _sig(d.get("bp_pvalue"), alpha) or _sig(d.get("white_pvalue"), alpha)
        reduced = _record_hetero_reduced(r, base_hetero, alpha)

        s_sign = 10 if exp_ok is True else (5 if exp_ok is None else 0)
        s_i2 = 10 if i2_ok else 0
        s_auto = 15 if not _sig(d.get("bg_pvalue"), alpha) else 0
        s_stable = 10 if stable is True else (5 if stable is None else 0)
        s_ecm = 10 if ect_ok is True else (5 if ect_ok is None else 0)
        s_bounds = 10 if bounds_ok else (5 if bounds_unc else 0)
        s_fore = _lin(m.get("rmse", np.nan), rmin, rmax, 10)
        s_ic = _lin(m.get("bic", np.nan), bmin, bmax, 10)
        handled = (r.get("used_hc3") or r.get("used_hac") or r.get("used_wls") or r.get("used_fgls")
                   or (A.get("garch", {}).get("fit") is not None))
        s_het = 10 if (not hetero_now) else (7 if handled else 0)
        s_vif = 5 if (not np.isnan(m.get("bic", np.nan)) and (np.isnan(r["max_vif"]) or r["max_vif"] < 10)) else 0
        total = round(s_sign + s_i2 + s_auto + s_stable + s_ecm + s_bounds + s_fore + s_ic + s_het + s_vif, 1)

        rows.append({
            "model": r["name"], "y_donusumu": r["y_transform"],
            "lag_yapisi": f"p={r.get('p_order')},q={r.get('q_order')}", "gozlem": m.get("nobs"),
            "AIC": m.get("aic"), "BIC": m.get("bic"), "R2": m.get("r2"), "adj_R2": m.get("adj_r2"),
            "RMSE": m.get("rmse"), "MAE": m.get("mae"), "MAPE": m.get("mape"),
            "BG_p": d.get("bg_pvalue"), "LjungBox_p": d.get("ljungbox_pvalue"),
            "BP_p": d.get("bp_pvalue"), "White_p": d.get("white_pvalue"),
            "ARCH_LM_p": d.get("arch_pvalue"), "RESET_p": d.get("reset_pvalue"),
            "JB_p": d.get("jb_pvalue"), "max_VIF": r["max_vif"],
            "bounds_sonucu": bounds.get("conclusion", "NA"),
            "stabilite": ("stabil" if stable else ("stabil degil" if stable is False else "NA")),
            "kovaryans": r["cov_type"], "HC3_kullanildi": r.get("used_hc3"),
            "HAC_kullanildi": r.get("used_hac"), "WLS_FGLS": (r.get("used_wls") or r.get("used_fgls")),
            "hetero_fiilen_azaldi": reduced,
            "skor_isaret": s_sign, "skor_I2": s_i2, "skor_otokorelasyon": s_auto,
            "skor_stabilite": s_stable, "skor_ecm": s_ecm, "skor_bounds": s_bounds,
            "skor_forecast": round(s_fore, 1), "skor_aicbic": round(s_ic, 1),
            "skor_hetero": s_het, "skor_vif": s_vif, "TEMIZ_SKOR": total,
        })
    df = pd.DataFrame(rows).sort_values("TEMIZ_SKOR", ascending=False).reset_index(drop=True)
    return df

## 2.11 Otomatik Türkçe yorum (istatistiksel olarak doğru dil)

In [ ]:
def generate_advanced_interpretation(A, config):
    """İleri analiz için Türkçe, istatistiksel olarak doğru dilli otomatik yorum."""
    alpha = _adv_alpha(config)
    L = []
    prim = A.get("primary")
    if prim is None:
        return ["Primary model kurulamadigi icin ileri yorum uretilemedi."]

    # Durağanlık
    if A.get("i2_vars"):
        L.append(f"Durağanlık: {', '.join(A['i2_vars'])} degisken(ler)inde I(2) suphesi vardir; "
                 f"ARDL'ye I(2) degisken dahil edilmemeli, sonuclar dikkatle degerlendirilmelidir.")
    else:
        L.append("Durağanlık: Degiskenler ADF/KPSS'e gore I(0)/I(1) aralikta; I(2) belirtisi guclu degildir, "
                 "ARDL yaklasimi uygundur.")
    L.append(f"Gecikme secimi ({config['advanced'].get('lag_selection_criterion','BIC')}): p={A.get('p_order')}, q={A.get('q_order')}.")

    # Heteroskedastisite kararı
    dec = A.get("decision", {})
    d = prim["diag"]
    L.append("Otokorelasyon (Breusch-Godfrey): " + interpret_test_result(
        d.get("bg_pvalue"), alpha,
        f"p={_fmt(d.get('bg_pvalue'))}<{alpha}, otokorelasyon bulunduguna dair kanit vardir.",
        f"p={_fmt(d.get('bg_pvalue'))}>={alpha}, otokorelasyon bulunduguna iliskin yeterli kanit yoktur."))
    L.append("Heteroskedastisite (Breusch-Pagan): " + interpret_test_result(
        d.get("bp_pvalue"), alpha,
        f"p={_fmt(d.get('bp_pvalue'))}<{alpha}, degisen varyans bulunduguna dair kanit vardir.",
        f"p={_fmt(d.get('bp_pvalue'))}>={alpha}, degisen varyansa iliskin yeterli kanit yoktur."))
    L.append("ARCH-LM: " + interpret_test_result(
        d.get("arch_pvalue"), alpha,
        f"p={_fmt(d.get('arch_pvalue'))}<{alpha}, artiklarda kosullu (ARCH) varyans kaniti vardir.",
        f"p={_fmt(d.get('arch_pvalue'))}>={alpha}, ARCH etkisine iliskin yeterli kanit yoktur."))
    if dec:
        L.append(f"Teşhis: {dec.get('case')}. Onerilen birincil kovaryans/strateji: {dec.get('primary_cov')}.")
        L.append(dec.get("interpretation", ""))

    # Remediation etkisi
    cmp = A.get("comparison", pd.DataFrame())
    if len(cmp) > 0:
        best = cmp.iloc[0]
        L.append(f"Karşılaştırma: En yuksek temiz skora sahip model '{best['model']}' "
                 f"(skor={best['TEMIZ_SKOR']}). Heteroskedastisite fiilen azaldi mi? {best['hetero_fiilen_azaldi']}.")

    # Bounds
    b = A.get("bounds", {})
    if b.get("available") and "F" in b:
        L.append(f"ARDL bounds testi (case {b.get('case')}): F={_fmt(b.get('F'))}, "
                 f"%{100*(1-alpha):.0f} sinirlar [I(0)={_fmt(b.get('I0_lower'))}, I(1)={_fmt(b.get('I1_upper'))}] "
                 f"-> {b.get('conclusion')}.")
    elif b.get("error"):
        L.append(f"ARDL bounds testi hesaplanamadi: {b['error']}")

    # ECM
    e = A.get("ecm", {})
    if "ect_coef" in e:
        if e.get("ect_ok"):
            L.append(f"ECM: Hata duzeltme katsayisi {_fmt(e['ect_coef'])} negatif ve anlamli "
                     f"(p={_fmt(e['ect_pvalue'])}); uzun donem dengeye donus mekanizmasi calismaktadir.")
        else:
            L.append(f"ECM: Hata duzeltme katsayisi {_fmt(e['ect_coef'])} (p={_fmt(e['ect_pvalue'])}); "
                     f"negatif ve anlamli degilse mekanizma teorik olarak zayif olabilir.")

    # Uzun dönem
    lr = A.get("long_run", pd.DataFrame())
    if len(lr) > 0:
        parts = [f"{r['degisken']}: {_fmt(r['uzun_donem_carpan'])} (p={_fmt(r['p_deger'])})" for _, r in lr.iterrows()]
        L.append("Uzun donem carpanlar (delta yontemi): " + "; ".join(parts) + ".")

    # GARCH
    g = A.get("garch", {})
    if g.get("fit") is not None:
        L.append(f"ARCH/GARCH: ARCH-LM anlamli oldugundan artiklar uzerinde varyans modeli kuruldu; "
                 f"BIC'e gore en uygun: {g.get('best_name')}. Forecast araliklari icin donemsel degisen varyans kullanilmalidir.")
    elif not g.get("available", True):
        L.append("ARCH/GARCH: 'arch' paketi kurulu olmadigindan varyans modeli kurulamadi.")

    # Forecast notu
    f = A.get("forecast", {})
    if f.get("df") is not None:
        L.append("Forecast: Primary (Level) ARDL icin senaryo bazli recursive tahmin uretildi. "
                 "Tahmin araliklari SABIT varyans varsayimina dayanir; heteroskedastisite/ARCH varsa "
                 "araliklar dikkatle yorumlanmalidir.")
    if config["advanced"].get("bias_correction", True):
        L.append("Log model kullanildiginda seviyeye donusumde bias duzeltmesi "
                 "level = exp(log_tahmin + 0.5*sigma^2) uygulanmalidir (CONFIG['advanced']['bias_correction']).")
    return [x for x in L if x]

## 2.12 İleri analiz grafikleri (her biri ayrı)

In [ ]:
def make_advanced_plots(A, config):
    """İleri analiz grafiklerini ayrı ayrı üretir/kaydeder."""
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    from statsmodels.graphics.gofplots import qqplot
    outdir = Path(config["advanced"].get("plot_dir", "plots_advanced"))
    outdir.mkdir(parents=True, exist_ok=True)
    prim = A.get("primary")
    if prim is None:
        return []
    res = prim["fit"]
    y, fitted, resid = prim["y"], res.fittedvalues, res.resid
    paths = []

    def _save(fig, nm):
        p = outdir / nm
        fig.savefig(p, dpi=110, bbox_inches="tight")
        paths.append(str(p))
        try:
            plt.show()
        except Exception:
            pass
        plt.close(fig)

    def _fig():
        return plt.subplots(figsize=(10, 4))

    try:
        fig, ax = _fig(); y.plot(ax=ax, label="gercek", color="#1F4E78")
        fitted.plot(ax=ax, label="fitted", color="#E1701A"); ax.legend(); ax.grid(alpha=0.3)
        ax.set_title("Primary: Gercek vs Fitted"); _save(fig, "a01_actual_fitted.png")

        fig, ax = _fig(); ax.plot(resid.index, resid.values, color="#555"); ax.axhline(0, color="red", lw=1)
        ax.set_title("Artiklar"); ax.grid(alpha=0.3); _save(fig, "a02_residuals.png")

        sr = resid / (resid.std() or 1.0)
        fig, ax = _fig(); ax.plot(sr.index, sr.values, color="#555")
        for k in (-3, -2, 2, 3): ax.axhline(k, color="red", lw=0.6, ls="--")
        ax.set_title("Standardize artiklar"); ax.grid(alpha=0.3); _save(fig, "a03_std_residuals.png")

        fig, ax = _fig(); ax.scatter(fitted.values, resid.values, s=12, color="#1F4E78")
        ax.axhline(0, color="red", lw=1); ax.set_xlabel("fitted"); ax.set_ylabel("artik")
        ax.set_title("Artik vs Kestirim"); ax.grid(alpha=0.3); _save(fig, "a04_resid_vs_fitted.png")

        fig, ax = _fig(); ax.plot(resid.index, (resid.values ** 2), color="#7030A0")
        ax.set_title("Artik karesi (zaman)"); ax.grid(alpha=0.3); _save(fig, "a05_resid_squared.png")

        rv = (resid ** 2).rolling(12, min_periods=4).mean()
        fig, ax = _fig(); rv.plot(ax=ax, color="#C00000")
        ax.set_title("Rolling artik varyansi (W=12)"); ax.grid(alpha=0.3); _save(fig, "a06_rolling_var.png")

        fig, ax = plt.subplots(figsize=(8, 4)); plot_acf(resid.values, ax=ax, lags=min(24, len(resid) // 2 - 1))
        ax.set_title("Artik ACF"); _save(fig, "a07_acf.png")
        fig, ax = plt.subplots(figsize=(8, 4)); plot_pacf(resid.values, ax=ax, lags=min(24, len(resid) // 2 - 1), method="ywm")
        ax.set_title("Artik PACF"); _save(fig, "a08_pacf.png")

        fig = qqplot(resid.values, line="s"); fig.set_size_inches(6, 5)
        fig.suptitle("Q-Q grafigi"); _save(fig, "a09_qq.png")

        try:
            rr = recursive_olsresiduals(sm.OLS(y, sm.add_constant(prim["X"], has_constant="add")).fit())
            cusum = rr[5]
            fig, ax = _fig(); ax.plot(range(len(cusum)), cusum, color="#1F4E78")
            ax.set_title("CUSUM"); ax.grid(alpha=0.3); _save(fig, "a10_cusum.png")
        except Exception:
            pass

        f = A.get("forecast", {})
        if f.get("df") is not None:
            fdf = f["df"]
            fig, ax = _fig()
            y.iloc[-min(36, len(y)):].plot(ax=ax, label="gercek", color="#1F4E78")
            ax.plot(fdf["date"], fdf["tahmin"], color="#C00000", marker="o", ms=3, label="tahmin")
            ax.fill_between(fdf["date"], fdf["alt_ci"], fdf["ust_ci"], color="#C00000", alpha=0.15)
            ax.legend(); ax.grid(alpha=0.3); ax.set_title("Forecast (sabit varyans araligi)")
            _save(fig, "a11_forecast.png")

        cmp = A.get("comparison", pd.DataFrame())
        if len(cmp) > 0:
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.bar(cmp["model"], cmp["TEMIZ_SKOR"], color="#1F4E78")
            ax.set_title("Model karsilastirma: TEMIZ_SKOR"); plt.xticks(rotation=45, ha="right")
            ax.grid(alpha=0.3, axis="y"); _save(fig, "a12_model_scores.png")

        # hetero öncesi/sonrası varyans (primary vs en iyi remediation)
        recs = A["records"]
        wf = next((r for r in recs if r.get("used_wls") or r.get("used_fgls")), None)
        if wf is not None:
            fig, ax = _fig()
            ax.plot(resid.index, resid.values ** 2, label="primary artik^2", color="#1F4E78", alpha=0.7)
            wr = wf["fit"].resid
            ax.plot(wr.index, (wr.values ** 2), label=f"{wf['name']} artik^2", color="#E1701A", alpha=0.7)
            ax.legend(); ax.grid(alpha=0.3); ax.set_title("Heteroskedastisite oncesi/sonrasi artik^2")
            _save(fig, "a13_hetero_before_after.png")
    except Exception as e:
        print("   Ileri grafik notu:", e)
    return paths

## 2.13 İleri analiz Excel çıktısı (20 sayfa)

In [ ]:
def export_advanced_excel(A, config):
    """İleri analizin tüm sonuçlarını 20 sayfalık düzenli bir Excel dosyasına yazar."""
    path = config["advanced"].get("output_excel_path", "econometric_diagnosis_output.xlsx")
    alpha = _adv_alpha(config)
    with pd.ExcelWriter(path, engine="xlsxwriter", datetime_format="yyyy-mm-dd",
                        date_format="yyyy-mm-dd") as writer:
        # 1 Config
        cfgd = {**{f"advanced.{k}": v for k, v in config["advanced"].items()},
                "target_col": config.get("target_col"), "frequency": config.get("frequency"),
                "significance_level": alpha}
        _write_sheet(writer, _kv(cfgd), "Config")
        # 2 Data_Quality
        _write_sheet(writer, A.get("data_quality", pd.DataFrame()), "Data_Quality")
        # 3 Descriptive_Stats (+ korelasyon aynı sayfada altına)
        _write_sheet(writer, A.get("descriptive", pd.DataFrame()), "Descriptive_Stats")
        # 4 Stationarity
        _write_sheet(writer, A.get("stationarity", pd.DataFrame()), "Stationarity_Tests",
                     pval_cols=["ADF_duzey_p", "KPSS_duzey_p", "ADF_fark_p", "KPSS_fark_p"])
        # 5 Lag_Search
        _write_sheet(writer, A.get("lag_search", pd.DataFrame()), "Lag_Search")
        # 6 Model_Comparison
        _write_sheet(writer, A.get("comparison", pd.DataFrame()), "Model_Comparison",
                     pval_cols=["BG_p", "LjungBox_p", "BP_p", "White_p", "ARCH_LM_p", "RESET_p", "JB_p"])
        # 7 Coefficients (tümü)
        coefs = []
        for r in A["records"]:
            c = r["coef_df"].copy(); c.insert(0, "model", r["name"]); coefs.append(c)
        _write_sheet(writer, (pd.concat(coefs, ignore_index=True) if coefs else pd.DataFrame()),
                     "Coefficients", pval_cols=["p_value"])
        # 8 Robust_Coefficients
        pr = A.get("primary_robust")
        _write_sheet(writer, (pr["coef_df"] if pr else pd.DataFrame()), "Robust_Coefficients",
                     pval_cols=["p_value"])
        # 9 Short_Run
        _write_sheet(writer, A.get("short_run", pd.DataFrame()), "Short_Run_Effects", pval_cols=["p_deger"])
        # 10 Long_Run
        _write_sheet(writer, A.get("long_run", pd.DataFrame()), "Long_Run_Effects", pval_cols=["p_deger"])
        # 11 Bounds
        _write_sheet(writer, _kv(A.get("bounds", {})), "Bounds_Test")
        # 12 Diagnostics
        drows = []
        for r in A["records"]:
            d = r["diag"]
            drows.append({"model": r["name"], "BG_p": d.get("bg_pvalue"), "LjungBox_p": d.get("ljungbox_pvalue"),
                          "DW": d.get("durbin_watson"), "BP_p": d.get("bp_pvalue"), "White_p": d.get("white_pvalue"),
                          "GQ_p": d.get("gq_pvalue"), "ARCH_LM_p": d.get("arch_pvalue"),
                          "RESET_p": d.get("reset_pvalue"), "JB_p": d.get("jb_pvalue"), "max_VIF": r["max_vif"]})
        _write_sheet(writer, pd.DataFrame(drows), "Diagnostics",
                     pval_cols=["BG_p", "LjungBox_p", "BP_p", "White_p", "GQ_p", "ARCH_LM_p", "RESET_p", "JB_p"])
        # 13 Heteroskedasticity
        hrows = []
        base_hetero, _, _, _ = _hetero_flags(A.get("primary", A["records"][0]), alpha) if A["records"] else (False, 0, 0, 0)
        for r in A["records"]:
            hrows.append({"model": r["name"], "BP_p": r["diag"].get("bp_pvalue"),
                          "White_p": r["diag"].get("white_pvalue"), "ARCH_LM_p": r["diag"].get("arch_pvalue"),
                          "BP_p_sonrasi": r["diag"].get("bp_pvalue_after", np.nan),
                          "kovaryans": r["cov_type"], "hetero_fiilen_azaldi": _record_hetero_reduced(r, base_hetero, alpha)})
        het_df = pd.DataFrame(hrows)
        dec = A.get("decision", {})
        het_df = pd.concat([het_df, pd.DataFrame([{"model": "== TESHIS ==", "kovaryans": dec.get("case", ""),
                                                   "hetero_fiilen_azaldi": dec.get("primary_cov", "")}])],
                           ignore_index=True)
        _write_sheet(writer, het_df, "Heteroskedasticity", pval_cols=["BP_p", "White_p", "ARCH_LM_p", "BP_p_sonrasi"])
        # 14 ARCH_GARCH
        g = A.get("garch", {})
        garch_df = g.get("table", pd.DataFrame())
        if g.get("best_name"):
            garch_df = pd.concat([garch_df, pd.DataFrame([{"model": "EN IYI", "AIC": g.get("best_name")}])],
                                 ignore_index=True)
        if len(garch_df) == 0:
            garch_df = _kv({"durum": g.get("note", "GARCH calistirilmadi")})
        _write_sheet(writer, garch_df, "ARCH_GARCH")
        # 15 Structural_Breaks
        sb = []
        for b in A.get("break_dates", []):
            sb.append({"tur": "kirilma_tarihi", "deger": b})
        for ch in A.get("chow", []):
            sb.append({"tur": f"Chow_{ch.get('break_date')}", "deger": f"F={_fmt(ch.get('F'))}, p={_fmt(ch.get('pvalue'))}"})
        cu = A.get("cusum", {})
        sb.append({"tur": "CUSUM", "deger": f"stat={_fmt(cu.get('stat'))}, p={_fmt(cu.get('pvalue'))}, stabil={cu.get('stable')}"})
        _write_sheet(writer, (pd.DataFrame(sb) if sb else pd.DataFrame([{"tur": "-", "deger": "kirilma bulunamadi"}])),
                     "Structural_Breaks")
        # 16 Outliers_Influence
        _write_sheet(writer, A.get("influence_flag", pd.DataFrame()), "Outliers_Influence", date_cols=["tarih"])
        # 17 Forecast_Scenarios
        _write_sheet(writer, build_scenarios_table(config), "Forecast_Scenarios")
        # 18 Forecast_Output
        f = A.get("forecast", {})
        fdf = f.get("df") if isinstance(f, dict) else None
        _write_sheet(writer, (fdf if fdf is not None else _kv({"durum": f.get("error", "forecast uretilmedi")})),
                     "Forecast_Output", date_cols=["date"])
        # 19 Automatic_Interpretation
        interp = A.get("interpretation", [])
        _write_sheet(writer, pd.DataFrame({"no": range(1, len(interp) + 1), "yorum": interp}), "Automatic_Interpretation")
        # 20 Errors_Warnings
        ew = [{"tur": "uyari", "mesaj": w} for w in A.get("warnings", [])]
        ew += [{"tur": f"hata ({e['adim']})", "mesaj": e["hata"]} for e in A.get("errors", [])]
        if not ew:
            ew = [{"tur": "info", "mesaj": "Uyari/hata yok."}]
        _write_sheet(writer, pd.DataFrame(ew), "Errors_Warnings")
    return path

## 2.14 İleri analizi çalıştır

Bu hücre ileri analizi çalıştırır ve `econometric_diagnosis_output.xlsx` üretir.
`CONFIG["advanced"]["enabled"]=False` ise atlanır. Hata olsa bile ana notebook
çıktısı etkilenmez.

In [ ]:
try:
    print("=" * 70)
    print("BOLUM 2 — Ileri heteroskedastisite teshis ve analiz")
    ADV = run_advanced_analysis(prepared_df, CONFIG, forecast_results)
    if ADV is None:
        print("Ileri analiz kapali (CONFIG['advanced']['enabled']=False).")
    else:
        n_models = len(ADV.get("records", []))
        print(f"  Kurulan/degerlendirilen model sayisi: {n_models}")
        if ADV.get("decision"):
            print(f"  Heteroskedastisite teshisi: {ADV['decision'].get('case')} "
                  f"-> {ADV['decision'].get('primary_cov')}")
        if CONFIG["advanced"].get("make_plots", True):
            try:
                make_advanced_plots(ADV, CONFIG)
            except Exception as e:
                print("  Ileri grafik hatasi:", e)
        adv_path = export_advanced_excel(ADV, CONFIG)
        print(f"  Ileri analiz Excel: {adv_path}")
        print("\n  --- Otomatik yorum (ozet) ---")
        for line in ADV.get("interpretation", [])[:12]:
            print("   -", line)
    print("=" * 70)
except Exception as _e:
    import traceback
    print("Ileri analiz calistirilamadi (ana notebook etkilenmedi):", _e)
    traceback.print_exc()

# BÖLÜM 3 — ARDL Heteroskedastisite Denetimi (gerçek statsmodels ARDL)

Bu bölüm, ARDL heteroskedastisite işleyişini **teknik olarak** denetler. Bölüm 1
ve 2'yi değiştirmez; `CONFIG["ardl_audit"]["enabled"]` ile açılır. Bölüm 2'deki
"advanced" katman ARDL'yi elle kurulan bir OLS tasarımıyla yaklaşıklıyordu ve
`ardl_order`/`ar_lags`/`dl_lags`/`causal` gibi yapıları göstermiyordu. Bu bölüm
**gerçek `statsmodels.tsa.ardl.ARDL`** nesnesi kurar ve şu ayrımı korur:

- HC3 artıkların yapısını **değiştirmez**; yalnızca standart hata, t, p ve GA'yı düzeltir.
- Bu yüzden HC3 sonrası Breusch–Pagan'ın anlamlı kalması **normaldir**.
- ARDL `p`/`q` yalnızca **ortalama denklemi** belirler; hata varyansını düzeltmez.

In [ ]:
def _ardl_resolve(prepared_df, config):
    """Denetim için bağımlı (seviye) seri ve bağımsız (seviye) DataFrame'i çözer."""
    ac = config.get("ardl_audit", {})
    meta = prepared_df.attrs["meta"]
    dep = ac.get("dependent") or config.get("target_col") or meta["target_base"]
    inds = ac.get("independent")
    if not inds:
        inds = config.get("advanced", {}).get("independent_variables") or \
            config.get("model_specs", {}).get(config.get("advanced", {}).get("base_model", "OLS"), {}).get("exog", [])
    inds = [v for v in inds if v in prepared_df.columns]
    # Tek indeksleme ile al (concat'in attrs karşılaştırma hatasından kaçın) ve
    # attrs'ı temizle ki sonraki pandas işlemleri prepared_df meta'sına takılmasın.
    data = prepared_df[[dep] + inds].apply(pd.to_numeric, errors="coerce").dropna()
    data = data.copy()
    data.attrs = {}
    return dep, inds, data


def _reconstruct_ardl_design(res, model, y, xdf, trend):
    """model._x yoksa ar_lags/dl_lags/trend'den tasarım matrisini yeniden kurar (fallback)."""
    eff = res.resid.index
    cols = {}
    if trend in ("c", "ct"):
        cols["const"] = pd.Series(1.0, index=y.index)
    for L in list(model.ar_lags or []):
        cols[f"{y.name}.L{L}"] = y.shift(L)
    for xn, lags in (model.dl_lags or {}).items():
        for L in lags:
            cols[f"{xn}.L{L}"] = xdf[xn].shift(L)
    if trend == "ct":
        cols["trend"] = pd.Series(np.arange(len(y)), index=y.index, dtype=float)
    design = pd.DataFrame(cols).reindex(eff)
    return design.values, list(design.columns)


def get_ardl_design(res, model, y, xdf, trend):
    """ARDL'nin gerçek tasarım matrisini güvenli biçimde döndürür (kaynağı belirtir)."""
    for obj in (model, getattr(res, "model", None)):
        cand = getattr(obj, "_x", None)
        if cand is not None:
            return np.asarray(cand, float), list(res.params.index), f"{type(obj).__name__}._x (private attribute)"
    X, names = _reconstruct_ardl_design(res, model, y, xdf, trend)
    return X, names, "manuel yeniden kurulum (ar_lags/dl_lags/trend fallback)"


def ensure_constant(X, names):
    """Tasarımda sabit terim yoksa ekler; varsa ikinci kez eklemez."""
    has_const = any(np.allclose(X[:, j], 1.0) for j in range(X.shape[1]))
    if has_const:
        return X, names, True
    Xc = sm.add_constant(X, has_constant="add")
    return Xc, ["const"] + list(names), False


def ardl_bic_grid(data, dep, inds, max_p, max_q, trend, causal, ic="bic"):
    """Tüm (p, q) adayları için AIC/BIC ızgarası — **sabit (ortak) etkin örneklemle**.

    Kritik düzeltme: Her adayı kendi örnekleminde tahmin etmek (statsmodels ARDL'nin
    varsayılanı) BIC'i karşılaştırılamaz kılar; çünkü farklı p farklı sayıda başlangıç
    gözlemini düşürür. Burada tüm adaylar `start = max(max_p, max_q)` gözleminden
    itibaren AYNI örneklem üzerinde tahmin edilir; böylece BIC/AIC gerçekten
    karşılaştırılabilir olur (singular/yetersiz olanlar atlanır).
    """
    n = len(data)
    start = int(max(max_p, max_q))
    yv = data[dep]
    rows = []
    for p in range(0, max_p + 1):
        for q in range(0, max_q + 1):
            cols = {}
            if trend in ("c", "ct"):
                cols["const"] = np.ones(n)
            for L in range(1, p + 1):
                cols[f"{dep}.L{L}"] = yv.shift(L).values
            for xn in inds:
                lo = 1 if causal else 0
                for L in range(lo, q + 1):
                    cols[f"{xn}.L{L}"] = data[xn].shift(L).values
            if trend == "ct":
                cols["trend"] = np.arange(n, dtype=float)
            X = np.column_stack(list(cols.values())) if cols else np.empty((n, 0))
            yy = yv.values
            Xs, ys = X[start:], yy[start:]
            good = ~np.any(np.isnan(np.column_stack([ys.reshape(-1, 1), Xs])), axis=1) if Xs.shape[1] else ~np.isnan(ys)
            if good.sum() < Xs.shape[1] + 3 or Xs.shape[1] == 0:
                rows.append({"p": p, "q": q, "nobs": int(good.sum()), "AIC": np.nan, "BIC": np.nan,
                             "HQIC": np.nan, "status": "atlandi: yetersiz/singular"})
                continue
            try:
                r = sm.OLS(ys[good], Xs[good]).fit()
                kk, nn = len(r.params), r.nobs
                hqic = float(-2 * r.llf + 2 * kk * np.log(np.log(nn))) if nn > 2 else np.nan
                rows.append({"p": p, "q": q, "nobs": int(nn), "AIC": float(r.aic),
                             "BIC": float(r.bic), "HQIC": hqic, "status": "OK"})
            except Exception as e:
                rows.append({"p": p, "q": q, "nobs": int(good.sum()), "AIC": np.nan, "BIC": np.nan,
                             "HQIC": np.nan, "status": f"atlandi: {type(e).__name__}"})
    df = pd.DataFrame(rows)
    crit = {"aic": "AIC", "bic": "BIC", "hqic": "HQIC"}.get(ic.lower(), "BIC")
    valid = df.dropna(subset=[crit])
    best = None if len(valid) == 0 else (int(valid.loc[valid[crit].idxmin(), "p"]),
                                         int(valid.loc[valid[crit].idxmin(), "q"]))
    return df, best, crit

## 3.1 Test yardımcıları: BP (klasik+Koenker), çoklu-lag ARCH-LM, güvenli White

In [ ]:
def bp_test_rows(resid, X_bp, nobs, k_reg, alpha):
    """Breusch–Pagan'ı tam ARDL tasarım matrisiyle klasik ve Koenker biçiminde çalıştırır."""
    rows = []
    for variant, robust in [("classic", False), ("koenker", True)]:
        try:
            lm, lm_p, f, f_p = het_breuschpagan(np.asarray(resid), np.asarray(X_bp), robust=robust)
            interp = bp_interpretation(lm_p, f_p, alpha)
            rows.append({"test_name": "Breusch-Pagan", "variant": variant,
                         "lm_stat": float(lm), "lm_pvalue": float(lm_p),
                         "f_stat": float(f), "f_pvalue": float(f_p),
                         "nobs": int(nobs), "n_regressors": int(k_reg), "interpretation": interp})
        except Exception as e:
            rows.append({"test_name": "Breusch-Pagan", "variant": variant, "lm_stat": np.nan,
                         "lm_pvalue": np.nan, "f_stat": np.nan, "f_pvalue": np.nan,
                         "nobs": int(nobs), "n_regressors": int(k_reg),
                         "interpretation": f"hesaplanamadi: {e}"})
    return rows


def bp_interpretation(lm_p, f_p, alpha):
    """LM ve F ayrımıyla BP yorumu (küçük/orta örneklemde LM aşırı reddedebilir)."""
    if f_p is not None and not np.isnan(f_p) and f_p < alpha:
        return "F versiyonuna gore heteroskedastisite var."
    if (lm_p is not None and not np.isnan(lm_p) and lm_p < alpha) and (f_p is None or np.isnan(f_p) or f_p >= alpha):
        return ("LM testi anlamli ancak F testi anlamli degil. "
                "Kucuk/orta orneklemde LM testi asiri reddediyor olabilir.")
    return "Heteroskedastisite bulunduguna iliskin yeterli kanit yok."


def arch_test_rows(resid, k_params, lags, alpha):
    """ARDL artıklarında farklı laglerle ARCH-LM; yetersiz gözlemli lagleri atlar."""
    r = np.asarray(pd.Series(resid).dropna())
    n = len(r)
    rows = []
    for lag in lags:
        if n <= lag + k_params + 10:
            rows.append({"lag": lag, "arch_lm_stat": np.nan, "arch_lm_pvalue": np.nan,
                         "arch_f_stat": np.nan, "arch_f_pvalue": np.nan,
                         "yorum": "SKIPPED: yetersiz gozlem"})
            continue
        try:
            lm, lm_p, f, f_p = het_arch(r, nlags=lag, ddof=k_params)
            yorum = ("ARCH etkisi var (kosullu degisen varyans)" if f_p < alpha
                     else "ARCH etkisine iliskin yeterli kanit yok")
            rows.append({"lag": lag, "arch_lm_stat": float(lm), "arch_lm_pvalue": float(lm_p),
                         "arch_f_stat": float(f), "arch_f_pvalue": float(f_p), "yorum": yorum})
        except Exception as e:
            rows.append({"lag": lag, "arch_lm_stat": np.nan, "arch_lm_pvalue": np.nan,
                         "arch_f_stat": np.nan, "arch_f_pvalue": np.nan, "yorum": f"hata: {e}"})
    return rows


def white_test_safe(resid, X_bp, alpha):
    """White testini serbestlik derecesi güvenliğiyle çalıştırır; aşırı parametrelenirse atlar."""
    n, p = np.asarray(X_bp).shape
    needed = p * (p + 1) // 2  # kare + etkileşim terimleri (sabit dahil tasarım)
    if n - needed < 10:
        return {"status": f"SKIPPED: yetersiz serbestlik derecesi (nobs={n}, aux_terim~{needed})",
                "lm_stat": np.nan, "lm_pvalue": np.nan, "f_stat": np.nan, "f_pvalue": np.nan}
    try:
        lm, lm_p, f, f_p = het_white(np.asarray(resid), np.asarray(X_bp))
        return {"status": "OK", "lm_stat": float(lm), "lm_pvalue": float(lm_p),
                "f_stat": float(f), "f_pvalue": float(f_p)}
    except Exception as e:
        return {"status": f"SKIPPED: {type(e).__name__} (muhtemelen rank eksikligi)",
                "lm_stat": np.nan, "lm_pvalue": np.nan, "f_stat": np.nan, "f_pvalue": np.nan}


def _rmse_resid(resid):
    r = np.asarray(pd.Series(resid).dropna())
    return float(np.sqrt(np.mean(r ** 2))) if len(r) else np.nan


def diag_row_for_model(name, y_eff, X_design, resid, k_params, aic, bic, alpha, arch_lag=12):
    """Bir model için tekrar-test satırı (Koenker-BP, White, ARCH, BG, RESET, AIC/BIC/RMSE)."""
    row = {"model": name, "AIC": aic, "BIC": bic, "RMSE": _rmse_resid(resid)}
    try:
        _, _, _, bpf = het_breuschpagan(np.asarray(resid), np.asarray(X_design), robust=True)
        row["BP_Koenker_F_p"] = float(bpf)
    except Exception:
        row["BP_Koenker_F_p"] = np.nan
    wt = white_test_safe(resid, X_design, alpha)
    row["White_p"] = wt["f_pvalue"]
    row["White_status"] = wt["status"]
    n = len(np.asarray(pd.Series(resid).dropna()))
    if n > arch_lag + k_params + 10:
        try:
            _, _, _, af = het_arch(np.asarray(resid), nlags=arch_lag, ddof=k_params)
            row["ARCH12_p"] = float(af)
        except Exception:
            row["ARCH12_p"] = np.nan
    else:
        row["ARCH12_p"] = np.nan
    try:
        aux = sm.OLS(np.asarray(y_eff), np.asarray(X_design)).fit()
        row["BG_p"] = float(acorr_breusch_godfrey(aux, nlags=min(12, max(1, n // 4)))[1])
        row["RESET_p"] = float(linear_reset(aux, power=2, use_f=True).pvalue)
    except Exception:
        row["BG_p"] = np.nan
        row["RESET_p"] = np.nan
    return row

## 3.2 Ana denetim akışı

In [ ]:
def run_ardl_hetero_audit(prepared_df, config):
    """Gerçek ARDL üzerinden tüm heteroskedastisite denetimini yürütür."""
    ac = config.get("ardl_audit", {})
    if not ac.get("enabled", False):
        return None
    if not HAVE_ADV.get("uecm", HAVE.get("ardl", False)) and not HAVE.get("ardl", False):
        return {"error": "statsmodels ARDL bulunamadi; denetim atlanamadi.", "checks": []}
    alpha = float(ac.get("alpha", 0.05))
    A = {"errors": [], "notes": [], "alpha": alpha}

    dep, inds, data = _ardl_resolve(prepared_df, config)
    A["dependent"], A["independent"] = dep, inds
    if not inds:
        return {"error": "Bagimsiz degisken bulunamadi (CONFIG['ardl_audit']['independent']).", "checks": []}

    trend, causal = ac.get("trend", "c"), bool(ac.get("causal", False))

    # 1) Lag seçimi ve gerçek ARDL yapısı
    sel = ardl_select_order(data[dep], maxlag=int(ac.get("max_p", 8)), exog=data[inds],
                            maxorder=int(ac.get("max_q", 4)), ic=ac.get("ic", "bic"),
                            trend=trend, causal=causal)
    model = sel.model
    res_classic = model.fit()
    res_hc3 = model.fit(cov_type="HC3")
    A["ardl_order"] = getattr(model, "ardl_order", None)
    A["ar_lags"] = getattr(model, "ar_lags", None)
    A["dl_lags"] = getattr(model, "dl_lags", None)
    A["causal"] = getattr(model, "causal", causal)
    A["trend"] = getattr(model, "trend", trend)
    A["res_classic"], A["res_hc3"], A["model"] = res_classic, res_hc3, model

    grid, best, crit = ardl_bic_grid(data, dep, inds, int(ac.get("max_p", 8)),
                                     int(ac.get("max_q", 4)), trend, causal, ac.get("ic", "bic"))
    A["bic_grid"], A["grid_best"], A["grid_crit"] = grid, best, crit
    p_sel = max(A["ar_lags"]) if A["ar_lags"] else 0
    q_sel = max([max(v) for v in (A["dl_lags"] or {}).values()] or [0])
    A["p_selected"], A["q_selected"] = p_sel, q_sel

    # 2) HC3 vs klasik
    A["hc3_vs_classic"] = {
        "same_params": bool(np.allclose(res_classic.params.values, res_hc3.params.values)),
        "same_fitted": bool(np.allclose(res_classic.fittedvalues.values, res_hc3.fittedvalues.values)),
        "same_resid": bool(np.allclose(res_classic.resid.values, res_hc3.resid.values)),
        "diff_bse": bool(not np.allclose(res_classic.bse.values, res_hc3.bse.values)),
        "diff_pvalues": bool(not np.allclose(res_classic.pvalues.values, res_hc3.pvalues.values)),
    }

    # 3) Tam tasarım matrisi + sabit + boyut kontrolleri
    y = data[dep].copy(); y.name = dep
    X_bp, names, src = get_ardl_design(res_classic, model, y, data[inds], trend)
    A["design_source"] = src
    resid = np.asarray(res_classic.resid)
    A["assert_rows"] = (len(resid) == X_bp.shape[0])
    A["assert_cols"] = (X_bp.shape[1] == len(res_classic.params))
    X_bp, names, had_const = ensure_constant(X_bp, names)
    A["had_constant"] = had_const
    A["design_shape"] = X_bp.shape
    k_params = len(res_classic.params)
    y_eff = (res_classic.fittedvalues + res_classic.resid).values  # etkin örneklemdeki gerçek y

    # 4-5) BP klasik + Koenker (tam matris)
    A["bp_rows"] = bp_test_rows(resid, X_bp, res_classic.nobs, X_bp.shape[1], alpha)

    # 6) HC3 sonrası artıklar aynı mı
    A["resid_identical_hc3"] = bool(np.allclose(res_classic.resid.values, res_hc3.resid.values))

    # 7) ARCH-LM çoklu lag
    A["arch_rows"] = arch_test_rows(resid, k_params, ac.get("arch_lags", [1, 3, 6, 12]), alpha)

    # 8) Güvenli White
    A["white"] = white_test_safe(resid, X_bp, alpha)

    # 9-10) Yapısal kırılma / etkili gözlem (etkin örneklem üzerinde OLS proxy)
    eff_index = res_classic.resid.index
    Xdf_eff = pd.DataFrame(X_bp, index=eff_index, columns=names)
    yser_eff = pd.Series(y_eff, index=eff_index, name=dep)
    Xnc = Xdf_eff.drop(columns=[c for c in Xdf_eff.columns if c == "const"])
    A["cusum"] = guard_call(lambda: cusum_stability(yser_eff, Xnc), A, "cusum") or {}
    infl = guard_call(lambda: influence_analysis(yser_eff, Xnc), A, "influence")
    A["influence_all"], A["influence_flag"] = (infl if infl is not None else (pd.DataFrame(), pd.DataFrame()))
    brks = list(ac.get("manual_break_dates", [])) + \
        [str(d.date()) for d in (guard_call(lambda: detect_residual_jumps(yser_eff, Xnc), A, "jumps") or [])]
    A["break_dates"] = list(dict.fromkeys(brks))

    # 11-12) Alternatif spesifikasyonlar (teknik olarak uygunsa)
    A["alternatives"] = build_ardl_alternatives(A, data, dep, inds, model, X_bp, names, y_eff,
                                                eff_index, k_params, alpha, config)

    # 13-15) Kontrol tablosu + yorum
    A["control_table"] = build_ardl_control_table(A)
    A["interpretation"] = build_ardl_audit_interpretation(A, config)
    return A


def guard_call(fn, A, label):
    try:
        return fn()
    except Exception as e:
        A["errors"].append({"adim": label, "hata": f"{type(e).__name__}: {e}"})
        return None

## 3.3 Alternatif spesifikasyonlar, kontrol tablosu, yorum, grafik ve Excel

In [ ]:
def _fit_ardl_like(endog, exog, p, q, trend, causal, fixed=None):
    """Aynı (p,q) yapısıyla bir ARDL varyantı fit eder; tasarım + artık döndürür."""
    m = _ARDL(endog, lags=p, exog=exog, order=q, trend=trend, causal=causal, fixed=fixed)
    r = m.fit()
    xa = getattr(m, "_x", None)
    X = np.asarray(xa, float) if xa is not None else None
    resid = np.asarray(r.resid)
    y_eff = (r.fittedvalues + r.resid).values
    return r, X, resid, y_eff, len(r.params)


def build_ardl_alternatives(A, data, dep, inds, model, X_bp, names, y_eff, eff_index,
                            k_params, alpha, config):
    """HC3 ana modeli koruyarak alternatif spesifikasyonları teknik uygunsa dener."""
    p, q = A["p_selected"], A["q_selected"]
    trend, causal = A["trend"], A["causal"]
    rows = []
    # Ana model (HC3 çıkarım modeli) — testler klasik artıklarla
    rows.append(diag_row_for_model("ARDL_HC3 (ana model)", y_eff, X_bp, np.asarray(A["res_classic"].resid),
                                   k_params, float(A["res_classic"].aic), float(A["res_classic"].bic), alpha))
    # Log-Y
    if (data[dep] > 0).all():
        try:
            r, X, res, ye, k = _fit_ardl_like(np.log(data[dep]), data[inds], p, q, trend, causal)
            if X is not None:
                rows.append(diag_row_for_model("ARDL_LogY", ye, X, res, k, float(r.aic), float(r.bic), alpha))
        except Exception as e:
            A["errors"].append({"adim": "alt_logy", "hata": str(e)})
    else:
        A["notes"].append("Log-Y atlandi: bagimli degiskende sifir/negatif deger var.")
    # Fark
    try:
        r, X, res, ye, k = _fit_ardl_like(data[dep].diff().dropna(), data[inds].reindex(data[dep].diff().dropna().index),
                                          p, q, trend, causal)
        if X is not None:
            rows.append(diag_row_for_model("ARDL_Difference", ye, X, res, k, float(r.aic), float(r.bic), alpha))
    except Exception as e:
        A["errors"].append({"adim": "alt_diff", "hata": str(e)})
    # Yapısal kırılma / pulse dummy'li ARDL
    if A.get("break_dates"):
        for kind, pref in [("level", "D_lvl"), ("pulse", "D_pulse")]:
            try:
                D = make_break_dummies(data.index, A["break_dates"], kinds=(kind,))
                r, X, res, ye, k = _fit_ardl_like(data[dep], data[inds], p, q, trend, causal, fixed=D)
                if X is not None:
                    rows.append(diag_row_for_model(f"ARDL_{pref}", ye, X, res, k, float(r.aic), float(r.bic), alpha))
            except Exception as e:
                A["errors"].append({"adim": f"alt_{pref}", "hata": str(e)})
    # WLS / FGLS (ARDL tasarım matrisi üzerinde, aynı etkin örneklem)
    Xnc = pd.DataFrame(X_bp, index=eff_index, columns=names).drop(
        columns=[c for c in names if c == "const"])
    ye_ser = pd.Series(y_eff, index=eff_index)
    try:
        w = fit_wls_variants(ye_ser, Xnc, config)
        if w:
            rows.append(diag_row_for_model(f"WLS({w['weight_name']})", y_eff, X_bp,
                                           np.asarray(w["res"].resid), len(w["res"].params),
                                           float(w["res"].aic), float(w["res"].bic), alpha))
    except Exception as e:
        A["errors"].append({"adim": "alt_wls", "hata": str(e)})
    try:
        fg = fit_fgls(ye_ser, Xnc, config)
        if fg:
            rows.append(diag_row_for_model("FGLS", y_eff, X_bp, np.asarray(fg["res"].resid),
                                           len(fg["res"].params), float(fg["res"].aic),
                                           float(fg["res"].bic), alpha))
    except Exception as e:
        A["errors"].append({"adim": "alt_fgls", "hata": str(e)})
    return pd.DataFrame(rows)


def build_ardl_control_table(A):
    """Section 14 PASS/FAIL/WARNING/SKIPPED kontrol tablosu."""
    h = A["hc3_vs_classic"]
    bp_koenker = next((r for r in A["bp_rows"] if r["variant"] == "koenker"), {})
    arch_ok = any(not np.isnan(r.get("arch_lm_pvalue", np.nan)) for r in A["arch_rows"])
    grid_best = A.get("grid_best")
    order_match = (grid_best is not None and grid_best == (A["p_selected"], A["q_selected"]))

    def st(cond):
        return "PASS" if cond else "FAIL"

    rows = [
        ("HC3 katsayilari degistiriyor mu?", "Hayir", "Hayir" if h["same_params"] else "Evet", st(h["same_params"])),
        ("HC3 standart hatalari degistiriyor mu?", "Evet", "Evet" if h["diff_bse"] else "Hayir", st(h["diff_bse"])),
        ("HC3 artiklari degistiriyor mu?", "Hayir", "Hayir" if h["same_resid"] else "Evet", st(h["same_resid"])),
        ("BP testi tam ARDL matrisiyle mi yapiliyor?", "Evet", A.get("design_source", ""),
         st(A.get("assert_rows") and A.get("assert_cols"))),
        ("Sabit terim var mi?", "Evet", "Evet", "PASS"),
        ("Residual ve X satirlari esit mi?", "Evet", f"resid={len(A['res_classic'].resid)} vs X={A['design_shape'][0]}",
         st(A.get("assert_rows"))),
        ("Koenker-BP raporlaniyor mu?", "Evet", "Evet" if bp_koenker else "Hayir", st(bool(bp_koenker))),
        ("F p-degeri raporlaniyor mu?", "Evet", f"F_p={_fmt(bp_koenker.get('f_pvalue'))}", st("f_pvalue" in bp_koenker)),
        ("ARCH-LM farkli laglerle calisiyor mu?", "Evet", f"lags={[r['lag'] for r in A['arch_rows']]}", st(arch_ok)),
        ("Effective sample tarihleri dogru mu?", "Evet",
         f"{A['res_classic'].resid.index.min().date()}..{A['res_classic'].resid.index.max().date()}",
         st(A.get("assert_rows"))),
        (f"p={A['p_selected']} q={A['q_selected']} gercek model yapisiyla uyumlu mu?", "Evet",
         f"grid_min({A.get('grid_crit')})={grid_best}", "PASS" if order_match else "WARNING"),
    ]
    return pd.DataFrame(rows, columns=["Kontrol", "Beklenen", "Gerceklesen", "Durum"])


def build_ardl_audit_interpretation(A, config):
    """Section 15 otomatik Türkçe yorum + net kullanılabilirlik kararı."""
    alpha = A["alpha"]
    L = []
    L.append(f"BIC'ye gore secilen ARDL yapisi: ardl_order={A['ardl_order']}, ar_lags={A['ar_lags']}, "
             f"dl_lags={A['dl_lags']}, causal={A['causal']}, trend='{A['trend']}'.")
    dl = A["dl_lags"] or {}
    uniform = len({tuple(v) for v in dl.values()}) <= 1 if dl else True
    L.append(f"p={A['p_selected']} bagimli degiskenin {A['p_selected']} gecikmesini; q={A['q_selected']} her "
             f"bagimsiz degiskenin 0..{A['q_selected']} gecikmelerini icerir. Lag yapisi degisken bazindadir "
             f"({'tum degiskenlerde ayni' if uniform else 'degiskene gore farkli'}); cari donem x "
             f"{'DAHIL' if A['causal'] is False else 'HARIC (causal=True)'}.")
    gb = A.get("grid_best")
    if gb == (A["p_selected"], A["q_selected"]):
        L.append(f"Lag secimi dogrulandi: sabit (ortak) etkin orneklemli {A.get('grid_crit')} izgarasinin en "
                 f"dusuk degeri de p={gb[0]}, q={gb[1]} modeline ait.")
    else:
        L.append(f"Not: ardl_select_order per-DEGISKEN gecikme aramasi yapar; sabit-orneklem uniform (p,q) "
                 f"izgarasinin en iyisi {gb} iken secilen modelin uniform karsiligi ({A['p_selected']},{A['q_selected']}). "
                 f"Fark, degisken-bazli lag esnekliginden kaynaklanir (hata degil), ancak uniform bir yapi "
                 f"tercih edilirse {gb} adayi da degerlendirilebilir.")
    h = A["hc3_vs_classic"]
    L.append(f"HC3 dogru uygulanmis: katsayilar ayni ({h['same_params']}), fitted ayni ({h['same_fitted']}), "
             f"artiklar ayni ({h['same_resid']}); yalnizca standart hatalar ({h['diff_bse']}) ve p-degerleri "
             f"({h['diff_pvalues']}) degisiyor.")
    bpc = next((r for r in A["bp_rows"] if r["variant"] == "classic"), {})
    bpk = next((r for r in A["bp_rows"] if r["variant"] == "koenker"), {})
    L.append(f"Breusch-Pagan (klasik): LM_p={_fmt(bpc.get('lm_pvalue'))}, F_p={_fmt(bpc.get('f_pvalue'))}.")
    L.append(f"Breusch-Pagan (Koenker, oncelikli): LM_p={_fmt(bpk.get('lm_pvalue'))}, "
             f"F_p={_fmt(bpk.get('f_pvalue'))} -> {bpk.get('interpretation')}")
    arch_sig = [r for r in A["arch_rows"] if not np.isnan(r.get("arch_f_pvalue", np.nan)) and r["arch_f_pvalue"] < alpha]
    if arch_sig:
        arch_txt = ", ".join("lag%d F_p=%s" % (r["lag"], _fmt(r["arch_f_pvalue"])) for r in arch_sig)
        L.append(f"ARCH-LM: {arch_txt} -> kosullu degisen varyans (ARCH) kaniti var.")
    else:
        L.append("ARCH-LM: Test edilen laglerde kosullu degisen varyansa (ARCH) iliskin yeterli kanit yok.")
    wt = A["white"]
    L.append(f"White testi: {wt['status']}" + (f", F_p={_fmt(wt.get('f_pvalue'))}" if wt["status"] == "OK" else ""))
    # heteroskedastisite türü
    hetero = (bpk.get("f_pvalue") is not None and not np.isnan(bpk.get("f_pvalue", np.nan)) and bpk["f_pvalue"] < alpha)
    het_type = ("kosullu (ARCH tipi)" if arch_sig else ("kesitsel/seviyeye bagli" if hetero else "belirgin degil"))
    L.append(f"Heteroskedastisite turu: {het_type}.")
    if A.get("break_dates"):
        L.append(f"Olasi yapisal kirilma/aykiri tarih adaylari: {', '.join(A['break_dates'])}. "
                 f"CUSUM stabil={A.get('cusum', {}).get('stable')}.")
    # Zorunlu HC3 ifadesi
    L.append("HC3 uygulandiktan sonra Breusch-Pagan testinin anlamli kalmasi, HC3 duzeltmesinin basarisiz "
             "oldugu anlamina gelmez. HC3 model artiklarini ve hata varyansini degistirmez; yalnizca "
             "katsayilarin standart hatalarini heteroskedastisiteye karsi dayanikli hale getirir.")
    # Karar
    autocorr_ok = True  # kullanici: otokorelasyon anlamli degil
    if arch_sig:
        verdict = ("Ana model (HC3) katsayi CIKARIMI icin kullanilabilir; ancak ARCH etkisi nedeniyle "
                   "TAHMIN ARALIKLARI icin ek varyans modeli (ARCH/GARCH) gerekir.")
        alt = "ARDL ortalama denklemi + artik-tabanli GARCH(1,1)"
    elif hetero:
        verdict = ("Otokorelasyon yok; heteroskedastisite var. Ana model (HC3) katsayi cikarimI icin "
                   "kullanilabilir (robust standart hatalarla). Nokta tahminleri sapmasizdir.")
        alt = "Gerekirse WLS/FGLS ile etkinlik; ya da log/fark donusumu ve yapisal kirilma dummy'si"
    else:
        verdict = "Heteroskedastisiteye iliskin guclu kanit yok; ana model dogrudan kullanilabilir."
        alt = "Ek spesifikasyona gerek yok"
    L.append(f"SONUC: {verdict}")
    L.append(f"Onerilen alternatif spesifikasyon: {alt}.")
    return L


def make_ardl_audit_plots(A, config):
    """Section 9 grafikleri — hepsi etkin örnek (resid) indeksine hizalı, ayrı ayrı."""
    from statsmodels.graphics.gofplots import qqplot
    outdir = Path(config["ardl_audit"].get("plot_dir", "plots_ardl_audit"))
    outdir.mkdir(parents=True, exist_ok=True)
    res = A["res_classic"]
    resid = res.resid
    fitted = res.fittedvalues
    idx = resid.index  # EFFECTIVE SAMPLE
    paths = []

    def _save(fig, nm):
        p = outdir / nm
        fig.savefig(p, dpi=110, bbox_inches="tight")
        paths.append(str(p))
        try:
            plt.show()
        except Exception:
            pass
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 3.4)); ax.plot(idx, resid.values, color="#1F4E78")
    ax.axhline(0, color="red", lw=1); ax.set_title("1) Artik (etkin ornek)"); ax.grid(alpha=0.3); _save(fig, "b01_resid.png")
    fig, ax = plt.subplots(figsize=(10, 3.4)); ax.plot(idx, resid.values ** 2, color="#7030A0")
    ax.set_title("2) Artik karesi"); ax.grid(alpha=0.3); _save(fig, "b02_resid_sq.png")
    fig, ax = plt.subplots(figsize=(10, 3.4)); (resid ** 2).rolling(6, min_periods=2).mean().plot(ax=ax, color="#C00000")
    ax.set_title("3) 6 donemlik rolling varyans"); ax.grid(alpha=0.3); _save(fig, "b03_rollvar6.png")
    fig, ax = plt.subplots(figsize=(10, 3.4)); (resid ** 2).rolling(12, min_periods=3).mean().plot(ax=ax, color="#C00000")
    ax.set_title("4) 12 donemlik rolling varyans"); ax.grid(alpha=0.3); _save(fig, "b04_rollvar12.png")
    fig, ax = plt.subplots(figsize=(6, 4.5)); ax.scatter(fitted.values, resid.values, s=12, color="#1F4E78")
    ax.axhline(0, color="red", lw=1); ax.set_xlabel("fitted"); ax.set_ylabel("artik")
    ax.set_title("5) Fitted vs Artik"); ax.grid(alpha=0.3); _save(fig, "b05_fit_resid.png")
    fig, ax = plt.subplots(figsize=(6, 4.5)); ax.scatter(fitted.values, np.abs(resid.values), s=12, color="#E1701A")
    ax.set_xlabel("fitted"); ax.set_ylabel("|artik|"); ax.set_title("6) Fitted vs |Artik|"); ax.grid(alpha=0.3)
    _save(fig, "b06_fit_absresid.png")
    fig = qqplot(resid.values, line="s"); fig.set_size_inches(6, 5); fig.suptitle("7) Q-Q plot"); _save(fig, "b07_qq.png")
    return paths


def export_ardl_audit_excel(A, config):
    """Denetim sonuçlarını ayrı bir Excel dosyasına yazar."""
    path = config["ardl_audit"].get("output_excel_path", "ardl_hetero_audit.xlsx")
    with pd.ExcelWriter(path, engine="xlsxwriter", datetime_format="yyyy-mm-dd", date_format="yyyy-mm-dd") as w:
        struct = _kv({"ardl_order": A["ardl_order"], "ar_lags": A["ar_lags"], "dl_lags": A["dl_lags"],
                      "causal": A["causal"], "trend": A["trend"], "p_selected": A["p_selected"],
                      "q_selected": A["q_selected"], "design_source": A["design_source"],
                      "design_shape": A["design_shape"], "dependent": A["dependent"],
                      "independent": ", ".join(A["independent"])})
        _write_sheet(w, struct, "ARDL_Structure")
        _write_sheet(w, A.get("bic_grid", pd.DataFrame()), "BIC_Grid")
        _write_sheet(w, _kv(A["hc3_vs_classic"]), "HC3_vs_Classic")
        _write_sheet(w, pd.DataFrame(A["bp_rows"]), "BP_Tests",
                     pval_cols=["lm_pvalue", "f_pvalue"])
        _write_sheet(w, pd.DataFrame(A["arch_rows"]), "ARCH_LM",
                     pval_cols=["arch_lm_pvalue", "arch_f_pvalue"])
        _write_sheet(w, _kv(A["white"]), "White_Test")
        _write_sheet(w, A.get("alternatives", pd.DataFrame()), "Alternatives",
                     pval_cols=["BP_Koenker_F_p", "White_p", "ARCH12_p", "BG_p", "RESET_p"])
        _write_sheet(w, A.get("influence_flag", pd.DataFrame()), "Outliers_Influence", date_cols=["tarih"])
        _write_sheet(w, A.get("control_table", pd.DataFrame()), "Control_Table")
        interp = A.get("interpretation", [])
        _write_sheet(w, pd.DataFrame({"no": range(1, len(interp) + 1), "yorum": interp}), "Interpretation")
        ew = [{"tur": "not", "mesaj": n} for n in A.get("notes", [])] + \
             [{"tur": f"hata ({e['adim']})", "mesaj": e["hata"]} for e in A.get("errors", [])]
        _write_sheet(w, (pd.DataFrame(ew) if ew else pd.DataFrame([{"tur": "info", "mesaj": "yok"}])), "Errors_Warnings")
    return path

## 3.4 ARDL heteroskedastisite denetimini çalıştır

Bu hücre gerçek ARDL yapısını, HC3 vs klasik karşılaştırmasını, tam matrisli
BP (klasik+Koenker), çoklu-lag ARCH-LM, güvenli White, alternatifleri, kontrol
tablosunu ve Türkçe yorumu üretir; `ardl_hetero_audit.xlsx` yazar.

In [ ]:
try:
    print("=" * 70)
    print("BOLUM 3 — ARDL HETEROSKEDASTISITE DENETIMI")
    AUDIT = run_ardl_hetero_audit(prepared_df, CONFIG)
    if AUDIT is None:
        print("Denetim kapali (CONFIG['ardl_audit']['enabled']=False).")
    elif AUDIT.get("error"):
        print("Denetim calistirilamadi:", AUDIT["error"])
    else:
        print("\n[1] Secilen ARDL yapisi (gercek statsmodels ARDL):")
        print("    ardl_order =", AUDIT["ardl_order"])
        print("    ar_lags    =", AUDIT["ar_lags"])
        print("    dl_lags    =", AUDIT["dl_lags"])
        print("    causal     =", AUDIT["causal"], "| trend =", AUDIT["trend"])
        print(f"    p={AUDIT['p_selected']}, q={AUDIT['q_selected']} | BIC grid min = {AUDIT.get('grid_best')}")
        print("    tasarim matrisi kaynagi:", AUDIT["design_source"])
        print("\n[2] Kontrol tablosu:")
        print(AUDIT["control_table"].to_string(index=False))
        print("\n[3] BP testleri (tam ARDL matrisi):")
        print(pd.DataFrame(AUDIT["bp_rows"])[["variant", "lm_pvalue", "f_pvalue", "interpretation"]].to_string(index=False))
        print("\n[4] ARCH-LM (coklu lag):")
        print(pd.DataFrame(AUDIT["arch_rows"])[["lag", "arch_lm_pvalue", "arch_f_pvalue", "yorum"]].to_string(index=False))
        print("\n[5] Alternatif spesifikasyonlar:")
        print(AUDIT["alternatives"].to_string(index=False))
        if CONFIG["ardl_audit"].get("make_plots", True):
            try:
                make_ardl_audit_plots(AUDIT, CONFIG)
            except Exception as e:
                print("  Denetim grafik hatasi:", e)
        ap = export_ardl_audit_excel(AUDIT, CONFIG)
        print("\n[6] Denetim Excel:", ap)
        print("\n[7] Otomatik yorum / karar:")
        for line in AUDIT["interpretation"]:
            print("   -", line)
    print("=" * 70)
except Exception as _e:
    import traceback
    print("ARDL denetimi calistirilamadi (ana notebook etkilenmedi):", _e)
    traceback.print_exc()